# GNN Powerflow V2.6 — Analysis

Loads trained runs, evaluates GNN vs. baseline solvers, produces comparison plots and statistics.

**Input**: sweep JSONs from `GNN_Powerflow_V2.6_Training.ipynb`

In [ ]:
# %%
import copy
import logging
import os
import pickle
import random
import time
from typing import Dict, List, Optional, Tuple
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict

# Fix for numba >= 0.60: version_version attribute was removed; inject it before pandapower imports it
import sys
print("pandapower already imported:", "pandapower" in sys.modules)
try:
    import numba._version as _nv
    if not hasattr(_nv, 'version_version'):
        import numba as _n; _nv.version_version = _n.__version__
except ImportError:
    pass

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pypsa
import pypowsybl as pp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Batch, Data, Dataset
from torch_geometric.nn import GATv2Conv, GCNConv, GraphConv
from tqdm import tqdm

import urllib

import statsmodels.api as sm
from statsmodels.formula.api import ols


import numba, pandapower as pdp
from pandapower.converter import from_ppc
import pandapower.networks as pn

# Suppress verbose output from PyPSA and dependencies during data generation
for _log in ("pypsa", "pypsa.pf", "pypsa.components", "numexpr", "linopy"):
    logging.getLogger(_log).setLevel(logging.WARNING)
logging.getLogger("numexpr").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

import re
from scipy import ndimage

In [ ]:
# ── Configurable paths ───────────────────────────────────────────────────────
# Set DATA_ROOT to wherever your data folders live.
# Use "." to keep everything relative to the repo (default).
# Example: DATA_ROOT = r"C:\Users\you\OneDrive\Data_PF_GNN"
import os as _os
DATA_ROOT             = r"C:\Users\STSI\OneDrive - USN\Data_PF_GNN"                                             # ← change this
GRID_MODEL_FILES      = _os.path.join(DATA_ROOT, "grid_model_files")    # static grid CSV/RAW files
TRAINING_NETWORKS_DIR = _os.path.join(DATA_ROOT, "training_networks_saved")  # dataset JSONs
TRAINING_RESULTS_DIR  = _os.path.join(DATA_ROOT, "training_results_saved")   # sweep result JSONs/CSVs


In [ ]:
#test and execution toggles
run_smoke_test=False # to run a smoke test on network generation
run_regression_test=False # to run a regression test on the GNN training loop (with very limited epochs and data)
run_powsybl_extract = False # extract data from powsybl for the first time, or re-extract after edits to the extraction code
run_cigre_extract = False # extract data from CIGRE for the first time, or re-extract after edits to the extraction code    
run_powsybl_extract = False
run_cigre_extract = False

In [ ]:
import itertools

# ── Style pools ──────────────────────────────────────────────────────────────
_LINE_STYLES  = ["-", "--", "-.", ":"]
_MARKERS      = ["", "o", "s", "^", "D", "v", "P", "X"]
_COLOR_CYCLE  = plt.rcParams["axes.prop_cycle"].by_key()["color"]

## Data Generation Functions
_Paste from V2.5.3 cells 20, 22, 23, 25, 26_

In [ ]:
#==================save and load genrated datasets========================


def load_dataset_list(index_path: str) -> list:
    """
    Reload a list of PyPSA networks saved by save_dataset_list.
    """
    import json

    with open(index_path, "r") as f:
        meta = json.load(f)

    save_dir = os.path.dirname(index_path) or "."
    networks: list[pypsa.Network] = []
    for nc_name in meta["files"]:
        nc_path = os.path.join(save_dir, nc_name)
        networks.append(pypsa.Network(nc_path))

    print(f"Loaded {len(networks)} networks from {index_path}")
    return networks



## Power System Helpers

## GNN Architecture

In [ ]:
# =============================================================================
# SECTION 5: GNN MODEL
# Must be defined here so pickle can reconstruct saved PowerFlowGNN models.
# =============================================================================
  
class PowerFlowGNN(nn.Module):
    """
    Graph Attention Network (GATv2) for power flow prediction.
    Predicts per-bus: [vmag, vang, P, Q]
    Predicts per-edge bilinear PTDF: hedges @ W @ hnodes.T
    """

    def __init__(
        self,
        node_features: int = 7,
        edge_features: int = 4,
        hidden_dim: int = 64, # Hiddem dimensions means the size of the feature vectors that are passed between layers in the GNN. A larger hidden_dim allows the model to capture more complex relationships, but also increases computational cost and risk of overfitting. 64 is a common choice for a balance between expressiveness and efficiency.
        num_layers: int = 3,# Number of GNN layers (graph convolutional layers). More layers allow the model to capture more complex interactions between nodes, but can also lead to over-smoothing where node features become too similar. 3 layers is a common choice for many graph tasks.
        heads: int = 4,# Number of attention heads in GAT layers. Multiple heads allow the model to attend to different aspects of the graph structure simultaneously. 4 heads is a common choice that provides a good balance between expressiveness and computational cost.
        dropout: float = 0.0,
        conv_type: str = "gatv2", # Placeholder for potential future extension to other convolution types
        head_mode:str = "standard", # or with new encoder
        conv_mode: str = "old",  # "old" | "residual" | "res_norm" | "res_norm_drop"
        drop_rate: float = 0.1,  # dropout rate for conv block (only used when conv_mode="res_norm_drop")
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.heads = heads
        self.dropout = dropout
        self.conv_type = conv_type
        self.head_mode = head_mode
        self.conv_mode = conv_mode
        self.drop_rate = drop_rate
        # 1. Node embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # 2. GAT convolution layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(self._make_conv(hidden_dim, hidden_dim, edge_features))

        # 2b. Optional LayerNorm + Dropout for residual conv modes
        if conv_mode in ("res_norm", "res_norm_drop"):
            self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)]) #Jordi
        if conv_mode == "res_norm_drop":
            self.drop = nn.Dropout(drop_rate)

        # 3. Output layers for different predictions
        if head_mode=="standard":
            # one prediction head for each of the properties we want to predict
            self.vmag_pred = nn.Linear(hidden_dim, 1)   # Voltage magnitude
            self.vang_pred = nn.Linear(hidden_dim, 1)   # Voltage angle
            self.p_pred = nn.Linear(hidden_dim, 1)      # Active power
            self.q_pred = nn.Linear(hidden_dim, 1)      # Reactive power
        if head_mode=="with_encoder":
            self.pq_head = nn.Linear(hidden_dim, 2)      # for PQ nodes
            self.pv_head = nn.Linear(hidden_dim, 2)      # For PV nodes
            self.slack_head = nn.Linear(hidden_dim, 2)   # For Slack nodes

        # STSI 26.02.11: Edge embedding MLP: (h_src || h_dst || edge_attr) -> hidden_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_features, hidden_dim),
            nn.LeakyReLU(),
        )

        # Bilinear PTDF parameter: h_edge^T W h_bus
        self.ptdf_W = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        
        self._initialize_weights()

    def _make_conv(self, in_dim: int, out_dim: int, edge_features: int):
        if self.conv_type == "gatv2":
            return GATv2Conv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=self.dropout,
            )
        elif self.conv_type == "gcn":
            return GCNConv(in_dim, out_dim, add_self_loops=False)
        elif self.conv_type == "graphconv":
            return GraphConv(in_dim, out_dim)
        else:
            raise ValueError(f"Unknown conv_type: {self.conv_type}")

    def _apply_conv(self, conv, h, edge_index, edge_attr):
        """Apply a single conv layer, dispatching on conv type."""
        if isinstance(conv, GATv2Conv):
            return conv(h, edge_index, edge_attr)
        else:
            return conv(h, edge_index)

    def forward(self, data, return_embeddings=False):
        """
        Forward pass: embed -> GAT layers -> node heads + bilinear PTDF.

        Args:
            data: PyG Data/Batch object
            return_embeddings: if True, return (node_pred, h_nodes, h_edges)
                               for use in PTDF loss outside forward
        Returns:
            (node_pred [num_nodes, 4], ptdf_pred [num_edges, num_nodes])
            or (node_pred, h_nodes, h_edges) if return_embeddings=True
        """
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Initial embedding
        h = self.node_embedding(x)

        # GAT layers — mode-aware conv block
        if self.conv_mode == "old":
            for conv in self.convs:
                h = self._apply_conv(conv, h, edge_index, edge_attr)
                h = F.leaky_relu(h)
        elif self.conv_mode == "residual":
            for conv in self.convs:
                residual = h
                h = self._apply_conv(conv, h, edge_index, edge_attr)
                h = F.gelu(h)
                h = h + residual
        elif self.conv_mode == "res_norm":
            for conv, norm in zip(self.convs, self.norms):
                residual = h
                h = self._apply_conv(conv, h, edge_index, edge_attr)
                h = norm(h)
                h = F.gelu(h)
                h = h + residual
        elif self.conv_mode == "res_norm_drop":
            for conv, norm in zip(self.convs, self.norms):
                residual = h
                h = self._apply_conv(conv, h, edge_index, edge_attr)
                h = norm(h)
                h = F.gelu(h) #Jordi
                h = self.drop(h)
                h = h + residual


        h_nodes = h  # Final node features after all GAT layers

        # Node heads
        # Predictions are made using the output of the last graph convolution layer

        if self.head_mode=="standard":
            vmag_pred = self.vmag_pred(h_nodes)
            vang_pred = self.vang_pred(h_nodes)
            p_pred = self.p_pred(h_nodes)
            q_pred = self.q_pred(h_nodes)
            node_pred = torch.cat([vmag_pred, vang_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 4]
        elif self.head_mode=="with_encoder":
            node_pred = torch.zeros(h_nodes.size(0), 4, device=h_nodes.device)
            pq_mask = data.pq_mask.bool()
            pv_mask = data.pv_mask.bool()
            slack_mask = data.slack_mask.bool()
            
            if pq_mask.any():
                node_pred[pq_mask,0:2]=self.pq_head(h_nodes[pq_mask])  # Predict Vmag, Vang for PQ nodes
            if pv_mask.any():
                pv_out=self.pv_head(h_nodes[pv_mask])  # Predict Vmag, Vang for PV nodes
                node_pred[pv_mask,1]=pv_out[:,0]  # Vmag from PV head
                node_pred[pv_mask,3]=pv_out[:,1]  # Q from PV head
            if slack_mask.any():
                slack_out=self.slack_head(h_nodes[slack_mask])  # Predict Vmag, Vang for Slack nodes
                node_pred[slack_mask, 2] = slack_out[:, 0]  # P from Slack head
                node_pred[slack_mask, 3] = slack_out[:, 1]  # Q from Slack head

        # Edge embeddings
        row, col = data.edge_index
        h_src = h_nodes[row]   # [num_edges, hidden_dim]
        h_dst = h_nodes[col]   # [num_edges, hidden_dim]
        edge_input = torch.cat([h_src, h_dst, edge_attr], dim=-1)  # [num_edges, 2*hidden_dim + edge_features]
        h_edges = self.edge_mlp(edge_input)  # [num_edges, hidden_dim]

        if return_embeddings:
            return node_pred, h_nodes, h_edges

        # Bilinear PTDF: H_W = h_edges @ W,  ptdf_pred = H_W @ h_nodes.T
        H_W = h_edges @ self.ptdf_W                   # [num_edges, hidden_dim]
        ptdf_pred = H_W @ h_nodes.T                   # [num_edges, num_nodes]
        return node_pred, ptdf_pred

    def _initialize_weights(self, gain=1.0):
        """
        STSI 24.09.25: Enhanced weight initialization for the entire network.
        """
        # 1. Initialize node embedding layer (Added 24.09.25)
        nn.init.xavier_uniform_(self.node_embedding.weight, gain=gain)
        nn.init.zeros_(self.node_embedding.bias)

        # 2. Initialize GAT convolution layers (Added 24.09.25)
        for conv in self.convs:
            if hasattr(conv, "lin_l") and conv.lin_l is not None:
                nn.init.xavier_uniform_(conv.lin_l.weight, gain=gain)
            if hasattr(conv, "lin_r") and conv.lin_r is not None:
                nn.init.xavier_uniform_(conv.lin_r.weight, gain=gain)
            if hasattr(conv, "lin_edge") and conv.lin_edge is not None:
                nn.init.xavier_uniform_(conv.lin_edge.weight, gain=gain)
            # GAT layers have multiple linear transformations

        # 3. Initialize output layers
        if self.head_mode == "standard":
            prediction_layers = [
                (self.vmag_pred, 1.0),
                (self.vang_pred, 0.0),
                (self.p_pred,    0.0),
                (self.q_pred,    0.0),
            ]
            for layer, bias_init in prediction_layers:
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.constant_(layer.bias, bias_init)
        elif self.head_mode == "with_encoder":
            for layer in (self.pq_head, self.pv_head, self.slack_head):
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.zeros_(layer.bias)
            # Vmag outputs (col 0 of pq_head) should start near 1.0 pu
            nn.init.constant_(self.pq_head.bias[0], 1.0)



In [ ]:
# =============================================================================
# SECTION 4: DATASET
# Must be defined here so evaluate_dc_baseline and other evaluation functions
# that construct PowerFlowDataset can run in this notebook.
# =============================================================================



class PowerFlowDataset(Dataset):
    """
    One data object per (network, snapshot) pair.

    The graph topology reflects the physical network topology:
    - buses → GNN nodes
    - lines + transformers → GNN edges (bidirectional)

    Each graph carries:
      x         : [n_buses, 7 or 8] node features (bus type flags, known P/Q/V/Vang, optional p_nom_share)
      y         : [n_buses, 4]       targets (vmag, vang, p, q)
      edge_index: [2, n_edges]       bidirectional
      edge_attr : [n_edges, 6]       edge features (r, x, b/2, tap, g_series, b_series)
      masks     : slack_mask, pv_mask, pq_mask
      y_line_p  : [n_fwd_lines]      forward-direction real power per supervised edge (PTDF supervision)
      ptdf_edge_row_idx : [n_edges]  PTDF row index for each edge (-1 for non-supervised/reverse)  # STSI 26.04.07
      network_idx: int
    """

    def __init__(
        self,
        networks: list,
        use_edge_features: bool = True,
        use_pnet_balance: bool = True,   # [STSI100526] kept for backfill compat (ignored internally)
        use_pnom_share: bool = False,     # [STSI100526] col7: p_nom_share (was col7=p_bal + col8=pnom_share before 100526)
        transform=None,
        pre_transform=None,
        ptdf_branch_mode: str = "lines",  # "lines" or "all" — controls PTDF supervision scope
    ):
        super().__init__(root=None, transform=transform, pre_transform=pre_transform)
        self.networks          = networks
        self.use_edge_features = use_edge_features
        self.use_pnom_share    = use_pnom_share
        self.ptdf_branch_mode  = ptdf_branch_mode #STSI260402: pass ptdf_branch_mode to _create_graph_data for edge feature control

        # Build flat index: list of (network_idx, timestep_idx)
        self._index = []
        for net_idx, net in enumerate(networks):
            for t_idx in range(len(net.snapshots)):
                self._index.append((net_idx, t_idx))

    def len(self):
        return len(self._index)

    def get(self, idx):
        net_idx, t_idx = self._index[idx]
        network = self.networks[net_idx]
        data = self._create_graph_data(network, t_idx, net_idx)
        return data

    def _create_graph_data(self, network: pypsa.Network, t_idx: int, net_idx: int) -> Data:
        buses     = list(network.buses.index)
        n_buses   = len(buses)
        bus_to_i  = {b: i for i, b in enumerate(buses)}
        snapshot  = network.snapshots[t_idx]

        # ── Bus type flags ────────────────────────────────────────────────────
        is_slack = torch.zeros(n_buses, dtype=torch.float)
        is_pv    = torch.zeros(n_buses, dtype=torch.float)
        is_pq    = torch.zeros(n_buses, dtype=torch.float)

        slack_mask = torch.zeros(n_buses, dtype=torch.bool)
        pv_mask    = torch.zeros(n_buses, dtype=torch.bool)
        pq_mask    = torch.zeros(n_buses, dtype=torch.bool)

        for gen_name, gen in network.generators.iterrows():
            bus_name = gen["bus"]
            if bus_name not in bus_to_i:
                continue
            i = bus_to_i[bus_name]
            ctrl = gen["control"]
            if ctrl == "Slack":
                is_slack[i] = 1.0
                slack_mask[i] = True
            elif ctrl == "PV":
                is_pv[i] = 1.0
                pv_mask[i] = True

        # Buses with no generator → PQ
        for i in range(n_buses):
            if not slack_mask[i] and not pv_mask[i]:
                is_pq[i]   = 1.0
                pq_mask[i] = True

        # ── Bus injections (P, Q) from PyPSA post-solve ───────────────────────
        # STSI260404: BUGFIX — use .loc[snapshot, buses] to force column order
        # to match network.buses.index.  Without this, buses_t columns may be
        # in lexicographic order (e.g. "Bus 10" before "Bus 2"), mis-aligning
        # P/Q/Vang with bus-type masks and the Y-matrix built from buses.index.
        p_bus = torch.tensor(
            network.buses_t.p.loc[snapshot, buses].values, dtype=torch.float
        )
        q_bus = torch.tensor(
            network.buses_t.q.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Voltages ─────────────────────────────────────────────────────────────
        v_mag = torch.tensor(
            network.buses_t.v_mag_pu.loc[snapshot, buses].values, dtype=torch.float
        )
        v_ang = torch.tensor(
            network.buses_t.v_ang.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Input features: GNN receives known values, zeros where unknown ────
        # Known before solve: bus type flags, P_inj for PQ+PV, Q_inj for PQ,
        #                     Vmag for PV+Slack, Vang for Slack (=0)
        #
        # x layout: [is_slack(0), is_PV(1), is_PQ(2), P(3), Q(4), Vmag(5), Vang(6)]
        #
        # NOTE: Slack P input x[:,3]=0 is a known issue — the model sees no
        # varying signal for slack P.  # [STSI100526] removed p_net_balance (was here)
        x_p    = p_bus.clone()
        x_q    = q_bus.clone()
        x_vmag = v_mag.clone()
        x_vang = v_ang.clone()

        # Mask unknowns to zero in input (prevents trivial leakage)
        x_p[slack_mask]    = 0.0   # Slack P unknown (to be predicted)
        x_q[slack_mask]    = 0.0   # Slack Q unknown
        x_q[pv_mask]       = 0.0   # PV Q unknown
        x_vmag[pq_mask]    = 0.0   # PQ Vmag unknown
        x_vang[pq_mask]    = 0.0   # PQ Vang unknown
        x_vang[pv_mask]    = 0.0   # PV Vang unknown

        x = torch.stack([is_slack, is_pv, is_pq, x_p, x_q, x_vmag, x_vang], dim=1)

        if self.use_pnom_share:  # [STSI100526] removed p_bal_col + p_net_balance; synced with Training
            # col7: static p_nom share per gen bus (p_nom_i / Σp_nom)
            # Tells the model each generator's relative capacity for slack distribution.
            p_nom_share_col = torch.zeros(n_buses, 1)
            p_noms = network.generators["p_nom"].values.astype(float)
            total_p_nom = float(p_noms.sum()) if p_noms.sum() > 0 else 1.0
            for gen_name, gen in network.generators.iterrows():
                b = gen["bus"]
                if b in bus_to_i:
                    p_nom_share_col[bus_to_i[b], 0] += gen["p_nom"] / total_p_nom
            x = torch.cat([x, p_nom_share_col], dim=1)

        # ── Target: full post-solve state for all buses ──────────────────────────
        y = torch.stack([v_mag, v_ang, p_bus, q_bus], dim=1)

        # ── Edge construction (lines + transformers) ──────────────────────────────
        edge_index_list = []
        edge_attr_list  = []
        forward_edge_mask = []   # True for first direction (not reverse duplicate)
        fwd_line_idx_list  = []   # one entry per forward supervised edge
        ptdf_edge_row_idx_list = []  # STSI 26.04.07: PTDF row index per edge for learned_edge mode

        branches = []
        #STSI260402: control which branches are included in PTDF supervision via self.ptdf_branch_mode
        # lines always included
        for line_name, line in network.lines.iterrows():
            if line["bus0"] in bus_to_i and line["bus1"] in bus_to_i:
                branches.append(
                    (
                        line.name,  # name (not used in PTDF but useful for debugging)
                        line["bus0"], line["bus1"],
                        line["r"], line["x"],
                        line.get("b", 0.0), 1.0, "line"
                    )
                )

        # STSI260404: BUGFIX — transformers must ALWAYS be added as graph edges
        # for GNN message passing, regardless of ptdf_branch_mode.
        # ptdf_branch_mode only controls which branches are supervised by PTDF
        # loss (via forward_edge_mask / fwd_line_idx_list), NOT graph topology.
        # Previously gated behind `if self.ptdf_branch_mode == "all":` which
        # silently severed the graph when mode was "lines" (the default).
        for tr_name, tr in network.transformers.iterrows():
            if tr["bus0"] in bus_to_i and tr["bus1"] in bus_to_i:
                tap = tr.get("tap_ratio", 1.0)
                tap = tap if not pd.isna(tap) else 1.0
                branches.append(
                    (
                        tr.name,  # name (not used in PTDF but useful for debugging)
                        tr["bus0"], tr["bus1"],
                        tr["r"], tr["x"],
                        tr.get("b", 0.0), float(tap), "trafo"
                    )
                )


        for branch_idx, (name, b0, b1, r, x_val, b_val, tap, btype) in enumerate(branches):
            i0, i1 = bus_to_i[b0], bus_to_i[b1]
            z      = complex(r, x_val)
            y_ser  = (1.0 / z) if abs(z) > 1e-12 else 0.0
            g_ser, b_ser = y_ser.real, y_ser.imag

            if self.use_edge_features:
                attr = [r, x_val, b_val / 2.0, tap, g_ser, b_ser]
            else:
                attr = [1.0]

            # Forward edge
            edge_index_list.append([i0, i1])
            edge_attr_list.append(attr)

            # PTDF supervision: only lines (or all branches) get True in forward_edge_mask
            is_supervised = (btype == "line") if self.ptdf_branch_mode == "lines" else True
            forward_edge_mask.append(is_supervised)
            if is_supervised:
                fwd_line_idx_list.append(branch_idx)
                ptdf_edge_row_idx_list.append(branch_idx)  # STSI 26.04.07: supervised forward edge
            else:
                ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: unsupervised forward edge

            # Reverse edge
            edge_index_list.append([i1, i0])
            edge_attr_list.append(attr)
            forward_edge_mask.append(False)
            ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: reverse edge

        if edge_index_list:
            edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
            edge_attr  = torch.tensor(edge_attr_list,  dtype=torch.float)
            fwd_mask   = torch.tensor(forward_edge_mask, dtype=torch.bool)
            line_idx   = torch.tensor(fwd_line_idx_list, dtype=torch.long)
            ptdf_edge_row_idx = torch.tensor(ptdf_edge_row_idx_list, dtype=torch.long)  # STSI 26.04.07
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr  = torch.zeros((0, len(attr) if branches else 1), dtype=torch.float)
            fwd_mask   = torch.zeros(0, dtype=torch.bool)
            line_idx   = torch.zeros(0, dtype=torch.long)
            ptdf_edge_row_idx = torch.zeros(0, dtype=torch.long)  # STSI 26.04.07


        try:
            ptdf_matrix = compute_ptdf_matrix(network)
            y_ptdf = torch.tensor(ptdf_matrix, dtype=torch.float)
        except Exception:
            n_lines = len(network.lines)
            y_ptdf = torch.zeros((n_lines, n_buses), dtype=torch.float)

        # STSI260402 Assertions — verify PTDF supervision is lines-only and consistent with y_ptdf
        n_lines = len(network.lines)
        assert int(fwd_mask.sum()) == n_lines, \
            f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines {n_lines}"
        assert y_ptdf.shape[0] == n_lines, \
            f"y_ptdf rows {y_ptdf.shape[0]} != n_lines {n_lines}"

        # ── Line active power flows ───────────────────────────────────────────
        if not network.lines_t.p0.empty:
            y_line_p = torch.tensor(
                network.lines_t.p0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_p = torch.zeros(len(network.lines), dtype=torch.float)

        # ── Assemble Data object ──────────────────────────────────────────────
        data = Data(
            x          = x,
            y          = y,
            edge_index = edge_index,
            edge_attr  = edge_attr,
        )

        # Bus-type masks — node-level bool tensors (critical for collate + loss)
        data.slack_mask = slack_mask   # [n_nodes] bool
        data.pv_mask    = pv_mask      # [n_nodes] bool
        data.pq_mask    = pq_mask      # [n_nodes] bool

        # Variable-size targets (handled specially in collate_with_ptdf)
        data.y_ptdf      = y_ptdf
        data.y_line_p    = y_line_p
        data.forward_edge_mask = fwd_mask
        data.ptdf_line_index = line_idx
        data.ptdf_edge_row_idx = ptdf_edge_row_idx  # STSI 26.04.07: per-edge PTDF row index for learned_edge mode

        # Graph-level metadata
        data.network_idx    = torch.tensor([net_idx], dtype=torch.long)

        return data




In [ ]:
# =============================================================================
# SECTION 7: LINE FLOW CALCULATION
# =============================================================================

def calculate_line_flows(network, vmag, vang, t_idx=None):
    """
    Calculate AC line flows from predicted voltages.

    Args:
        network: PyPSA network object
        vmag:    Voltage magnitudes [num_buses], numpy array, p.u.
        vang:    Voltage angles [num_buses], numpy array, radians
        t_idx:   Snapshot index (used for nominal values only)
    Returns:
        dict with keys p0, p1, q0, q1 - each a numpy array of length num_lines (p.u.)
    """
    num_lines = len(network.lines)
    p0 = np.zeros(num_lines)
    p1 = np.zeros(num_lines)
    q0 = np.zeros(num_lines)
    q1 = np.zeros(num_lines)

    all_buses = list(network.buses.index)
    # BUGFIX: use bus_to_idx dict instead of string parsing (topology-safe)
    bus_to_idx = {bus_name: idx for idx, bus_name in enumerate(all_buses)}

    for i, line in enumerate(network.lines.index):
        from_bus = network.lines.loc[line, "bus0"]
        to_bus = network.lines.loc[line, "bus1"]

        # BUGFIX: use dict lookup instead of int(bus.split('-')[1]) - 1
        from_idx = bus_to_idx[from_bus]
        to_idx = bus_to_idx[to_bus]

        r = network.lines.loc[line, "r"]
        x = network.lines.loc[line, "x"]
        b = network.lines.loc[line, "b"] if "b" in network.lines.columns else 0.0

        # Line impedance and admittance
        z = complex(r, x)
        y_series = 1.0 / z if abs(z) > 1e-12 else 0.0
        y_shunt = complex(0, b / 2.0)

        # Complex voltages at from/to buses
        V_from = vmag[from_idx] * np.exp(1j * vang[from_idx])
        V_to = vmag[to_idx] * np.exp(1j * vang[to_idx])

        # Line current from sending end
        I_from = (V_from - V_to) * y_series + V_from * y_shunt
        I_to = (V_to - V_from) * y_series + V_to * y_shunt

        S_from = V_from * np.conj(I_from)
        S_to = V_to * np.conj(I_to)

        s_nom = network.lines.loc[line, "s_nom"]
        # Convert from p.u. to MW/MVAr using system base
        #base_mva = getattr(network, "sn_mva", 100.0)

        p0[i] = S_from.real #* base_mva # removed the base_mva scaling to keep flows in p.u. for better numerical stability and consistency with the rest of the model's inputs/outputs
        q0[i] = S_from.imag #* base_mva
        p1[i] = -S_to.real #* base_mva  # convention: positive = into bus
        q1[i] = -S_to.imag #* base_mva

    return {"p0": p0, "p1": p1, "q0": q0, "q1": q1}


def build_line_results_from_flows(network, flows, snapshot_label):
    """
    Helper to build a line_results-style DataFrame for one snapshot
    from a dict produced by calculate_line_flows.
    """
    idx = pd.Index([snapshot_label], name="snapshot")
    cols = pd.MultiIndex.from_product(
        [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
    )
    df = pd.DataFrame(index=idx, columns=cols, dtype=float)
    for i, line in enumerate(network.lines.index):
        df.loc[snapshot_label, (line, "P0 pu")] = flows["p0"][i]
        df.loc[snapshot_label, (line, "P1 pu")] = flows["p1"][i]
        df.loc[snapshot_label, (line, "Q0 pu")] = flows["q0"][i]
        df.loc[snapshot_label, (line, "Q1 pu")] = flows["q1"][i]
    return df



In [ ]:
# =============================================================================
# SECTION 9: INFERENCE & PREDICTION
# =============================================================================

def _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset=0, device=None):
    """Augment data.edge_attr with learned PTDF rows for learned_edge models.

    Shared helper used by evaluate_model, compare_pf_conventional_vs_gnn,
    predict_network_results_with_masks, and evaluate_gnn_on_test_set.

    Args:
        data:          single PyG Data object (already moved to device)
        ptdf_params:   list[nn.Parameter] or None
        max_buses:     int, PTDF parameter width
        net_idx:       int, network index within the dataset
        ptdf_offset:   int, added to net_idx to get global ptdf_params index
        device:        torch device (inferred from data.edge_attr if None)

    Returns:
        data with edge_attr augmented in-place. No-op if ptdf_params is None.
    """
    if ptdf_params is None or max_buses <= 0:
        return data
    if device is None:
        device = data.edge_attr.device
    row_idx = data.ptdf_edge_row_idx  # [E] line index or -1
    ptdf_rows = torch.zeros(data.edge_attr.size(0), max_buses, device=device)
    valid = (row_idx >= 0) & (row_idx < ptdf_params[net_idx + ptdf_offset].size(0))
    if valid.any():
        ptdf_rows[valid] = ptdf_params[net_idx + ptdf_offset][row_idx[valid]].to(device)
    data.edge_attr = torch.cat([data.edge_attr, ptdf_rows], dim=-1)
    return data


def model_results(network, print_results=False, plot_results=False, compute_ptdf=False):
    """
    Run PyPSA power flow and store results in DataFrames.
    Print if print_results=True, plot if plot_results=True.

    DISTSLACK: pass distribute_slack=True to use PyPSA's distributed slack solver.
    """
    # DISTSLACK: use distribute_slack=True to properly handle distributed slack
    network.pf(use_seed=True, distribute_slack=False)  # DISTSLACK: change to True when using distributed slack

    snapshots = network.snapshots
    buses = network.buses.index
    lines = network.lines.index

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]),
        dtype=float,
    )

    for t in snapshots:
        for bus in buses:
            bus_results.loc[t, (bus, "P pu")]    = network.buses_t.p.loc[t, bus]
            bus_results.loc[t, (bus, "Q pu")]  = network.buses_t.q.loc[t, bus]
            bus_results.loc[t, (bus, "V pu")]    = network.buses_t.v_mag_pu.loc[t, bus]
            bus_results.loc[t, (bus, "Angle deg")] = np.degrees(network.buses_t.v_ang.loc[t, bus])
        for line in lines:
            line_results.loc[t, (line, "P0 pu")]   = network.lines_t.p0.loc[t, line]
            line_results.loc[t, (line, "P1 pu")]   = network.lines_t.p1.loc[t, line]
            line_results.loc[t, (line, "Q0 pu")] = network.lines_t.q0.loc[t, line]
            line_results.loc[t, (line, "Q1 pu")] = network.lines_t.q1.loc[t, line]

    if print_results:
        print(bus_results)
        print(line_results)
    if plot_results:
        plot_network_results(network, bus_results)

    ptdf_df = None
    if compute_ptdf:
        ptdf_df = compute_ptdf_matrix(network)

    return bus_results, line_results, ptdf_df


def batched_gnn_forward(model, data_list):
    """Run GNN on a batch of graphs in a single forward pass."""
    exclude_keys = ['y_ptdf', 'ptdf_line_index', 'y_line_p']
    batch = Batch.from_data_list(data_list, exclude_keys=exclude_keys)
    out = model(batch)
    if isinstance(out, tuple):
        out = out[0]
    results = []
    for g in range(len(data_list)):
        mask = (batch.batch == g)
        results.append(out[mask])
    return results


def predict_network_results_with_masks(
    model, network, use_edge_features=True, debug=False, use_pnom_share=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None, max_buses=0, ptdf_offset=0,
):
    """
    Run GNN prediction on a network and return bus/line results with prediction masks.

    DISTSLACK: prediction masks now also flag distributed slack buses correctly.
    Distributed slack buses predict P and Q (same as single slack).
    STSI 26.04.07: Augments edge_attr when ptdf_params is provided (learned_edge).

    Args:
        model:   Trained PowerFlowGNN model
        network: PyPSA network object (must have solved power flow for feature extraction)
    Returns:
        bus_results, line_results, prediction_masks
    """
    dataset = PowerFlowDataset(
        [network],
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
    )
    snapshots = network.snapshots
    buses = list(network.buses.index)
    bus_to_idx = {b: i for i, b in enumerate(buses)}

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product(
            [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
        ),
        dtype=float,
    )
    # Prediction masks: which properties were predicted (not taken from input)
    prediction_masks = {
        "P":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Q":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "V":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Angle": pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
    }

    model.eval()
    with torch.no_grad():
        for t_idx, t in enumerate(snapshots):
            data = dataset[t_idx]

            # STSI 26.04.07: augment edge_attr for learned_edge models
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx=0, ptdf_offset=ptdf_offset)

            # Fallback: zero-pad if model expects wider edge_attr (old runs without stored ptdf_params)
            if hasattr(model, 'convs') and model.convs and hasattr(model.convs[0], 'lin_edge') and model.convs[0].lin_edge is not None:
                _expected = model.convs[0].lin_edge.in_channels
                if data.edge_attr.size(1) < _expected:
                    data.edge_attr = torch.cat([data.edge_attr, torch.zeros(data.edge_attr.size(0), _expected - data.edge_attr.size(1))], dim=1)

            node_pred, _ = model(data)  # [num_buses, 4]: vmag, vang, p, q

            pred_vmag = node_pred[:, 0]
            pred_vang = node_pred[:, 1]
            pred_p    = node_pred[:, 2]
            pred_q    = node_pred[:, 3]

            for i, bus in enumerate(buses):
                bus_type = "PQ"  # default
                connected_gens = network.generators[network.generators.bus == bus]
                if len(connected_gens) > 0:
                    ctrl = connected_gens.iloc[0]["control"]
                    if ctrl in ("Slack", "PV"):
                        bus_type = ctrl

                if bus_type == "PQ":
                    # For PQ buses: predict vmag, vang
                    pred_vmag_i = pred_vmag[i]
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]   # Known
                    pred_q_i    = network.buses_t.q.loc[t, bus]   # Known
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = False
                    prediction_masks["V"].loc[t, bus]     = True
                    prediction_masks["Angle"].loc[t, bus] = True

                elif bus_type == "PV":
                    # For PV buses: known vmag, predict vang, q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]         # Known
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = True

                else:
                    # Slack (single or distributed): known vmag, vang; predict p, q
                    # DISTSLACK: for distributed slack, vmag is known but angle
                    # reference is only soft-fixed; we still use the known angle
                    # from the PF solution as input and predict P, Q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = network.buses_t.v_ang.loc[t, bus]     # Known (reference)
                    pred_p_i    = pred_p[i]
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = True
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = False

                bus_results.loc[t, (bus, "P pu")]      = pred_p_i.item() if torch.is_tensor(pred_p_i) else pred_p_i
                bus_results.loc[t, (bus, "Q pu")]      = pred_q_i.item() if torch.is_tensor(pred_q_i) else pred_q_i
                bus_results.loc[t, (bus, "V pu")]      = pred_vmag_i.item() if torch.is_tensor(pred_vmag_i) else pred_vmag_i
                bus_results.loc[t, (bus, "Angle deg")] = np.degrees(
                    pred_vang_i.item() if torch.is_tensor(pred_vang_i) else pred_vang_i
                )

            if debug:
                print(f"Predicted angles should be small radians: {pred_vang[:3].numpy()}")
                print(f"Predicted angles in degrees: {np.degrees(pred_vang[:3].numpy())}")
                print(f"Angle range: {pred_vang.min():.6f} to {pred_vang.max():.6f} radians")

            # Calculate line flows from predicted voltages
            vmag_np = np.array([
                bus_results.loc[t, (bus, "V pu")] for bus in buses
            ], dtype=float)
            vang_np = np.radians(np.array([
                bus_results.loc[t, (bus, "Angle deg")] for bus in buses
            ], dtype=float))
            pred_line_flows = calculate_line_flows(network, vmag_np, vang_np, t_idx)

            for i, line in enumerate(network.lines.index):
                line_results.loc[t, (line, "P0 pu")]   = pred_line_flows["p0"][i]
                line_results.loc[t, (line, "P1 pu")]   = pred_line_flows["p1"][i]
                line_results.loc[t, (line, "Q0 pu")] = pred_line_flows["q0"][i]
                line_results.loc[t, (line, "Q1 pu")] = pred_line_flows["q1"][i]

    return bus_results, line_results, prediction_masks

def evaluate_pypsa_timing(networks, verbose=True):
    """Measure PyPSA AC PF wall time by re-running pf() on a network copy.
    
    Runs pf() on all snapshots at once per network (same as normal usage),
    divides by snapshot count for per-snapshot average.
    Error = 0 by definition (PyPSA is the ground truth).
    """
    all_times_ms = []
    for network in networks:
        net_copy = network.copy()
        t0 = time.perf_counter()
        net_copy.pf(use_seed=True)
        t1 = time.perf_counter()
        ms_per_snap = (t1 - t0) * 1000.0 / max(1, len(network.snapshots))
        all_times_ms.append(ms_per_snap)

    avg_ms = float(np.mean(all_times_ms))
    metrics = {
        'vmag_mae': 0.0, 'vang_mae': 0.0,
        'p_mae': 0.0, 'q_mae': 0.0,
        'line_flow_mae': 0.0, 'line_flow_q0_mae': 0.0,
        'inference_time_per_snapshot_ms': avg_ms,
        'n_snapshots': sum(len(n.snapshots) for n in networks),
    }
    if verbose:
        print(f"PyPSA AC timing: {avg_ms:.3f} ms/snap avg over {len(networks)} network(s)")
    return metrics

def evaluate_pypsa_timing_single(networks, verbose=True):
    """Measure PyPSA AC PF time per individual snapshot (no batching)."""
    times_ms = []
    for net in networks:
        for snap in net.snapshots:
            net_copy = net.copy()
            net_copy.set_snapshots([snap])
            t0 = time.perf_counter()
            net_copy.pf(use_seed=True)
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000)
    
    avg_ms = float(np.mean(times_ms))
    metrics = {
        'vmag_mae': 0.0, 'vang_mae': 0.0,
        'p_mae': 0.0, 'q_mae': 0.0,
        'line_flow_mae': 0.0, 'line_flow_q0_mae': 0.0,
        'inference_time_per_snapshot_ms': avg_ms,
    }
    if verbose:
        print(f"PyPSA AC (single): {avg_ms:.3f} ms/snap over {len(times_ms)} snapshots")
    return metrics

# STSI 26.04.07: added backfill_line_flow_metrics to compute line flow errors separately from bus errors, and to support learned_edge models with augmented edge_attr for PTDF rows.

def backfill_gnn_timing(
    runs: list,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    n_warmup: int = 5,
    batch_size: int = 64,
    device=None,
):
    """
    Backfill per-snapshot timing into sweep runs with live model objects.
    
    Three timing levels:
      solve_time_ms:       single forward pass per snapshot
      batch_solve_time_ms: batched forward pass / N snapshots
      postproc_time_ms:    mask overrides + cpu transfer + line flow calc
      total_time_ms:       solve + postproc (single-snapshot end-to-end)
    """
    if device is None:
        device = torch.device('cpu')
    
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    n_snaps = len(dataset)
    print(f"Dataset: {n_snaps} snapshots, {n_warmup} warmup, batch_size={batch_size}")
    
    for ri, run in enumerate(runs):
        model = run['model']
        model.to(device)
        model.eval()
        
        ptdf_params = run.get('_ptdf_params', None)
        max_buses   = run.get('_max_buses', 0)
        
        # ── Phase 1: Per-snapshot timing ──
        solve_times = []
        postproc_times = []
        
        with torch.no_grad():
            for i in range(n_snaps + n_warmup):
                idx = i % n_snaps
                data = dataset[idx].to(device)
                net_idx, _ = dataset._index[idx]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)
                
                # Level 1: pure forward pass
                t0 = time.perf_counter()
                out = model(data)
                if isinstance(out, tuple):
                    out = out[0]
                t_solve = time.perf_counter() - t0
                
                # Level 2: post-processing
                t1 = time.perf_counter()
                x = data.x
                bus_type = x[:, 0]
                pv_mask    = (bus_type == 1)
                slack_mask = (bus_type == 2)
                
                vmag_pred = out[:, 0].clone()
                vang_pred = out[:, 1].clone()
                vmag_pred[pv_mask]    = x[pv_mask, 5]
                vmag_pred[slack_mask] = x[slack_mask, 5]
                vang_pred[slack_mask] = x[slack_mask, 6]
                
                vmag_np = vmag_pred.cpu().numpy()
                vang_np = vang_pred.cpu().numpy()
                calculate_line_flows(test_networks[net_idx], vmag_np, vang_np)
                t_post = time.perf_counter() - t1
                
                if i >= n_warmup:
                    solve_times.append(t_solve)
                    postproc_times.append(t_post)
        
        # ── Phase 2: Batched timing ──
        all_data = []
        with torch.no_grad():
            for i in range(n_snaps):
                data = dataset[i].to(device)
                net_idx, _ = dataset._index[i]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)
                all_data.append(data)
            
            # Warmup
            for _ in range(3):
                _ = batched_gnn_forward(model, all_data[:batch_size])
            
            # Timed batched forward in chunks
            batch_times = []
            batch_counts = []
            for start in range(0, n_snaps, batch_size):
                chunk = all_data[start:start + batch_size]
                t0 = time.perf_counter()
                _ = batched_gnn_forward(model, chunk)
                batch_times.append(time.perf_counter() - t0)
                batch_counts.append(len(chunk))
        
        batch_ms_per_snap = sum(batch_times) * 1000 / sum(batch_counts)
        
        # ── Store results ──
        solve_ms    = np.mean(solve_times) * 1000
        postproc_ms = np.mean(postproc_times) * 1000
        total_ms    = solve_ms + postproc_ms
        
        run['test_metrics']['solve_time_ms']       = solve_ms
        run['test_metrics']['postproc_time_ms']     = postproc_ms
        run['test_metrics']['total_time_ms']        = total_ms
        run['test_metrics']['batch_solve_time_ms']  = batch_ms_per_snap
        
        tag = run.get('tag', run.get('physics_label', '?'))
        print(f"  Run {ri:2d}: {tag:30s} | single={solve_ms:.2f}ms  batch={batch_ms_per_snap:.3f}ms  post={postproc_ms:.2f}ms  total={total_ms:.2f}ms")
    
    print(f"\nBackfilled timing for {len(runs)} runs.")

def backfill_line_flow_metrics(
    runs: list,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    device=None,
):
    """
    Backfill line_flow_mae and line_flow_q0_mae into runs that are missing them.
    Requires live model objects in run['model'] and solved test_networks.
    """
    if device is None:
        device = torch.device('cpu')

    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    n_snaps = len(dataset)

    for ri, run in enumerate(runs):
        tm = run.get("test_metrics", {})
        if "line_flow_mae" in tm and not np.isnan(tm["line_flow_mae"]):
            print(f"  Run {ri}: already has line_flow_mae={tm['line_flow_mae']:.6f}, skipping")
            continue

        model = run['model']
        model.to(device)
        model.eval()

        ptdf_params = run.get('_ptdf_params', None)
        max_buses   = run.get('_max_buses', 0)

        lflow_p_errors = []
        lflow_q_errors = []

        with torch.no_grad():
            for i in range(n_snaps):
                data = dataset[i].to(device)
                net_idx, t_idx = dataset._index[i]
                network = test_networks[net_idx]
                snapshot = network.snapshots[t_idx]

                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)

                out = model(data)
                if isinstance(out, tuple):
                    out = out[0]

                x = data.x
                bus_type   = x[:, 0]
                pv_mask    = (bus_type == 1)
                slack_mask = (bus_type == 2)

                vmag_pred = out[:, 0].clone()
                vang_pred = out[:, 1].clone()
                vmag_pred[pv_mask]    = x[pv_mask, 5]
                vmag_pred[slack_mask] = x[slack_mask, 5]
                vang_pred[slack_mask] = x[slack_mask, 6]

                vmag_np = vmag_pred.cpu().numpy()
                vang_np = vang_pred.cpu().numpy()

                try:
                    flows = calculate_line_flows(network, vmag_np, vang_np)

                    if not network.lines_t.p0.empty and snapshot in network.lines_t.p0.index:
                        true_p0 = network.lines_t.p0.loc[snapshot].values
                        pred_p0 = np.array(flows["p0"])
                        lflow_p_errors.append(np.mean(np.abs(pred_p0 - true_p0)))

                    if not network.lines_t.q0.empty and snapshot in network.lines_t.q0.index:
                        true_q0 = network.lines_t.q0.loc[snapshot].values
                        pred_q0 = np.array(flows["q0"])
                        lflow_q_errors.append(np.mean(np.abs(pred_q0 - true_q0)))
                except Exception as e:
                    if not lflow_p_errors:
                        print(f"    [Run {ri}] line flow calc failed: {e}")

        tm["line_flow_mae"]    = float(np.mean(lflow_p_errors)) if lflow_p_errors else float("nan")
        tm["line_flow_q0_mae"] = float(np.mean(lflow_q_errors)) if lflow_q_errors else float("nan")
        run["test_metrics"] = tm

        tag = run.get("tag", run.get("physics_label", "?"))
        print(f"  Run {ri}: {tag:30s} | p0_mae={tm['line_flow_mae']:.6f}  q0_mae={tm['line_flow_q0_mae']:.6f}  ({len(lflow_p_errors)}/{n_snaps} snaps)")

    print(f"\nBackfilled line flow metrics for {len(runs)} runs.")

def backfill_all(
    runs: list,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    n_warmup: int = 5,
    batch_size: int = 64,
    device=None,
    force: bool = False,
):
    """
    Single-pass backfill of timing + line flow metrics into sweep runs.

    Stores in test_metrics:
      solve_time_ms, postproc_time_ms, total_time_ms, batch_solve_time_ms,
      line_flow_mae, line_flow_q0_mae
    
    Skips runs that already have all keys unless force=True.
    """
    ALL_KEYS = {"solve_time_ms", "postproc_time_ms", "total_time_ms",
                "batch_solve_time_ms", "line_flow_mae", "line_flow_q0_mae"}

    if device is None:
        device = torch.device('cpu')

    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    n_snaps = len(dataset)
    print(f"Dataset: {n_snaps} snapshots, {n_warmup} warmup, batch_size={batch_size}")

    for ri, run in enumerate(runs):
        tm = run.setdefault("test_metrics", {})

        # Skip if all keys already present (and not NaN for line flows)
        if not force and ALL_KEYS.issubset(tm.keys()):
            lf = tm.get("line_flow_mae", float("nan"))
            if not np.isnan(lf):
                print(f"  Run {ri}: already complete, skipping (use force=True to redo)")
                continue

        model = run['model']
        model.to(device)
        model.eval()

        ptdf_params = run.get('_ptdf_params', None)
        max_buses   = run.get('_max_buses', 0)

        solve_times, postproc_times = [], []
        lflow_p_errors, lflow_q_errors = [], []
        # STSI 26.04.07: if ptdf_params is None, we can't augment edge_attr with learned PTDF rows, which may cause errors or degraded performance for models that expect those features. We check for this and skip timing if it looks like a mismatch.
        # Guard: skip learned_edge runs whose PTDF params were lost on save/load
        if ptdf_params is None:
            first_conv = model.convs[0]
            if hasattr(first_conv, 'lin_edge') and first_conv.lin_edge is not None:
                le = first_conv.lin_edge
                expected_edge_dim = getattr(le, 'in_channels', None) or getattr(le, 'in_features', None)
                sample_edge_dim = dataset[0].edge_attr.shape[1]
                if expected_edge_dim is not None and expected_edge_dim > sample_edge_dim:
                    tag = run.get("tag", run.get("physics_label", "?"))
                    print(f"  Run {ri}: SKIPPED — learned_edge PTDF params not available "
                          f"(model expects {expected_edge_dim} edge feats, data has {sample_edge_dim}) | {tag}")
                    continue
        # ── Phase 1: Per-snapshot timing + line flows ──
        with torch.no_grad():
            for i in range(n_snaps + n_warmup):
                idx = i % n_snaps
                data = dataset[idx].to(device)
                net_idx, t_idx = dataset._index[idx]
                network  = test_networks[net_idx]
                snapshot = network.snapshots[t_idx]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)

                # — Solve (timed) —
                t0 = time.perf_counter()
                out = model(data)
                if isinstance(out, tuple):
                    out = out[0]
                t_solve = time.perf_counter() - t0

                # — Post-processing (timed) —
                t1 = time.perf_counter()
                x = data.x
                bus_type   = x[:, 0]
                pv_mask    = (bus_type == 1)
                slack_mask = (bus_type == 2)

                vmag_pred = out[:, 0].clone()
                vang_pred = out[:, 1].clone()
                vmag_pred[pv_mask]    = x[pv_mask, 5]
                vmag_pred[slack_mask] = x[slack_mask, 5]
                vang_pred[slack_mask] = x[slack_mask, 6]

                vmag_np = vmag_pred.cpu().numpy()
                vang_np = vang_pred.cpu().numpy()

                flows = calculate_line_flows(network, vmag_np, vang_np)
                t_post = time.perf_counter() - t1

                # Only record after warmup
                if i >= n_warmup:
                    solve_times.append(t_solve)
                    postproc_times.append(t_post)

                    # Line flow errors (always, not just after warmup — but we gate on i>=n_warmup)
                    try:
                        if not network.lines_t.p0.empty and snapshot in network.lines_t.p0.index:
                            true_p0 = network.lines_t.p0.loc[snapshot].values
                            pred_p0 = np.array(flows["p0"])
                            lflow_p_errors.append(np.mean(np.abs(pred_p0 - true_p0)))

                        if not network.lines_t.q0.empty and snapshot in network.lines_t.q0.index:
                            true_q0 = network.lines_t.q0.loc[snapshot].values
                            pred_q0 = np.array(flows["q0"])
                            lflow_q_errors.append(np.mean(np.abs(pred_q0 - true_q0)))
                    except Exception as e:
                        if not lflow_p_errors:
                            print(f"    [Run {ri}] line flow error: {e}")

        # ── Phase 2: Batched timing ──
        all_data = []
        with torch.no_grad():
            for i in range(n_snaps):
                data = dataset[i].to(device)
                net_idx, _ = dataset._index[i]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)
                all_data.append(data)

            # Warmup
            for _ in range(3):
                _ = batched_gnn_forward(model, all_data[:batch_size])

            batch_times, batch_counts = [], []
            for start in range(0, n_snaps, batch_size):
                chunk = all_data[start:start + batch_size]
                t0 = time.perf_counter()
                _ = batched_gnn_forward(model, chunk)
                batch_times.append(time.perf_counter() - t0)
                batch_counts.append(len(chunk))

        batch_ms_per_snap = sum(batch_times) * 1000 / max(1, sum(batch_counts))

        # ── Store everything ──
        solve_ms    = float(np.mean(solve_times)) * 1000
        postproc_ms = float(np.mean(postproc_times)) * 1000

        tm["solve_time_ms"]       = solve_ms
        tm["postproc_time_ms"]    = postproc_ms
        tm["total_time_ms"]       = solve_ms + postproc_ms
        tm["batch_solve_time_ms"] = batch_ms_per_snap
        tm["line_flow_mae"]       = float(np.mean(lflow_p_errors)) if lflow_p_errors else float("nan")
        tm["line_flow_q0_mae"]    = float(np.mean(lflow_q_errors)) if lflow_q_errors else float("nan")

        tag = run.get("tag", run.get("physics_label", "?"))
        print(
            f"  Run {ri}: {tag:30s} | "
            f"single={solve_ms:.2f}ms  batch={batch_ms_per_snap:.3f}ms  "
            f"post={postproc_ms:.2f}ms  total={solve_ms+postproc_ms:.2f}ms | "
            f"p0_mae={tm['line_flow_mae']:.6f}  q0_mae={tm['line_flow_q0_mae']:.6f}"
        )

    print(f"\nBackfilled timing + line flows for {len(runs)} runs.")



## Comparison & Solver Baselines

In [ ]:
def evaluate_gnn_on_test_set(
    model,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnom_share: bool = None,  # None = auto-detect from model.node_embedding.in_features
    device=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None,
    max_buses: int = 0,
    ptdf_offset: int = 0,
    # [STSI260502]: batch_size for batched timing measurement
    batch_size: int = 64,
    # [STSI260502]: flag to enable dangling bus postprocessing
    postprocess_dangling: bool = True,
    verbose: bool = False,
) -> dict:
    """
    Evaluate GNN on test networks snapshot-by-snapshot.

    KEY FIX (2026-03-25): After inference, override known variables with ground
    truth inputs — otherwise scatter plots show model predictions for variables
    the model was never asked to predict (Vmag/Vang at slack/PV), making results
    appear much worse than they are.

    Override rules (mirror of _masked_mse_loss):
        PV    → vmag_eval[pv]    = x[pv,    5]  (known Vmag)
        Slack → vmag_eval[slack] = x[slack,  5]  (known Vmag = v_set)
        Slack → vang_eval[slack] = x[slack,  6]  (known Vang = 0)

    STSI 26.04.07: When ptdf_params is not None (learned_edge mode), augments
    data.edge_attr with learned PTDF rows before model forward pass so the
    model receives the same edge feature width it was trained with.

    [STSI260502]: Added multi-level timing:
      solve_time_ms:       pure forward pass per snapshot
      postproc_time_ms:    mask overrides + cpu transfer + line flow calc
      total_time_ms:       solve + postproc (single-snapshot end-to-end)
      batch_solve_time_ms: batched forward pass amortized per snapshot
    Dataset construction and data.to(device) excluded from all timers.

    Returns dict with MAE/RMSE for Vmag, Vang, P, Q + line flow MAE + timing.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
    )

    vmag_errors, vang_errors, p_errors, q_errors = [], [], [], []
    vmag_true_all, vang_true_all, p_true_all, q_true_all = [], [], [], []

    lflow_errors = []
    lflow_q0_errors = []
    all_true_flows, all_pred_flows = [], []
    all_true_q0_flows, all_pred_q0_flows = [], []
    n_snapshots = 0

    # [STSI260502]: per-snapshot timing lists (solve and postproc measured separately)
    solve_times = []
    postproc_times = []
    # [STSI260502]: collect prepared data for batched timing later
    all_data_for_batch = []

    with torch.no_grad():
        for i in range(len(dataset)):
            data = dataset[i].to(device)

            # STSI 26.04.07: augment edge_attr for learned_edge models
            net_idx, _ = dataset._index[i]
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset, device)

            # [STSI260502]: save a copy for batched timing (before forward modifies anything)
            all_data_for_batch.append(data.clone())

            # [STSI260502]: Level 1 — time pure forward pass only
            t_solve_start = time.perf_counter()
            node_pred, *_ = model(data, return_embeddings=True)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_solve = time.perf_counter() - t_solve_start

            # [STSI260502]: Level 2 — time post-processing (masks + line flows)
            t_post_start = time.perf_counter()

            # ── Extract masks ─────────────────────────────────────────────────
            slack_mask = data.slack_mask.bool()
            pv_mask    = data.pv_mask.bool()
            pq_mask    = data.pq_mask.bool()
            
            # ── Dangling bus post-processing ──────────────────────────────────
            if postprocess_dangling:
                node_pred = postprocess_dangling_buses(node_pred, data)
            
            # ── Raw predictions ───────────────────────────────────────────────
            vmag_eval = node_pred[:, 0].clone()
            vang_eval = node_pred[:, 1].clone()
            p_eval    = node_pred[:, 2].clone()
            q_eval    = node_pred[:, 3].clone()

            # ── Override known variables with ground-truth inputs ─────────────
            # PV: Vmag is known (set by voltage regulator)
            if pv_mask.any():
                vmag_eval[pv_mask] = data.x[pv_mask, 5]

            # Slack: both Vmag and Vang=0 are known
            if slack_mask.any():
                vmag_eval[slack_mask] = data.x[slack_mask, 5]
                vang_eval[slack_mask] = data.x[slack_mask, 6]  # should be ~0

            # PQ: P and Q are known inputs
            if pq_mask.any():
                p_eval[pq_mask] = data.x[pq_mask, 3]
                q_eval[pq_mask] = data.x[pq_mask, 4]

            # PV: P is known input
            if pv_mask.any():
                p_eval[pv_mask] = data.x[pv_mask, 3]

            # ── Ground truth ──────────────────────────────────────────────────
            vmag_true = data.y[:, 0]
            vang_true = data.y[:, 1]
            p_true    = data.y[:, 2]
            q_true    = data.y[:, 3]

            vmag_errors.append((vmag_eval - vmag_true).abs().mean().item())
            vang_errors.append((vang_eval - vang_true).abs().mean().item())
            p_errors.append((p_eval - p_true).abs().mean().item())
            q_errors.append((q_eval - q_true).abs().mean().item())
            vmag_true_all.append(vmag_true.detach())
            vang_true_all.append(vang_true.detach())
            p_true_all.append(p_true.detach())
            q_true_all.append(q_true.detach())
            n_snapshots += 1

            # ── Step 4 (partial): line flow MAE ──────────────────────────────
            if test_networks is not None and hasattr(dataset, '_index') and i < len(dataset._index):
                try:
                    net_idx, t_idx = dataset._index[i]
                    network  = test_networks[net_idx]
                    snapshot = network.snapshots[t_idx]
                    if not network.lines_t.p0.empty:
                        true_flows = network.lines_t.p0.loc[snapshot].values
                        vmag_np = vmag_eval.cpu().numpy()
                        vang_np = vang_eval.cpu().numpy()
                        flows = calculate_line_flows(network, vmag_np, vang_np)
                        pred_flows = np.array(flows["p0"])  # already in p.u.
                        err = np.abs(pred_flows - true_flows).mean()
                        lflow_errors.append(err)
                        all_true_flows.append(true_flows)
                        all_pred_flows.append(pred_flows)
                    if not network.lines_t.q0.empty:
                        true_q0 = network.lines_t.q0.loc[snapshot].values
                        pred_q0 = np.array(flows["q0"])  # already in p.u.
                        lflow_q0_errors.append(np.abs(pred_q0 - true_q0).mean())
                        all_true_q0_flows.append(true_q0)
                        all_pred_q0_flows.append(pred_q0)
                except Exception as _lfe:
                    if not lflow_errors:  # only warn once
                        print(f"[line_flow_mae] skipped: {_lfe}")

            # [STSI260502]: end post-processing timer (includes masks + line flows)
            t_post = time.perf_counter() - t_post_start
            solve_times.append(t_solve)
            postproc_times.append(t_post)

    # [STSI260502]: Phase 2 — batched timing measurement
    # Warmup batched forward
    n_batch_warmup = 3
    with torch.no_grad():
        if all_data_for_batch:
            for _ in range(n_batch_warmup):
                _ = batched_gnn_forward(model, all_data_for_batch[:batch_size])
            
            batch_times = []
            batch_counts = []
            for start in range(0, len(all_data_for_batch), batch_size):
                chunk = all_data_for_batch[start:start + batch_size]
                t0 = time.perf_counter()
                _ = batched_gnn_forward(model, chunk)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                batch_times.append(time.perf_counter() - t0)
                batch_counts.append(len(chunk))
            batch_ms_per_snap = sum(batch_times) * 1000 / max(1, sum(batch_counts))
        else:
            batch_ms_per_snap = float('nan')

    metrics = {
        "vmag_mae":      float(np.mean(vmag_errors)),
        "vang_mae":      float(np.mean(vang_errors)),
        "p_mae":         float(np.mean(p_errors)),
        "q_mae":         float(np.mean(q_errors)),
        "vmag_rmse":     float(np.sqrt(np.mean([e**2 for e in vmag_errors]))),
        "vang_rmse":     float(np.sqrt(np.mean([e**2 for e in vang_errors]))),
        "vmag_true_std": float(torch.cat(vmag_true_all).std().item()),
        "vang_true_std": float(torch.cat(vang_true_all).std().item()),
        "p_true_std":    float(torch.cat(p_true_all).std().item()),
        "q_true_std":    float(torch.cat(q_true_all).std().item()),
    }
    if lflow_errors:
        metrics["line_flow_mae"] = float(np.mean(lflow_errors))
    if lflow_q0_errors:
        metrics["line_flow_q0_mae"] = float(np.mean(lflow_q0_errors))

    if all_true_flows:
        raw = {
            "p0_true": np.concatenate(all_true_flows),
            "p0_pred": np.concatenate(all_pred_flows),
        }
        if all_true_q0_flows:
            raw["q0_true"] = np.concatenate(all_true_q0_flows)
            raw["q0_pred"] = np.concatenate(all_pred_q0_flows)
        metrics["line_flow_raw"] = raw
    if verbose:     
        print("----------------\n Evaluation Metrics (known variables overridden) ---------------")
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                print(f"  {k:20s}: {v:.6f}")

    # [STSI260502]: monolithic timing kept for backward compat
    inference_time_s = sum(solve_times) + sum(postproc_times)
    metrics["inference_time_s"] = inference_time_s
    metrics["inference_time_per_snapshot_ms"] = (
        1000.0 * inference_time_s / max(1, n_snapshots)
    )
    
    # [STSI260502]: new multi-level timing metrics
    solve_ms    = float(np.mean(solve_times)) * 1000
    postproc_ms = float(np.mean(postproc_times)) * 1000
    metrics["solve_time_ms"]       = solve_ms
    metrics["postproc_time_ms"]    = postproc_ms
    metrics["total_time_ms"]       = solve_ms + postproc_ms
    metrics["batch_solve_time_ms"] = batch_ms_per_snap
    if verbose:
        print(f"\n  --- Timing breakdown (per snapshot) ---")
        print(f"  {'solve_time_ms':20s}: {solve_ms:.3f}")
        print(f"  {'postproc_time_ms':20s}: {postproc_ms:.3f}")
        print(f"  {'total_time_ms':20s}: {solve_ms + postproc_ms:.3f}")
        print(f"  {'batch_solve_time_ms':20s}: {batch_ms_per_snap:.3f}")

    return metrics

def postprocess_dangling_buses(node_pred, data, p_thresh=1e-4):
    """
    Fix predictions for dangling buses (degree-1 PQ buses with P=Q=0).
    Physics: no current flows → Vmag/Vang must equal neighbor's values.
    Modifies vmag (col 0) and vang (col 1) only; leaves P, Q untouched.
    """
    pred = node_pred.clone()
    x = data.x          # [N,7]: [is_slack, is_pv, is_pq, P, Q, Vmag, Vang]
    ei = data.edge_index  # [2, E]
    N = x.shape[0]

    # Degree (edges appear in both directions in PyG undirected graphs)
    deg = torch.zeros(N, dtype=torch.long, device=x.device) # check for degree-1 buses
    deg.scatter_add_(0, ei[0], torch.ones(ei.shape[1], dtype=torch.long, device=x.device)) # count outgoing edges

    dangling = (deg == 1) & (x[:, 2] > 0.5) & (x[:, 3].abs() < p_thresh) & (x[:, 4].abs() < p_thresh)   # degree-1 PQ buses with P≈0 and Q≈0
    if not dangling.any(): # early exit if no dangling buses
        return pred 

    for i in dangling.nonzero(as_tuple=True)[0].tolist(): # for each dangling bus
        nbrs = ei[1][ei[0] == i]   # check which bus it's connected to (should be exactly one neighbor in an undirected graph)
        nbrs = torch.cat([nbrs, ei[0][ei[1] == i]])  # check both directions to be safe
        nbrs = nbrs[nbrs != i] # exclude self-loop if present
        if len(nbrs) == 0: # 
            continue
        j = nbrs[0].item()  # neighbor index (should be only one)
        pred[i, 0] = pred[j, 0]  # vmag ← neighbour # STSI 26.05.0: this is a simple heuristic fix for dangling buses — we copy the neighbor's Vmag and Vang predictions, since a dangling bus with no load should have the same voltage as its single neighbor. This is a post-processing step that modifies the model's raw predictions to enforce a known physical constraint, and should improve accuracy for these edge cases.
        pred[i, 1] = pred[j, 1]  # vang ← neighbour
    return pred

def evaluate_dc_baseline(
    test_networks,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Run PyPSA linear (DC) power flow on all test-set snapshots and compute
    the same MAE metrics as evaluate_gnn_on_test_set.

    DC capabilities:
        vmag:          flat 1.0 p.u. (DC assumption) — error at PQ buses
        vang:          solved by lpf()               — error at PQ+PV buses
        p injections:  lossless balance              — error at Slack bus
        q injections:  zero (DC cannot compute Q)    — error at PV+Slack buses
        line_flow_mae: p0 from lpf() vs AC p0        — valid comparison
        line_flow_q0:  not computed by DC            — reported as nan

    Returns a dict compatible with evaluate_gnn_on_test_set output.
    """
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    index_map = {(ni, ti): k for k, (ni, ti) in enumerate(dataset._index)}

    vmag_errors, vang_errors, p_errors, q_errors = [], [], [], []
    lflow_errors = []
    n_snapshots = 0
    t_solve = 0.0

    for net_idx, network in enumerate(test_networks):
        if verbose and net_idx % 20 == 0:
            print(f"  [DC] network {net_idx + 1}/{len(test_networks)}")

        net_copy = network.copy()
        t0 = time.time()
        try:
            net_copy.lpf()
        except Exception as e:
            if verbose:
                print(f"  [DC] lpf() failed on network {net_idx}: {e}")
            continue
        t_solve += time.time() - t0

        buses = network.buses.index

        for t_idx, snap in enumerate(network.snapshots):
            ds_idx = index_map.get((net_idx, t_idx))
            if ds_idx is None:
                continue

            data       = dataset[ds_idx]
            slack_mask = data.slack_mask.bool().numpy()
            pv_mask    = data.pv_mask.bool().numpy()
            pq_mask    = data.pq_mask.bool().numpy()
            x_np       = data.x.numpy()
            y_np       = data.y.numpy()

            vmag_true = y_np[:, 0]
            vang_true = y_np[:, 1]
            p_true    = y_np[:, 2]
            q_true    = y_np[:, 3]

            # DC predictions --------------------------------------------------
            vmag_dc = np.ones(len(buses))                                      # flat 1.0 p.u.
            vang_dc = net_copy.buses_t.v_ang.loc[snap].reindex(buses).values   # from DC solve
            p_dc    = x_np[:, 3].copy()                                        # given P inputs
            q_dc    = np.zeros(len(buses))                                     # DC: Q=0

            # DC slack P: lossless power balance
            if slack_mask.any():
                p_dc[slack_mask] = -p_true[~slack_mask].sum() / slack_mask.sum()

            # Apply same known-variable overrides as evaluate_gnn_on_test_set -
            vmag_dc[pv_mask]    = x_np[pv_mask,    5]   # PV: Vmag given
            vmag_dc[slack_mask] = x_np[slack_mask, 5]   # Slack: Vmag given
            vang_dc[slack_mask] = x_np[slack_mask, 6]   # Slack: Vang=0 given
            p_dc[pq_mask]       = x_np[pq_mask,    3]   # PQ: P given
            q_dc[pq_mask]       = x_np[pq_mask,    4]   # PQ: Q given
            p_dc[pv_mask]       = x_np[pv_mask,    3]   # PV: P given

            vmag_errors.append(np.mean(np.abs(vmag_dc - vmag_true)))
            vang_errors.append(np.mean(np.abs(vang_dc - vang_true)))
            p_errors.append(np.mean(np.abs(p_dc    - p_true)))
            q_errors.append(np.mean(np.abs(q_dc    - q_true)))
            n_snapshots += 1

            # DC line flows (p0 only — q0 is zero in DC) ----------------------
            if not network.lines_t.p0.empty and snap in network.lines_t.p0.index \
                    and snap in net_copy.lines_t.p0.index:
                true_p0 = network.lines_t.p0.loc[snap].values
                dc_p0   = net_copy.lines_t.p0.loc[snap].values
                # PyPSA p0 is in MW; normalise to p.u. using sn_mva
                sn_mva  = getattr(network, "sn_mva", 100.0)
                lflow_errors.append(np.mean(np.abs(dc_p0 / sn_mva - true_p0)))

    metrics = {
        "vmag_mae":    float(np.mean(vmag_errors)) if vmag_errors else float("nan"),
        "vang_mae":    float(np.mean(vang_errors)) if vang_errors else float("nan"),
        "vang_rmse":   float(np.sqrt(np.mean([e**2 for e in vang_errors]))) if vang_errors else float("nan"),
        "p_mae":       float(np.mean(p_errors))    if p_errors    else float("nan"),
        "q_mae":       float(np.mean(q_errors))    if q_errors    else float("nan"),
        "line_flow_mae":     float(np.mean(lflow_errors)) if lflow_errors else float("nan"),
        "line_flow_q0_mae":  float("nan"),   # DC cannot compute reactive line flows
        "inference_time_s":               t_solve,
        "inference_time_per_snapshot_ms": 1000.0 * t_solve / max(1, n_snapshots),
        "n_snapshots":  n_snapshots,
    }

    if verbose:
        print(
            f"\n[DC baseline] {n_snapshots} snapshots — "
            f"pure solve time {t_solve:.2f}s "
            f"({metrics['inference_time_per_snapshot_ms']:.3f} ms/snap)"
        )
        for k in ["vmag_mae", "vang_mae", "p_mae", "q_mae", "line_flow_mae"]:
            print(f"  {k:25s}: {metrics[k]:.5f}")

    return metrics

In [ ]:
#=============================================================================
# SECTION 10b: COMPARISON WITH OTHER SOLVERS
#=============================================================================

#import numpy as np
#import time
from pypower.api import runpf, ppoption
from pypower.idx_bus import (BUS_I, BUS_TYPE, PD, QD, GS, BS,
                              BUS_AREA, VM, VA, BASE_KV, ZONE, VMAX, VMIN)
from pypower.idx_gen import (GEN_BUS, PG, QG, QMAX, QMIN, VG,
                              MBASE, GEN_STATUS, PMAX, PMIN)
from pypower.idx_brch import (F_BUS, T_BUS, BR_R, BR_X, BR_B, RATE_A,
                               RATE_B, RATE_C, TAP, SHIFT, BR_STATUS, PF, QF, PT, QT)


def pypsa_to_ppc(network, snapshot, base_mva=1.0):
    """Convert a PyPSA network at a single snapshot to a PyPower ppc dict.
    
    Assumes network values are already in p.u. (v_nom=1.0, line lengths=0).
    All outputs remain in p.u. with the given base_mva.
    
    Returns:
        ppc, bus_names, bus_idx, line_names, trafo_names
    """
    bus_names = network.buses.index.tolist()
    bus_idx = {name: i + 1 for i, name in enumerate(bus_names)}  # 1-indexed
    n = len(bus_names)

    # Bus type sets from generators
    slack_buses = set(network.generators[network.generators['control'] == 'Slack']['bus'])
    pv_buses = set(network.generators[network.generators['control'] == 'PV']['bus'])

    # Aggregate loads per bus for this snapshot
    lp = network.loads_t.p_set.loc[snapshot]
    lq = network.loads_t.q_set.loc[snapshot]
    bus_p, bus_q = {}, {}
    for lname, ldata in network.loads.iterrows():
        b = ldata['bus']
        bus_p[b] = bus_p.get(b, 0.0) + lp.get(lname, 0.0)
        bus_q[b] = bus_q.get(b, 0.0) + lq.get(lname, 0.0)

    # --- Bus matrix (13 cols) ---
    bus_mat = np.zeros((n, 13))
    for i, bname in enumerate(bus_names):
        btype = 3 if bname in slack_buses else (2 if bname in pv_buses else 1)
        bus_mat[i, BUS_I]    = i + 1
        bus_mat[i, BUS_TYPE] = btype
        bus_mat[i, PD]       = bus_p.get(bname, 0.0)
        bus_mat[i, QD]       = bus_q.get(bname, 0.0)
        bus_mat[i, VM]       = 1.0
        bus_mat[i, VA]       = 0.0
        bus_mat[i, BASE_KV]  = network.buses.loc[bname, 'v_nom']
        bus_mat[i, VMAX]     = 1.1
        bus_mat[i, VMIN]     = 0.9

    # Get solved generator outputs for this snapshot
    gen_p_solved, gen_q_solved = {}, {}
    if len(network.generators_t.p) > 0:
        row_gp = network.generators_t.p.loc[snapshot]
        for gname in network.generators.index:
            if gname in row_gp.index:
                gen_p_solved[gname] = float(row_gp[gname])
    if hasattr(network.generators_t, 'q') and len(network.generators_t.q) > 0:
        row_gq = network.generators_t.q.loc[snapshot]
        for gname in network.generators.index:
            if gname in row_gq.index:
                gen_q_solved[gname] = float(row_gq[gname])

    # --- Gen matrix (10 cols) ---
    gen_rows = []
    for gname, gdata in network.generators.iterrows():
        b = gdata['bus']
        vg = float(network.buses_t.v_mag_pu.loc[snapshot, b])
        row = np.zeros(10)
        row[GEN_BUS]    = bus_idx[b]
        row[VG]         = vg
        row[MBASE]      = base_mva
        row[GEN_STATUS] = 1
        row[QMAX] = 9999.0;  row[QMIN] = -9999.0
        row[PMAX] = 9999.0;  row[PMIN] = -9999.0
        if gdata['control'] != 'Slack':
            row[PG] = gen_p_solved.get(gname, 0.0)  # PV: fixed P, pypower solves Q
        if gdata['control'] == 'PQ':
            row[QG] = gen_q_solved.get(gname, 0.0)  # PQ: both fixed
        gen_rows.append(row)

    gen_mat = np.array(gen_rows)

    # --- Branch matrix — lines ---
    branch_rows = []
    line_names = network.lines.index.tolist()
    for lname, line in network.lines.iterrows():
        row = np.zeros(13)
        row[F_BUS]     = bus_idx[line['bus0']]
        row[T_BUS]     = bus_idx[line['bus1']]
        row[BR_R]      = line['r']
        row[BR_X]      = line['x']
        row[BR_B]      = line['b']
        row[RATE_A]    = line['s_nom']
        row[BR_STATUS] = 1
        row[TAP]       = 0.0   # 0 → treated as 1.0 by pypower
        branch_rows.append(row)

    # --- Branch matrix — transformers (use raw r/x, not r_pu/x_pu) ---
    trafo_names = network.transformers.index.tolist()
    for tname, tr in network.transformers.iterrows():
        row = np.zeros(13)
        row[F_BUS]     = bus_idx[tr['bus0']]
        row[T_BUS]     = bus_idx[tr['bus1']]
        row[BR_R]      = tr['r']
        row[BR_X]      = tr['x']
        row[BR_B]      = tr.get('b', 0.0)
        row[RATE_A]    = tr['s_nom']
        row[TAP]       = tr['tap_ratio'] if tr['tap_ratio'] != 0 else 1.0
        row[SHIFT]     = tr['phase_shift']
        row[BR_STATUS] = 1
        branch_rows.append(row)

    branch_mat = np.array(branch_rows)

    ppc = {'version': '2', 'baseMVA': base_mva,
           'bus': bus_mat, 'gen': gen_mat, 'branch': branch_mat}

    return ppc, bus_names, bus_idx, line_names, trafo_names


def evaluate_pypower_baseline(networks, snapshots=None, base_mva=1.0, verbose=True):
    """Run PyPower AC PF on all networks/snapshots, compare to PyPSA ground truth.
    
    All comparisons are in p.u.  Returns dict compatible with scatter plot functions:
      vmag_mae, vang_mae, p_mae, q_mae, line_flow_mae, line_flow_q0_mae,
      inference_time_per_snapshot_ms, n_snapshots, n_converged, n_failed
    """
    ppopt = ppoption(VERBOSE=0, OUT_ALL=0)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    for network in networks:
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        # Check which time-series exist in ground truth
        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for snap in snaps:
            # --- Convert & ----
            ppc, _, bus_idx, _, _ = pypsa_to_ppc(network, snap, base_mva)
            # --- Solve (timed) ---
            t0 = time.perf_counter()
            result, success = runpf(ppc, ppopt)
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not success:
                n_failed += 1
                continue
            n_converged += 1

            # Sort result buses by index
            res_bus = result['bus'][np.argsort(result['bus'][:, BUS_I].astype(int))]

            # --- Voltage comparison (p.u. / radians) ---
            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values
            pp_vmag = res_bus[:, VM]
            pp_vang = np.deg2rad(res_bus[:, VA])

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # --- Bus P/Q injection (p.u.) ---
            if has_bus_p or has_bus_q:
                pp_gen = result['gen']
                pp_bus_pgen = np.zeros(n_buses)
                pp_bus_qgen = np.zeros(n_buses)
                for g in range(pp_gen.shape[0]):
                    bidx = int(pp_gen[g, GEN_BUS]) - 1
                    pp_bus_pgen[bidx] += pp_gen[g, PG]
                    pp_bus_qgen[bidx] += pp_gen[g, QG]
                if has_bus_p:
                    pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                    pp_p_inj = pp_bus_pgen - res_bus[:, PD]
                    all_p_err.extend(np.abs(pypsa_p - pp_p_inj))
                if has_bus_q:
                    pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                    pp_q_inj = pp_bus_qgen - res_bus[:, QD]
                    all_q_err.extend(np.abs(pypsa_q - pp_q_inj))

            # --- Line flows (p.u.) — lines only, first n_lines branches ---
            if has_line_p:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                all_line_p_err.extend(np.abs(pypsa_lp - result['branch'][:n_lines, PF]))
            if has_line_q:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                all_line_q_err.extend(np.abs(pypsa_lq - result['branch'][:n_lines, QF]))

    total = n_converged + n_failed
    metrics = {
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_q0_mae': float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"PyPower AC baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics

def evaluate_fdlf_baseline(networks, snapshots=None, base_mva=1.0, verbose=True):
    """Run PyPower Fast Decoupled (XB) PF on all networks/snapshots.
    Identical to evaluate_pypower_baseline but uses PF_ALG=2 (FDLF-XB).
    """
    ppopt = ppoption(VERBOSE=0, OUT_ALL=0, PF_ALG=2)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    for network in networks:
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for snap in snaps:
            ppc, _, bus_idx, _, _ = pypsa_to_ppc(network, snap, base_mva)

            t0 = time.perf_counter()
            result, success = runpf(ppc, ppopt)
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not success:
                n_failed += 1
                continue
            n_converged += 1

            res_bus = result['bus'][np.argsort(result['bus'][:, BUS_I].astype(int))]

            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values
            pp_vmag = res_bus[:, VM]
            pp_vang = np.deg2rad(res_bus[:, VA])

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            if has_bus_p or has_bus_q:
                pp_gen = result['gen']
                pp_bus_pgen = np.zeros(n_buses)
                pp_bus_qgen = np.zeros(n_buses)
                for g in range(pp_gen.shape[0]):
                    bidx = int(pp_gen[g, GEN_BUS]) - 1
                    pp_bus_pgen[bidx] += pp_gen[g, PG]
                    pp_bus_qgen[bidx] += pp_gen[g, QG]
                if has_bus_p:
                    pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                    pp_p_inj = pp_bus_pgen - res_bus[:, PD]
                    all_p_err.extend(np.abs(pypsa_p - pp_p_inj))
                if has_bus_q:
                    pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                    pp_q_inj = pp_bus_qgen - res_bus[:, QD]
                    all_q_err.extend(np.abs(pypsa_q - pp_q_inj))

            if has_line_p:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                all_line_p_err.extend(np.abs(pypsa_lp - result['branch'][:n_lines, PF]))
            if has_line_q:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                all_line_q_err.extend(np.abs(pypsa_lq - result['branch'][:n_lines, QF]))

    total = n_converged + n_failed
    metrics = {
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_q0_mae': float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"FDLF (XB) baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics

def evaluate_pandapower_baseline(networks, snapshots=None, base_mva=1.0, verbose=True):
    """Run pandapower AC NR PF on all networks/snapshots.
    Path: pypsa_to_ppc → from_ppc (untimed) → runpp (timed).
    With baseMVA=1.0, pandapower MW/MVAr values equal p.u. numerically.
    """
    import pandapower as pdp
    from pandapower.converter import from_ppc
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    for network in networks:
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for snap in snaps:
            # Conversion (untimed — same overhead as any solver needing its own format)
            ppc, _, _, _, _ = pypsa_to_ppc(network, snap, base_mva)
            try:
                pp_net = from_ppc(ppc, f_hz=50.0)
            except Exception as e:
                n_failed += 1
                if verbose and n_failed <= 3:
                    print(f"  [pandapower] from_ppc failed: {e}")
                continue

            # Solve (timed)
            t0 = time.perf_counter()
            try:
                pdp.runpp(pp_net, algorithm='nr', init='flat', numba=True)
            except Exception as e:
                n_failed += 1
                if verbose and n_failed <= 3:
                    print(f"  [pandapower] runpp failed: {e}")
                continue
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not pp_net.converged:
                n_failed += 1
                continue
            n_converged += 1

            # Results — baseMVA=1 so MW == p.u. numerically
            pp_vmag = pp_net.res_bus.vm_pu.values[:n_buses]
            pp_vang = np.deg2rad(pp_net.res_bus.va_degree.values[:n_buses])

            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # Bus P/Q: res_bus.p_mw is net injection (gen-load) in MW (= p.u. @ base=1)
            if has_bus_p:
                pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                pp_p = -pp_net.res_bus.p_mw.values[:n_buses]
                all_p_err.extend(np.abs(pypsa_p - pp_p))
            if has_bus_q:
                pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                pp_q = -pp_net.res_bus.q_mvar.values[:n_buses]
                all_q_err.extend(np.abs(pypsa_q - pp_q))

            # Line flows
            n_res_lines = len(pp_net.res_line)
            if has_line_p and n_res_lines >= n_lines:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                pp_lp = pp_net.res_line.p_from_mw.values[:n_lines]
                all_line_p_err.extend(np.abs(pypsa_lp - pp_lp))
            if has_line_q and n_res_lines >= n_lines:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                pp_lq = pp_net.res_line.q_from_mvar.values[:n_lines]
                all_line_q_err.extend(np.abs(pypsa_lq - pp_lq))

    total = n_converged + n_failed
    metrics = {
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_q0_mae': float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"pandapower NR baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics

# wrapper to run all the baselines and build ref_single and ref_batch dicts for scatter plots
def build_ref_baselines(networks, n_sample=20, ref_single=None, ref_batch=None):
    """
    Build (or incrementally fill) ref_single and ref_batch baseline timing dicts.

    Parameters
    ----------
    networks  : full list of networks (train+val+test); test split reconstructed internally
    n_sample  : number of test networks to time (default 20)
    ref_single : existing dict to fill; keys already present are skipped
    ref_batch  : existing dict to fill; keys already present are skipped

    Returns
    -------
    (ref_single, ref_batch)
    """
    if ref_single is None:
        ref_single = {}
    if ref_batch is None:
        ref_batch = {}

    _, _, test_nets = reconstruct_test_split(networks)
    sample = test_nets[:n_sample]
    print(f"Using {len(sample)} test networks for baseline timing.")

    # single-snapshot baselines
    if "PyPower AC" not in ref_single:
        print("Running PyPower AC...")
        ref_single["PyPower AC"] = evaluate_pypower_baseline(sample)
    if "PyPSA single" not in ref_single:
        print("Running PyPSA single...")
        ref_single["PyPSA single"] = evaluate_pypsa_timing_single(sample)
    if "FDLF" not in ref_single:
        print("Running FDLF...")
        ref_single["FDLF"] = evaluate_fdlf_baseline(sample)
    if "pandapower" not in ref_single:
        print("Running pandapower...")
        ref_single["pandapower"] = evaluate_pandapower_baseline(sample)

    # batch baselines
    if "PyPSA batch" not in ref_batch:
        print("Running PyPSA batch...")
        ref_batch["PyPSA batch"] = evaluate_pypsa_timing(sample)
    if "DC lpf" not in ref_batch:
        print("Running DC lpf...")
        ref_batch["DC lpf"] = evaluate_dc_baseline(sample)

    print("\nref_single timings (ms/snap):")
    for k, v in ref_single.items():
        print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")
    print("ref_batch timings (ms/snap):")
    for k, v in ref_batch.items():
        print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")

    return ref_single, ref_batch

## Hyperparameter Sweep Helpers

In [ ]:
# Hyperparameter helpers

def _get_run_styles(runs):
    # [STSI100526] Removed physics_mode/n_node_features (dead); added conv_mode/drop_rate
    varying = [
        k for k in ("physics_label", "weight_ptdf",
                    "head_mode", "conv_mode", "drop_rate",
                    "batch_size", "lr", "conv_type", "y_matrix_source")
        if len({str(x.get(k)) for x in runs}) > 1
    ]
    if not varying:
        varying = ["physics_label"]

    dim_values = {}
    for k in varying:
        seen = {}
        for r in runs:
            v = str(r.get(k, ""))
            if v not in seen:
                seen[v] = len(seen)
        dim_values[k] = seen

    styles = {}
    for r in runs:
        rid = id(r)
        c_idx  = dim_values[varying[0]][str(r.get(varying[0], ""))] if len(varying) >= 1 else 0
        ls_idx = dim_values[varying[1]][str(r.get(varying[1], ""))] if len(varying) >= 2 else 0
        mk_idx = dim_values[varying[2]][str(r.get(varying[2], ""))] if len(varying) >= 3 else 0

        styles[rid] = (
            _COLOR_CYCLE[c_idx  % len(_COLOR_CYCLE)],
            _LINE_STYLES[ls_idx % len(_LINE_STYLES)],
            _MARKERS[mk_idx     % len(_MARKERS)],
        )
    return styles

def build_legend_label(r, runs):
    # [STSI100526] Removed physics_mode/n_node_features (dead); added conv_mode/drop_rate
    varying = [
        k for k in [
            "physics_label",
            "weight_ptdf",
            "ptdf_loss_mode",
            "ptdf_branch_mode",
            "ptdf_alpha",
            "head_mode",
            "conv_mode",
            "drop_rate",
            "batch_size",
            "lr",
            "conv_type",
            "y_matrix_source",
            "warmup_epochs",
            "hidden_dim",
            "num_layers",
        ]
        if len({str(x.get(k)) for x in runs}) > 1
    ]

    if not varying:
        varying = ["physics_label"]

    parts = []

    # [STSI100526] Removed physics_mode label (always-on)
    if "physics_label" in varying:
        parts.append(r.get("physics_label", ""))

    if "weight_ptdf" in varying:
        parts.append(f"ptdf={r.get('weight_ptdf')}")
    if "ptdf_loss_mode" in varying:
        parts.append(f"mode={r.get('ptdf_loss_mode')}")
    if "ptdf_branch_mode" in varying:
        parts.append(f"br={r.get('ptdf_branch_mode')}")
    if "ptdf_alpha" in varying and r.get("ptdf_loss_mode") == "mixed":
        parts.append(f"a={r.get('ptdf_alpha')}")

    if "y_matrix_source" in varying:
        parts.append(f"y={r.get('y_matrix_source')}")
    if "warmup_epochs" in varying:
        parts.append(f"wu={r.get('warmup_epochs')}")
    if "hidden_dim" in varying:
        parts.append(f"h={r.get('hidden_dim')}")
    if "num_layers" in varying:
        parts.append(f"L={r.get('num_layers')}")
    if "batch_size" in varying:
        parts.append(f"bs={r.get('batch_size')}")
    if "lr" in varying:
        parts.append(f"lr={r.get('lr'):.0e}")
    if "conv_type" in varying:
        parts.append(r.get("conv_type", ""))
    # [STSI100526] Synced with Training: head_mode/conv_mode/drop_rate (removed n_node_features)
    if "head_mode" in varying:
        parts.append(f"hm={r.get('head_mode', 'standard')}")
    if "conv_mode" in varying:
        parts.append(f"cm={r.get('conv_mode', 'old')}")
    if "drop_rate" in varying:
        parts.append(f"dr={r.get('drop_rate', 0.1)}")

    return " | ".join(parts)


RUN_LABEL_COMPACT_THRESHOLD = 8
RUN_LABEL_INDEX_THRESHOLD = 14


def build_compact_legend_label(r, runs):
    full_label = build_legend_label(r, runs)
    full_parts = [part for part in full_label.split(" | ") if part]
    known_prefixes = (
        "ptdf=", "mode=", "br=", "a=", "y=", "wu=",
        "h=", "L=", "bs=", "lr=", "hm=", "cm=", "dr=",
    )
    parts = []

    if full_parts and not full_parts[0].startswith(known_prefixes):
        parts.append(full_parts[0])

    for part in full_parts:
        if part.startswith("ptdf="):
            parts.append(part.replace("ptdf=", "pt"))
        elif part.startswith("mode="):
            parts.append(part.replace("mode=", "pm="))
        elif part.startswith(("br=", "a=", "y=", "wu=", "h=", "L=", "bs=", "lr=", "hm=", "cm=", "dr=")):
            parts.append(part)

    if not parts:
        return full_label

    deduped = []
    seen = set()
    for part in parts:
        if part and part not in seen:
            deduped.append(part)
            seen.add(part)
    return " | ".join(deduped)


def resolve_run_label_policy(
    runs,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    index_prefix="R",
):
    if label_mode == "auto":
        if len(runs) >= index_threshold:
            active_mode = "index"
        elif len(runs) >= compact_threshold:
            active_mode = "compact"
        else:
            active_mode = "full"
    else:
        active_mode = label_mode

    if active_mode == "compact":
        compact_candidates = [build_compact_legend_label(r, runs) for r in runs]
        if len(set(compact_candidates)) < len(compact_candidates):
            active_mode = "full"

    entries = []
    for i, r in enumerate(runs, start=1):
        full_label = build_legend_label(r, runs)
        compact_label = build_compact_legend_label(r, runs)
        short_label = f"{index_prefix}{i}"
        plot_label = {
            "full": full_label,
            "compact": compact_label,
            "index": short_label,
        }[active_mode]
        mapping_label = f"{short_label}: {full_label}"
        legend_label = mapping_label if active_mode == "index" else plot_label
        entries.append({
            "run": r,
            "full_label": full_label,
            "compact_label": compact_label,
            "short_label": short_label,
            "plot_label": plot_label,
            "legend_label": legend_label,
            "mapping_label": mapping_label,
        })
    return active_mode, entries


def _index_label_entries(label_entries):
    return {id(entry["run"]): entry for entry in label_entries}


def _resolve_annotation_mode(annotation_mode, label_mode):
    if annotation_mode != "auto":
        return annotation_mode
    return {
        "full": "all",
        "compact": "first-per-run",
        "index": "none",
    }[label_mode]

def plot_all_runs_training_curves(
    runs,
    title_prefix="All runs",
    marker_every=10,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Plot train/val loss for all runs.
    - Color    separates the first varying hyperparameter (e.g. physics_label).
    - Linestyle separates the second varying hyperparameter (e.g. weight_ptdf).
    - Marker   separates the third varying hyperparameter (e.g. batch_size / lr).
    - marker_every: place a marker every N epochs (keeps plots readable).
    """
    styles = _get_run_styles(runs)
    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        h = r["history"]
        color, ls, mk = styles[id(r)]
        n = len(h["train_total"])
        markevery = max(1, n // marker_every) if mk else None

        ax1.plot(
            h["train_total"], label=label, alpha=0.85,
            color=color, linestyle=ls, marker=mk,
            markevery=markevery, markersize=5,
        )
        ax2.plot(
            h["val_total"], label=label, alpha=0.85,
            color=color, linestyle=ls, marker=mk,
            markevery=markevery, markersize=5,
        )

    for ax, title, ylabel in [
        (ax1, f"{title_prefix} — Training Loss", "Training Loss"),
        (ax2, f"{title_prefix} — Validation Loss", "Validation Loss"),
    ]:
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=8, ncol=max(1, len(runs) // 8))
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_loss_components(
    runs,
    title_prefix="Loss Components",
    marker_every=10,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Plot train/val loss for each component (MSE, Physics, Angle-ref, PTDF)
    in a 2×2 grid of subplots, with each subplot showing train (solid) and
    val (dashed) curves for all runs.
    """
    styles = _get_run_styles(runs)
    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    components = [
        ("mse", "MSE Loss"),
        ("phys", "Physics Loss"),
        ("angle_ref", "Angle-Ref Loss"),
        ("ptdf", "PTDF Loss"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(24, 14))
    fig.suptitle(f"{title_prefix} (solid=train, dashed=val)", fontsize=13)
    axes = axes.flatten()

    for ax, (key, comp_title) in zip(axes, components):
        for r in runs:
            label = label_entry_by_run[id(r)]["legend_label"]
            h = r["history"]
            train_key = f"train_{key}"
            val_key = f"val_{key}"

            if train_key not in h or not h[train_key]:
                continue

            color, ls, mk = styles[id(r)]
            n = len(h[train_key])
            markevery = max(1, n // marker_every) if mk else None

            ax.plot(
                h[train_key], label=f"{label} (train)", alpha=0.85,
                color=color, linestyle="-", marker=mk,
                markevery=markevery, markersize=5,
            )
            ax.plot(
                h[val_key], label=f"{label} (val)", alpha=0.85,
                color=color, linestyle="--", marker=mk,
                markevery=markevery, markersize=5,
            )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(f"{title_prefix} — {comp_title}")
        ax.grid(True, alpha=0.3)

    from matplotlib.lines import Line2D
    run_handles = []
    for entry in label_entries:
        run = entry["run"]
        color, _, mk = styles[id(run)]
        run_handles.append(
            Line2D(
                [0], [0],
                color=color,
                linestyle="-",
                marker=mk if mk else None,
                markersize=5,
                label=entry["legend_label"],
            )
        )
    if run_handles:
        fig.legend(
            handles=run_handles,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=max(1, min(4, len(run_handles))),
            fontsize=8,
            frameon=True,
        )
        plt.tight_layout(rect=[0, 0.10, 1, 0.95])
    else:
        plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

def plot_all_runs_accuracy_metrics(
    runs,
    title_prefix="All runs",
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Bar chart comparison of test accuracy metrics across runs.
    Reads from r["test_metrics"]["overall_metrics"] for mean/std per metric.
    Metrics: vmag_mae, vang_rmse, p_mae, q_mae.
    """
    metrics = [
        ("vmag_mae", "Vmag MAE  [p.u.]"),
        ("vang_rmse", "Vang RMSE [rad]"),
        ("p_mae", "P MAE     [p.u.]"),
        ("q_mae", "Q MAE     [p.u.]"),
        ("line_flow_mae", "Line Flow MAE [p.u.]"),
        ("line_flow_q0_mae", "Line Flow Q0 MAE [p.u.]"),
    ]
    x = list(range(len(runs)))
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    styles = _get_run_styles(runs)
    colors = [styles[id(r)][0] for r in runs]
    tick_labels = [entry["plot_label"] for entry in label_entries]
    tick_rotation = 0 if active_label_mode == "index" else 30
    tick_ha = "center" if active_label_mode == "index" else "right"
    tick_fontsize = 9 if active_label_mode == "index" else 8

    fig, axes = plt.subplots(1, len(metrics), figsize=(max(16, 2.5 * len(metrics) + len(runs)), 6))
    fig.suptitle(f"{title_prefix} — Test Accuracy Metrics", fontsize=13)
    for ax, (metric_key, metric_label) in zip(axes, metrics):
        means, stds = [], []
        for r in runs:
            tm = r.get("test_metrics", {})
            val = tm.get(metric_key, float("nan"))
            if isinstance(val, dict):
                mean = float(val.get("mean", float("nan")))
                std = float(val.get("std", 0.0))
            else:
                mean = float(val)
                std = 0.0
            means.append(mean)
            stds.append(std)

        bars = ax.bar(x, means, yerr=stds, capsize=5, alpha=0.8, color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, rotation=tick_rotation, ha=tick_ha, fontsize=tick_fontsize)
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.grid(True, axis="y", alpha=0.3)

        valid_means = [m for m in means if m == m]
        top = max(valid_means) if valid_means else 1.0
        for bar, mean in zip(bars, means):
            if mean == mean:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + top * 0.02,
                    f"{mean:.4f}",
                    ha="center", va="bottom", fontsize=7,
                )

    if active_label_mode == "index":
        legend_lines = [
            plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                       label=label_entries[i]["mapping_label"])
            for i in range(len(runs))
        ]
        fig.legend(
            handles=legend_lines,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=max(1, min(3, len(runs))),
            fontsize=8,
            frameon=True,
        )
        plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    else:
        plt.tight_layout()
    plt.show()

def print_sweep_summary(runs_list, top_n=5):
    """
    Print a ranked summary table of hyperparameter sweep results.
    Shows top_n configurations by test loss.
    """
    sorted_runs = sorted(runs_list, key=lambda r: r["test_metrics"]["test_total"])
    # [STSI100526] Added head_mode/conv_mode columns to summary table
    print(f"{'Rank':<5}{'w_phys':<9}{'w_ptdf':<9}{'head':<14}{'conv':<14}"
          f"{'bs':<6}{'lr':<11}"
          f"{'Train':<11}{'Val':<11}{'Test':<11}{'Time(s)':<9}")
    print("-" * 120)
    for i, run in enumerate(sorted_runs[:top_n], 1):
        tm = run["test_metrics"]
        print(
            f"{i:<5}{run['weight_physics']:<9}{run['weight_ptdf']:<9}"
            f"{run.get('head_mode','standard'):<14}{run.get('conv_mode','old'):<14}"
            f"{run['batch_size']:<6}{run['lr']:<11.2e}"
            f"{run['final_train_loss']:<11.6f}{run['final_val_loss']:<11.6f}"
            f"{tm['test_total']:<11.6f}{run['training_time']:<9.1f}"
        )

def _sanitise_filename(s: str) -> str:
    """Replace characters illegal on Windows filesystems with safe alternatives."""
    for ch in r'\/:*?"<>|=':
        s = s.replace(ch, "-")
    return s

def load_sweep_results(pkl_path: str) -> list:
    """
    Load a full sweep result list from a pickle file.
    Returns the runs list as saved by save_sweep_results or run_hparam_sweep.
    """
    with open(pkl_path, "rb") as f:
        runs = pickle.load(f)
    print(f"Loaded {len(runs)} runs from {pkl_path}")
    return runs


def load_sweep_metrics(csv_path: str) -> pd.DataFrame:
    """
    Load just the metrics CSV (no models) for fast analysis.
    """
    df = pd.read_csv(csv_path)
    print(f"Loaded metrics for {len(df)} runs from {csv_path}")
    return df

def reconstruct_test_split(networks, seed=42, train_frac=0.70, val_frac=0.15):
    """
    Reproduce the train/val/test split that train_power_flow_gnn uses internally.
    Returns (train_networks, val_networks, test_networks).
    Use this only as a fallback for old runs that don't store test_networks.
    """
    import random as _random
    net_copy = list(networks)
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    _random.shuffle(net_copy)
    n = len(net_copy)
    n_train = int(train_frac * n)
    n_val = int((train_frac + val_frac) * n) - n_train
    train_nets = net_copy[:n_train]
    val_nets = net_copy[n_train : n_train + n_val]
    test_nets = net_copy[n_train + n_val :]
    return train_nets, val_nets, test_nets

def filter_runs(runs, criteria=None):
    """
    Filter runs such that for each column in criteria, the run value matches
    one of the allowed values for that column.

    - criteria is a dict: {col_name: [allowed_val1, allowed_val2, ...], ...}
    - columns combine with AND
    - values per column combine with OR
    """
    if criteria is None:
        criteria = {'weight_physics': ['0.1']}

    # convert to sets for efficiency
    crit_sets = {col: set(vals) for col, vals in criteria.items()}

    relevant_runs = []
    for r in runs:
        if all(r.get(col) in valset for col, valset in crit_sets.items()):
            relevant_runs.append(r)

    return relevant_runs

def check_hparam_results(runs,
                         networks=None,
                         training_networks=None,
                         title_prefix:str="Runs Analysis", 
                         factors=["weight_physics","weight_ptdf","head_mode","conv_mode"], 
                         interactions=[("weight_physics","conv_mode"),("weight_ptdf","conv_mode")],
                         test_network=None,
                         run_factorial_analysis=True,
                         ref_baselines: dict = None,
                         ref_baselines_batch: dict = None,   # [STSI260502] batch baseline group
                         physics_comparison_results: bool = True, # [STSI260504] whether to plot the timing vs accuracy scatter with marginal histograms
                         postprocess_dangling: bool = True,  # [STSI070526] dangling bus vmag/vang fix
                         use_pnom_share: Optional[bool] = None,  # None=auto from model.in_features; True/False forces mode
                         ):
    # ── Determine test networks for evaluation ───────────────────────────────
    eval_networks = networks  # explicit override, if given
    if eval_networks is None:
        # Try to use test_networks stored during training
        stored = runs[0].get("test_networks") if runs else None
        if stored is not None:
            eval_networks = stored
            print(f"[check_hparam_results] Using stored test split ({len(eval_networks)} networks)")
        elif training_networks is not None:
            # Reconstruct from the full network list + seed
            seed = runs[0].get("seed", 42) if runs else 42
            _, _, eval_networks = reconstruct_test_split(training_networks, seed=seed)
            print(f"[check_hparam_results] Reconstructed test split ({len(eval_networks)} networks)")

    _, label_entries = resolve_run_label_policy(runs)
    label_entry_by_run = _index_label_entries(label_entries)

    # ── Re-evaluate any runs missing line_flow_mae / line_flow_q0_mae ────────
    REQUIRED_METRICS = {"vmag_mae", "vang_rmse", "p_mae", "q_mae",
                            "line_flow_mae", "line_flow_q0_mae",
                            "vmag_true_std", "vang_true_std", "p_true_std", "q_true_std",
                            "inference_time_per_snapshot_ms",
                            # [STSI260502]: multi-level timing
                            "solve_time_ms", "postproc_time_ms", "total_time_ms",
                            "batch_solve_time_ms"}
        
        if eval_networks is not None:
            for r in runs:
                tm = r.get("test_metrics") or {}
                missing = REQUIRED_METRICS - set(tm.keys())
                if missing:
                    _lbl = label_entry_by_run[id(r)]["legend_label"]
                    print(f"[check_hparam_results] Re-evaluating '{_lbl}' "
                        f"— missing metrics: {missing}")
                    hparams_r     = r.get("hparams") or {}
                    ptdf_params_r = r.get("_ptdf_params") or hparams_r.get("ptdf_params", None)
                    max_buses_r   = r.get("_max_buses") or hparams_r.get("max_buses", 0)
                    ptdf_offset_r = r.get("_ptdf_offset") or hparams_r.get("ptdf_offset", 0)
                    try:
                        new_metrics = evaluate_gnn_on_test_set(
                            r["model"],
                            eval_networks,
                            use_edge_features=True,
                            use_pnom_share=use_pnom_share,
                            ptdf_params=ptdf_params_r,
                            max_buses=max_buses_r,
                            ptdf_offset=ptdf_offset_r,
                            postprocess_dangling=postprocess_dangling,
                        )
                        r["test_metrics"] = new_metrics
                    except RuntimeError as e:
                        print(f"  [WARNING] Re-evaluation skipped for '{_lbl}': {e}")


    # ── Training curves & accuracy metrics ───────────────────────────────────────

    plot_all_runs_training_curves(runs,title_prefix=title_prefix)
    plot_loss_components(runs, title_prefix=title_prefix)
    

    plot_all_runs_accuracy_metrics(runs,title_prefix=title_prefix)

    plot_inference_time_bar(runs, title_prefix=title_prefix)
    
    # Combined timing vs accuracy scatter (all baselines, log x-axis)
    _all_baselines = {**(ref_baselines or {}), **(ref_baselines_batch or {})}
    plot_timing_vs_accuracy(runs, x_metric="infer_ms_per_snap", y_metric="vmag_mae",
                            title_prefix=title_prefix,
                            ref_baselines=_all_baselines or None,
                            timing_key="both", x_axis_log=True)
    # ── Factorial analysis ────────────────────────────────────────────────────────
    results, coefs = None, None
    if run_factorial_analysis:
        results, coefs = factorial_analysis(
            runs=runs,
            factors=factors,
            interactions=interactions,
            response_metrics=["vmag_mae", "vang_rmse", "p_mae", "q_mae"],
        )
    
        if results:
            print(results["p_mae"].summary())

    # ── Per-network visual comparison (qualitative) ─────────────────────────────
    if test_network is not None:
        plot_single_network_comparisons(
            runs,
            test_network=test_network,
            title_prefix=title_prefix,
            use_pnom_share=use_pnom_share,
        )


    # Build models dict using shared label policy
    models_dict = {}
    model_summary = {}
    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        ptdf_params = r.get("_ptdf_params")
        max_buses   = r.get("_max_buses", 0)
        if ptdf_params is not None and max_buses > 0:
            models_dict[label] = {
                "model": r["model"],
                "ptdf_params": ptdf_params,
                "max_buses": max_buses,
            }
        else:
            models_dict[label] = r["model"]
        _tm = r.get("test_metrics") or {}
        model_summary[label] = {
            "test_metrics": {k: v for k, v in _tm.items() if k != "line_flow_raw"},
        }
    print("Model dict build")
   
    if eval_networks is not None:
        multi_net_metrics = evaluate_models_on_networks(
            models_dict,
            eval_networks,
            use_edge_features=True,
            use_pnom_share=use_pnom_share,
        )
        print("multi_net_metrics done!")
        for mnm_lbl, mnm_vals in multi_net_metrics.items():
            if mnm_lbl in model_summary:
                model_summary[mnm_lbl]["multi_net_metrics"] = mnm_vals
        plot_model_metrics_across_networks(
            models_dict,
            eval_networks,

            title_prefix=title_prefix,
            use_pnom_share=use_pnom_share,

        )        
        # ── Full test-set scatter & error histograms ────────────────────
        if physics_comparison_results:
            models_dict_full = {}
            for r in runs:
                lbl = label_entry_by_run[id(r)]["legend_label"]
                m = r["model"]
                ptdf_params_r = r.get("_ptdf_params")
                max_buses_r   = r.get("_max_buses", 0)
                if ptdf_params_r is not None and max_buses_r > 0:
                    models_dict_full[lbl] = {
                        "model": m,
                        "ptdf_params": ptdf_params_r,
                        "max_buses": max_buses_r,
                    }
                else:
                    models_dict_full[lbl] = m
            try:
                all_preds = _collect_test_set_predictions(models_dict_full, eval_networks, use_pnom_share=use_pnom_share)
                plot_full_test_scatter(all_preds, title_prefix=title_prefix)
                plot_error_histograms(all_preds, title_prefix=title_prefix)
                plot_errors_vs_operating_range(all_preds, n_bins=40, title_prefix=title_prefix)
                # ── Collect error stats per model ───────────────────────────
                col_names = ["V_mag (pu)", "V_ang (deg)", "P (MW)", "Q (MVAr)"]
                for es_lbl, es_data in all_preds.items():
                    error_stats = {}
                    for col_idx, cname in enumerate(col_names):
                        err = es_data["pred"][:, col_idx] - es_data["true"][:, col_idx]
                        ci_lo, ci_hi = np.percentile(err, [2.5, 97.5])
                        error_stats[cname] = {
                            "mae": float(np.mean(np.abs(err))),
                            "mean": float(np.mean(err)),
                            "std": float(np.std(err)),
                            "ci_lo": float(ci_lo),
                            "ci_hi": float(ci_hi),
                        }
                    if es_lbl in model_summary:
                        model_summary[es_lbl]["error_stats"] = error_stats
            except Exception as exc:
                print(f"[WARNING] scatter/histogram plots failed: {exc}")

        return results, coefs, multi_net_metrics, model_summary

In [ ]:
## =============================================================================
# SECTION 11b: Factorial analysis
# =============================================================================
_FACTOR_LABELS = {
    "n_buses":        "System size (n buses)",
    "bus_count_range": "Bus count range",
    "dataset_size":   "Dataset size (scenarios)",
    "weight_physics": "Physics loss weight",
    "weight_ptdf":    "PTDF loss weight",
    "num_layers":     "GNN layers",
    "w_phys":         "Physics loss weight",
    "ptdf_loss_mode": "PTDF loss mode",
    "physics_mode":   "Physics mode",
    "hidden_dim":     "Hidden dim",
    "batch_size":     "Batch size",
    "lr":             "Learning rate",
    "warmup_epochs":  "Warmup epochs",
    "vmag_mae":         "Vmag MAE [p.u.]",
    "vang_mae":         "Vang MAE [p.u.]",
    "vang_rmse":        "Vang RMSE [p.u.]",
    "p_mae":            "P MAE [p.u.]",
    "q_mae":            "Q MAE [p.u.]",
    "line_flow_mae":    "Line flow MAE [p.u.]",
    "line_flow_q0_mae": "Line flow Q0 MAE [p.u.]",
    "vmag_mae_norm":    "Vmag MAE / σ(true)",
    "vang_mae_norm":    "Vang MAE / σ(true)",
    "p_mae_norm":       "P MAE / σ(true)",
    "q_mae_norm":       "Q MAE / σ(true)",
}

_SYS_TO_BUSES = {
    "ieee9": 9, "ieee14": 14, "cigre14": 14,
    "ieee30": 30, "ieee57": 57, "ieee39": 39, "ieee118": 118,
}

def create_comparison_dataframe(runs_list):
    """Create a comprehensive comparison DataFrame from sweep results."""
    rows = []
    for r in runs_list:
        tm = r["test_metrics"]
        rows.append({
            "tag":           r["tag"],
            "w_phys":        r["weight_physics"],
            "w_ptdf":        r["weight_ptdf"],
            "bs":            r["batch_size"],
            "lr":            r["lr"],
            "train_loss":    r["final_train_loss"],
            "val_loss":      r["final_val_loss"],
            "test_total":    tm["test_total"],
            "test_mse":      tm["test_mse"],
            "test_physics":  tm["test_physics"],
            "test_ptdf":     tm["test_ptdf"],
            "training_time": r["training_time"],
        })
    return pd.DataFrame(rows)



def _get_factor_value(r, f):
    if f == "n_buses":
        v = r.get("n_buses")
        if v is None:
            bs = r.get("base_system") or (r.get("hparams") or {}).get("base_system", "")
            v = _SYS_TO_BUSES.get(str(bs).lower(), float("nan"))
        return float(v) if v is not None else float("nan")
    elif f == "dataset_size":
        v = r.get("dataset_size")
        if v is None:
            v = r.get("num_scenarios") or (r.get("hparams") or {}).get("num_scenarios")
        return float(v) if v is not None else float("nan")
    elif f == "max_buses":
        return float(r.get("_max_buses", 0))
    else:
        return r.get(f)


def _full_label(term):
    parts = term.split(":")
    translated = []
    for p in parts:
        if "[" in p:
            base, suffix = p.split("[", 1)
            translated.append(_FACTOR_LABELS.get(base, base) + " [" + suffix)
        else:
            translated.append(_FACTOR_LABELS.get(p, p))
    return " × ".join(translated)


def factorial_analysis(runs, response_metrics=None, factors=None, interactions=None):
    if response_metrics is None:
        response_metrics = ["vmag_mae", "vang_rmse", "p_mae", "q_mae"]
    if factors is None:
        factors = ["weight_physics", "q_residual_mode", "weight_ptdf", "ptdf_loss_mode"]

    records = []
    for r in runs:
        tm = r.get("test_metrics", {})
        rec = {}
        for m in response_metrics:
            rec[m] = float(r.get(m, tm.get(m, float("nan"))))
        for f in factors:
            rec[f] = _get_factor_value(r, f)
        records.append(rec)
    df_raw = pd.DataFrame(records)

    df = df_raw.copy()
    encoding    = {}
    factor_type = {}

    for f in factors:
        levels = sorted(df[f].dropna().unique(), key=lambda v: (str(type(v)), v))
        n = len(levels)
        is_numeric = all(isinstance(v, (int, float)) for v in levels)

        if f == "physics_mode":
            if n != 2:
                print(f"  WARNING: 'physics_mode' has {n} levels → expected 2; skipping.")
                continue
            lo, hi = sorted(levels)
            encoding[f] = {lo: -1, hi: +1}
            df[f] = df[f].map(encoding[f])
            factor_type[f] = "binary"
            print(f"  {f:30s} [binary]   : {str(lo):>10} → -1,  {str(hi):>10} → +1")
            continue

        if n == 2:
            lo, hi = levels[0], levels[1]
            encoding[f] = {lo: -1, hi: +1}
            df[f] = df[f].map(encoding[f])
            factor_type[f] = "binary"
            print(f"  {f:30s} [binary]   : {str(lo):>10} → -1,  {str(hi):>10} → +1")
        elif n >= 3 and is_numeric:
            factor_type[f] = "numeric"
            print(f"  {f:30s} [numeric]  : {levels} → as-is (linear trend)")
        elif n >= 3 and not is_numeric:
            df[f] = df[f].astype(str)
            factor_type[f] = "categorical"
            print(f"  {f:30s} [categoric]: {levels} → treatment coded, ref={levels[0]}")
        else:
            print(f"  WARNING: '{f}' has {n} levels and unknown type — skipping.")
            continue

    print()
    valid_factors = [f for f in factors if f in factor_type]
    if not valid_factors:
        print("[factorial_analysis] No varying factors — skipping OLS regression.")
        return {}, {}
    df = df.dropna(subset=valid_factors)

    if interactions is None:
        active_interactions = [
            (a, b)
            for i, a in enumerate(valid_factors)
            for b in valid_factors[i+1:]
        ]
    else:
        active_interactions = [
            (a, b) for a, b in interactions
            if a in valid_factors and b in valid_factors
        ]

    print("Active interactions:")
    for a, b in active_interactions:
        print(f"  {a} : {b}")
    print()

    rename  = {f: f"F{i}" for i, f in enumerate(valid_factors)}
    reverse = {v: k for k, v in rename.items()}
    df_model = df[valid_factors + response_metrics].rename(columns=rename)

    def _formula_term(f):
        fn = rename[f]
        return f"C({fn})" if factor_type[f] == "categorical" else fn

    mains  = " + ".join(_formula_term(f) for f in valid_factors)
    twoway = " + ".join(
        f"{_formula_term(a)}:{_formula_term(b)}"
        for a, b in active_interactions
    )
    formula_rhs = mains + (" + " + twoway if twoway else "")

    def _n_params(f):
        levels = sorted(df[f].dropna().unique(), key=lambda v: (str(type(v)), v))
        return len(levels) - 1 if factor_type[f] == "categorical" else 1

    n_main_params = sum(_n_params(f) for f in valid_factors)
    n_interaction_params = sum(
        _n_params(a) * _n_params(b) for a, b in active_interactions
    )
    n_params = 1 + n_main_params + n_interaction_params
    df_error = len(df_model) - n_params
    print(f"Runs: {len(df_model)}   Parameters: {n_params}   Error df: {df_error}")
    print(f"Formula RHS: {formula_rhs}")
    print()

    results   = {}
    all_coefs = {}

    def _restore(term):
        parts = term.split(":")
        restored = []
        for p in parts:
            p_clean = p.replace("C(", "").replace(")", "")
            base = p_clean.split("[")[0]
            suffix = ("[" + p_clean.split("[")[1]) if "[" in p_clean else ""
            restored.append(reverse.get(base, base) + suffix)
        return ":".join(restored)

    for metric in response_metrics:
        formula = f"{metric} ~ {formula_rhs}"
        model   = ols(formula, data=df_model).fit()
        results[metric] = model

        coef = model.params.copy()
        pval = model.pvalues.copy()
        coef.index = [_restore(t) for t in coef.index]
        pval.index = coef.index

        all_coefs[metric] = pd.DataFrame({
            "coefficient": coef,
            "p-value":     pval,
            "significant": pval.apply(lambda p: "***" if p < 0.001
                                       else ("**" if p < 0.01
                                       else ("*"  if p < 0.05 else ""))),
        })
        print("=" * 60)
        print(f"Response: {metric}   R²={model.rsquared:.4f}   Adj-R²={model.rsquared_adj:.4f}")
        print("=" * 60)
        print(all_coefs[metric].to_string())
        print()

    # ── Coefficient plot ──────────────────────────────────────────────────────
    n_metrics = len(response_metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(6 * n_metrics, 5))
    if n_metrics == 1:
        axes = [axes]

    for ax_idx, (ax, metric) in enumerate(zip(axes, response_metrics)):
        # BUG FIX: was `model.params` (last loop's model) — now correctly per-metric
        coef = results[metric].params.drop("Intercept")
        pval = results[metric].pvalues.drop("Intercept")
        coef.index = [_restore(t) for t in coef.index]
        pval.index = coef.index
        labels = [_full_label(lbl) for lbl in coef.index]
        colors = ["tomato" if p < 0.05 else "steelblue" for p in pval]
        ax.barh(labels, coef.values, color=colors, alpha=0.8)
        if ax_idx > 0:
            ax.tick_params(labelleft=False)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(
            f"{_FACTOR_LABELS.get(metric, metric)}\n"
            f"R²={results[metric].rsquared:.3f}  adj-R²={results[metric].rsquared_adj:.3f}"
        )
        ax.set_xlabel("Effect coefficient")
        ax.grid(True, axis="x", alpha=0.3)
        max_abs = np.nanmax(np.abs(coef.values)) if len(coef) > 0 else 1.0
        for i, (val, p) in enumerate(zip(coef.values, pval)):
            stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
            if stars:
                ax.text(val + np.sign(val) * max_abs * 0.03,
                        i, stars, va="center", fontsize=9, color="tomato")

    from matplotlib.patches import Patch
    fig.legend(
        handles=[
            Patch(facecolor="tomato",    label="p < 0.05 (significant)"),
            Patch(facecolor="steelblue", label="p ≥ 0.05 (not significant)"),
        ],
        loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.06)
    )
    plt.suptitle("Factorial effect coefficients", y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Factor-metric correlation heatmap ─────────────────────────────────────
    plot_factor_metric_heatmap(
        runs,
        response_metrics=response_metrics,
        factors=factors + ["n_buses", "dataset_size"],
    )

    return results, all_coefs


def plot_factor_metric_heatmap(runs, response_metrics=None, factors=None):
    if response_metrics is None:
        response_metrics = ["vmag_mae", "vang_rmse", "p_mae", "q_mae",
                            "line_flow_mae", "line_flow_q0_mae"]
    if factors is None:
        factors = ["n_buses", "dataset_size", "weight_physics",
                   "weight_ptdf", "num_layers", "w_phys"]

    _system_to_buses = {
        "ieee9": 9, "ieee14": 14, "ieee30": 30, "ieee57": 57,
        "ieee118": 118, "ieee39": 39, "cigre14": 14,
    }

    def _n_buses(r):
        bs = r.get("base_system") or (r.get("hparams") or {}).get("base_system", "")
        return _system_to_buses.get(str(bs).lower(), float("nan"))

    def _dataset_size(r):
        v = r.get("num_scenarios") or (r.get("hparams") or {}).get("num_scenarios")
        return float(v) if v is not None else float("nan")

    factor_accessors = {
        "n_buses":        _n_buses,
        "dataset_size":   _dataset_size,
        "weight_physics": lambda r: r.get("weight_physics"),
        "weight_ptdf":    lambda r: r.get("weight_ptdf"),
        "num_layers":     lambda r: r.get("num_layers") or (r.get("hparams") or {}).get("num_layers"),
        "w_phys":         lambda r: r.get("w_phys") or r.get("weight_physics"),
    }

    available_factors = []
    seen = set()
    for f in factors:
        if f not in seen and f in factor_accessors:
            available_factors.append(f)
            seen.add(f)

    records = []
    for r in runs:
        row = {}
        for f in available_factors:
            v = factor_accessors[f](r)
            try:
                row[f] = float(v) if v is not None else float("nan")
            except (TypeError, ValueError):
                row[f] = float("nan")
        tm = r.get("test_metrics") or {}
        for m in response_metrics:
            v = tm.get(m, float("nan"))
            if isinstance(v, dict):
                v = v.get("mean", float("nan"))
            row[m] = float(v) if v is not None else float("nan")
        records.append(row)

    df = pd.DataFrame(records)

    valid_factors = [
        f for f in available_factors
        if len(df[f].dropna()) >= 2 and df[f].dropna().std() > 0
    ]
    valid_metrics = [m for m in response_metrics if len(df[m].dropna()) >= 2]

    if not valid_factors or not valid_metrics:
        print("[plot_factor_metric_heatmap] Not enough data to compute correlations.")
        return

    from scipy import stats as _stats
    corr_matrix = np.full((len(valid_factors), len(valid_metrics)), np.nan)
    pval_matrix = np.full_like(corr_matrix, np.nan)

    for i, f in enumerate(valid_factors):
        for j, m in enumerate(valid_metrics):
            both = df[[f, m]].dropna()
            if len(both) >= 3:
                r_val, p_val = _stats.pearsonr(both[f], both[m])
                corr_matrix[i, j] = r_val
                pval_matrix[i, j] = p_val

    fig, ax = plt.subplots(
        figsize=(max(6, 1.4 * len(valid_metrics)), max(3, 0.9 * len(valid_factors)))
    )
    im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax, label="Pearson r")
    ax.set_xticks(range(len(valid_metrics)))
    ax.set_xticklabels(
        [_FACTOR_LABELS.get(m, m) for m in valid_metrics],
        rotation=30, ha="right", fontsize=9,
    )
    ax.set_yticks(range(len(valid_factors)))
    ax.set_yticklabels([_FACTOR_LABELS.get(f, f) for f in valid_factors], fontsize=9)
    ax.set_title("Factor–metric correlation heatmap (Pearson r)", fontsize=11)

    for i in range(len(valid_factors)):
        for j in range(len(valid_metrics)):
            r_val = corr_matrix[i, j]
            p_val = pval_matrix[i, j]
            if np.isnan(r_val):
                continue
            stars = ("***" if p_val < 0.001 else
                     "**"  if p_val < 0.01  else
                     "*"   if p_val < 0.05  else "")
            text_color = "white" if abs(r_val) > 0.6 else "black"
            ax.text(j, i, f"{r_val:.2f}{stars}",
                    ha="center", va="center", fontsize=8, color=text_color)

    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# SECTION 11c: CROSS-SYSTEM FACTORIAL ANALYSIS
# =============================================================================

_SYSTEM_TO_BUSES = {
    "ieee9": 9, "ieee14": 14, "cigre14": 14,
    "ieee30": 30, "ieee57": 57, "ieee39": 39, "ieee118": 118,
}


def multi_system_factorial_analysis(
    runs_by_system: dict,
    response_metrics=None,
    factors=None,
    interactions=None,
    normalization:str = "std",  # "std" to normalise MAE by true std, or "zscore", None for raw MAE
):
    """
    Factorial analysis across multiple train systems using each run's own
    same-system test metrics (no cross-testing).

    Injects 'n_buses' and 'dataset_size' into shallow-copied run dicts so
    factorial_analysis can treat them as ordinary numeric factors.

    Args:
        runs_by_system:  {"ieee9": runs_list, "cigre14": runs_list, ...}
        response_metrics, factors, interactions: passed to factorial_analysis.
            'n_buses', 'max_buses', and 'dataset_size' are always appended to factors.
    """
    if response_metrics is None:
        response_metrics = ["vmag_mae_norm", "vang_mae_norm", "p_mae_norm", "q_mae_norm"]

    if factors is None:
        factors = ["weight_physics", "weight_ptdf", "num_layers", "max_buses"]
    # Always include system-level factors, deduplicated
    extended_factors = factors + ["n_buses", "max_buses", "dataset_size"]
    seen = set()
    extended_factors = [f for f in extended_factors if not (f in seen or seen.add(f))]

    # ── Build augmented run list ──────────────────────────────────────────────
    combined_runs = []
    for sys_name, runs in runs_by_system.items():
        #n_buses = _SYSTEM_TO_BUSES.get(sys_name.lower(), float("nan"))
        for r in runs:
            r_aug = dict(r)   # shallow copy — don't mutate the original
            r_aug["_sys_name"] = sys_name   # tag for zscore grouping
            bs = (r.get("base_system") or "").lower()
            r_aug["n_buses"] = float(_SYSTEM_TO_BUSES.get(bs, float("nan")))
            #r_aug["n_buses"]      = n_buses
            r_aug["max_buses"]    = float(r.get("_max_buses", 0))
            r_aug["dataset_size"] = (
                r.get("num_scenarios")
                or r.get("dataset_size")
                or (r.get("hparams") or {}).get("num_scenarios", float("nan"))
            )
            combined_runs.append(r_aug)
    
    # ── Normalise MAE ─────────────────────────────────────────────────────────
    _MAE_PAIRS = [
        ("vmag_mae", "vmag_true_std"),
        ("vang_mae", "vang_true_std"),
        ("p_mae",    "p_true_std"),
        ("q_mae",    "q_true_std"),
    ]

    if normalization == "std":
        for r_aug in combined_runs:
            tm = r_aug.get("test_metrics", {})
            for mae_key, std_key in _MAE_PAIRS:
                mae = tm.get(mae_key, float("nan"))
                std = tm.get(std_key, float("nan"))
                r_aug[f"{mae_key}_norm"] = mae / std if std and not np.isnan(std) else float("nan")
        if any(m in response_metrics for m in ["vmag_mae_norm","vang_mae_norm","p_mae_norm","q_mae_norm"]):
            missing = sum(1 for r in combined_runs if np.isnan(r.get("vmag_mae_norm", float("nan"))))
            if missing:
                raise ValueError(
                    f"{missing}/{len(combined_runs)} runs are missing '*_true_std' in test_metrics. "
                    f"Re-run refresh_test_metrics_with_std(runs) first, or use normalization='zscore'."
                )

    elif normalization == "zscore":
        from collections import defaultdict
        for mae_key, _ in _MAE_PAIRS:
            groups = defaultdict(list)
            for r_aug in combined_runs:
                tm = r_aug.get("test_metrics", {})
                mae = tm.get(mae_key, float("nan"))
                if not np.isnan(mae):
                    groups[r_aug["_sys_name"]].append(mae)
            group_stats = {}
            for sys, vals in groups.items():
                if len(vals) < 2:
                    print(f"  WARNING: '{sys}' has only {len(vals)} run(s) for '{mae_key}' — zscore undefined.")
                    group_stats[sys] = (vals[0] if vals else float("nan"), float("nan"))
                else:
                    group_stats[sys] = (float(np.mean(vals)), float(np.std(vals, ddof=1)))
            for r_aug in combined_runs:
                tm = r_aug.get("test_metrics", {})
                mae = tm.get(mae_key, float("nan"))
                sys = r_aug["_sys_name"]
                mu, sigma = group_stats.get(sys, (float("nan"), float("nan")))
                if not np.isnan(mae) and not np.isnan(sigma) and sigma > 0:
                    r_aug[f"{mae_key}_norm"] = (mae - mu) / sigma
                else:
                    r_aug[f"{mae_key}_norm"] = float("nan")

    else:
        # normalization=None — copy raw MAE to *_norm keys for uniform downstream use
        for r_aug in combined_runs:
            tm = r_aug.get("test_metrics", {})
            for mae_key, _ in _MAE_PAIRS:
                r_aug[f"{mae_key}_norm"] = tm.get(mae_key, float("nan"))

    # Guard: Warn if any  factor has NaN values after augmentation
    for f in extended_factors:
        n_nan = sum(1 for r in combined_runs if np.isnan(r.get(f, float("nan"))))
        if n_nan:
            print(f"  WARNING: Factor '{f}' has {n_nan}/{len(combined_runs)} NaN values after augmentation.")
    print(f"multi_system_factorial_analysis: {len(combined_runs)} runs total "
          f"from {list(runs_by_system.keys())}")
    print(f"Factors: {extended_factors}")
    print(f"Responses: {response_metrics}\n")

    return factorial_analysis(
        runs=combined_runs,
        response_metrics=response_metrics,
        factors=extended_factors,
        interactions=interactions,
    )



def refresh_test_metrics_with_std(runs_by_system: dict, training_networks_by_system: dict, verbose=True):
    """
    Re-evaluates each run's model on its reconstructed test split to populate
    vmag_true_std, vang_true_std, p_true_std, q_true_std in test_metrics.

    Uses reconstruct_test_split with the seed stored in each run, matching
    exactly what train_power_flow_gnn used during training.

    Args:
        runs_by_system:              {"ieee9": runs_list, "cigre14": runs_list, ...}
        training_networks_by_system: {"ieee9": networks_ieee9_large, "cigre14": networks_cigre14_large, ...}
    """
    _STD_KEYS = ["vmag_true_std", "vang_true_std", "p_true_std", "q_true_std"]
    skipped, updated = 0, 0

    for sys_name, runs in runs_by_system.items():
        training_networks = training_networks_by_system.get(sys_name)
        if training_networks is None:
            print(f"  [{sys_name}] skip — no training_networks provided")
            skipped += len(runs)
            continue

        for i, r in enumerate(runs):
            model = r.get("model")
            if model is None:
                if verbose:
                    print(f"  [{sys_name}][{i}] skip — model not available")
                skipped += 1
                continue

            seed = r.get("seed", 42)
            _, _, test_nets = reconstruct_test_split(training_networks, seed=seed)

            if verbose:
                print(f"  [{sys_name}][{i}] seed={seed}, {len(test_nets)} test nets ...", end=" ", flush=True)

            hparams_r     = r.get("hparams") or {}
            ptdf_params_r = r.get("_ptdf_params") or hparams_r.get("ptdf_params", None)
            max_buses_r   = int(r.get("_max_buses") or hparams_r.get("max_buses", 0))
            ptdf_offset_r = int(r.get("_ptdf_offset") or hparams_r.get("ptdf_offset", 0))

            try:
                new_metrics = evaluate_gnn_on_test_set(
                    model=model,
                    test_networks=test_nets,
                    ptdf_params=ptdf_params_r,
                    max_buses=max_buses_r,
                    ptdf_offset=ptdf_offset_r,
                    postprocess_dangling=True,
                )
            except RuntimeError as e:
                print(f"[WARNING] skipped [{sys_name}][{i}]: {e}")
                skipped += 1
                continue

            for k in _STD_KEYS:
                r["test_metrics"][k] = new_metrics[k]

            if verbose:
                print(f"vmag_std={new_metrics.get('vmag_true_std', float('nan')):.4f}")
            updated += 1

    print(f"\nrefresh_test_metrics_with_std: {updated} updated, {skipped} skipped.")


## Visualisation Functions

In [ ]:
# =============================================================================
# SECTION 12: VISUALIZATION
# =============================================================================

def get_bus_type_label(network, bus):
    """Return a human-readable bus type label for plot titles."""
    connected_gens = network.generators[network.generators.bus == bus]
    if len(connected_gens) > 0:
        ctrl = connected_gens.iloc[0]["control"]
        if ctrl in ("Slack", "PV"):
            return ctrl
    return "PQ"


def plot_network_results(network, results_df=None):
    """
    Plot bus voltage and power results over time.
    If results_df is None, runs power flow first.
    """
    if results_df is None:
        network.pf(use_seed=True)
        buses = list(network.buses.index)
        snapshots = network.snapshots
        results_df = pd.DataFrame(
            index=snapshots,
            columns=pd.MultiIndex.from_product(
                [buses, ["P pu", "Q pu", "V pu", "Angle deg"]]
            ),
            dtype=float,
        )
        for t in snapshots:
            for bus in buses:
                results_df.loc[t, (bus, "P pu")]      = network.buses_t.p.loc[t, bus]
                results_df.loc[t, (bus, "Q pu")]    = network.buses_t.q.loc[t, bus]
                results_df.loc[t, (bus, "V pu")]      = network.buses_t.v_mag_pu.loc[t, bus]
                results_df.loc[t, (bus, "Angle deg")] = np.degrees(network.buses_t.v_ang.loc[t, bus])

    buses = results_df.columns.get_level_values(0).unique()
    n_buses = len(buses)
    fig, axs = plt.subplots(n_buses, 2, figsize=(14, 2.5 * n_buses), sharex=True)
    if n_buses == 1:
        axs = axs.reshape(1, -1)

    for i, bus in enumerate(buses):
        bus_label = get_bus_type_label(network, bus)
        axs[i, 0].plot(results_df.index, results_df[(bus, "V pu")],
                       marker="o", linewidth=2)
        axs[i, 0].set_ylabel("V (p.u.)", fontsize=11)
        axs[i, 0].set_title(f"Voltage - {bus} ({bus_label})", fontsize=11, fontweight="bold")
        axs[i, 0].grid(True, alpha=0.3)

        axs[i, 1].plot(results_df.index, results_df[(bus, "P pu")],
                       marker="o", linewidth=2, label="P pu")
        axs[i, 1].plot(results_df.index, results_df[(bus, "Q pu")],
                       marker="s", linewidth=2, label="Q pu")
        axs[i, 1].set_ylabel("Power", fontsize=11)
        axs[i, 1].set_title(f"Power - {bus} ({bus_label})", fontsize=11, fontweight="bold")
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)

    fig.text(0.5, 0.02, "Time Step", ha="center", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


def plot_line_results(network, results_df=None):
    """
    Plot line flow results over time.
    If results_df is None, runs power flow first.
    """
    if results_df is None:
        network.pf(use_seed=True)
        lines = list(network.lines.index)
        snapshots = network.snapshots
        results_df = pd.DataFrame(
            index=snapshots,
            columns=pd.MultiIndex.from_product(
                [lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
            ),
            dtype=float,
        )
        for t in snapshots:
            for line in lines:
                results_df.loc[t, (line, "P0 pu")]   = network.lines_t.p0.loc[t, line]
                results_df.loc[t, (line, "P1 pu")]   = network.lines_t.p1.loc[t, line]
                results_df.loc[t, (line, "Q0 pu")] = network.lines_t.q0.loc[t, line]
                results_df.loc[t, (line, "Q1 pu")] = network.lines_t.q1.loc[t, line]

    lines = results_df.columns.get_level_values(0).unique()
    n_lines = len(lines)
    fig, axs = plt.subplots(n_lines, 2, figsize=(14, 2.5 * n_lines), sharex=True)
    if n_lines == 1:
        axs = axs.reshape(1, -1)

    for i, line in enumerate(lines):
        from_bus = network.lines.loc[line, "bus0"]
        to_bus   = network.lines.loc[line, "bus1"]
        from_label = get_bus_type_label(network, from_bus)
        to_label   = get_bus_type_label(network, to_bus)
        line_name  = f"{line} ({from_bus}→{to_bus})"
        s_nom = network.lines.loc[line, "s_nom"]

        # Active Power P - Left subplot
        axs[i, 0].plot(results_df.index, results_df[(line, "P0 pu")],
                       marker="o", linewidth=2, label=f"P from {from_bus}")
        axs[i, 0].plot(results_df.index, -results_df[(line, "P1 pu")],
                       marker="s", linewidth=2, label=f"P to {to_bus}")
        axs[i, 0].axhline(y=s_nom,  color="r", linestyle="--", alpha=0.7, label=f"Snom {s_nom:.2f} pu")
        axs[i, 0].axhline(y=-s_nom, color="r", linestyle="--", alpha=0.7)
        axs[i, 0].set_ylabel("P (pu)", fontsize=11)
        axs[i, 0].set_title(f"Active Power - {line_name}", fontsize=11, fontweight="bold")
        axs[i, 0].legend(fontsize=9)
        axs[i, 0].grid(True, alpha=0.3)

        # Reactive Power Q - Right subplot
        axs[i, 1].plot(results_df.index, results_df[(line, "Q0 pu")],
                       marker="o", linewidth=2, label=f"Q from {from_bus}")
        axs[i, 1].plot(results_df.index, -results_df[(line, "Q1 pu")],
                       marker="s", linewidth=2, label=f"Q to {to_bus}")
        axs[i, 1].set_ylabel("Q (pu)", fontsize=11)
        axs[i, 1].set_title(f"Reactive Power - {line_name}", fontsize=11, fontweight="bold")
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)

    fig.text(0.5, 0.02, "Time Step", ha="center", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


def plot_transformer_results(network, results_df=None):
    """
    Plot transformer flow results separately.
    Similar to plot_line_results but for transformers.
    """
    if len(network.transformers) == 0:
        logger.warning("No transformers to plot")
        return

    # If results_df not provided, run power flow
    if results_df is None:
        network.pf(use_seed=True)
        trafos = list(network.transformers.index)
        snapshots = network.snapshots
        results_df = pd.DataFrame(
            index=snapshots,
            columns=pd.MultiIndex.from_product(
                [trafos, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
            ),
            dtype=float,
        )
        for t in snapshots:
            for trafo in trafos:
                results_df.loc[t, (trafo, "P0 pu")]   = network.transformers_t.p0.loc[t, trafo]
                results_df.loc[t, (trafo, "P1 pu")]   = network.transformers_t.p1.loc[t, trafo]
                results_df.loc[t, (trafo, "Q0 pu")] = network.transformers_t.q0.loc[t, trafo]
                results_df.loc[t, (trafo, "Q1 pu")] = network.transformers_t.q1.loc[t, trafo]

    n_trafos = len(network.transformers)
    fig, axs = plt.subplots(n_trafos, 2, figsize=(14, 2.5 * n_trafos), sharex=True)
    if n_trafos == 1:
        axs = axs.reshape(1, -1)

    for i, trafo in enumerate(network.transformers.index):
        from_bus = network.transformers.loc[trafo, "bus0"]
        to_bus   = network.transformers.loc[trafo, "bus1"]
        trafo_name = f"{trafo} ({from_bus}→{to_bus})"
        s_nom = network.transformers.loc[trafo, "s_nom"]

        # Active Power
        axs[i, 0].plot(results_df.index, results_df[(trafo, "P0 pu")],
                       marker="o", linewidth=2, label=f"P from {from_bus}")
        axs[i, 0].plot(results_df.index, -results_df[(trafo, "P1 pu")],
                       marker="s", linewidth=2, label=f"P to {to_bus}")
        axs[i, 0].axhline(y=s_nom,  color="r", linestyle="--", alpha=0.7, label=f"Snom {s_nom:.2f} pu")
        axs[i, 0].axhline(y=-s_nom, color="r", linestyle="--", alpha=0.7)
        axs[i, 0].set_ylabel("P (pu)", fontsize=11)
        axs[i, 0].set_title(f"Active Power - {trafo_name}", fontsize=11, fontweight="bold")
        axs[i, 0].legend(fontsize=9)
        axs[i, 0].grid(True, alpha=0.3)

        # Reactive Power
        axs[i, 1].plot(results_df.index, results_df[(trafo, "Q0 pu")],
                       marker="o", linewidth=2, label=f"Q from {from_bus}")
        axs[i, 1].plot(results_df.index, -results_df[(trafo, "Q1 pu")],
                       marker="s", linewidth=2, label=f"Q to {to_bus}")
        axs[i, 1].set_ylabel("Q (pu)", fontsize=11)
        axs[i, 1].set_title(f"Reactive Power - {trafo_name}", fontsize=11, fontweight="bold")
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)

    # Add common x-axis label
    fig.text(0.5, 0.02, "Time Step", ha="center", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()

def plot_test_metrics_barchart(df, metric="test_total", top_n=10, title=None):
    """
    Bar chart of a chosen test metric for the top_n configurations (sorted ascending).
    """
    df_sorted = df.sort_values(metric).head(top_n)
    labels = [
        f"w={row.w_phys}, pt={row.w_ptdf}, bs={row.bs}, lr={row.lr:.0e}"
        for _, row in df_sorted.iterrows()
    ]
    values = df_sorted[metric].values

    plt.figure(figsize=(12, 6))
    x = np.arange(len(values))
    plt.bar(x, values, alpha=0.8)
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(title or f"Top {top_n} configs by {metric}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# SECTION 12b: VISUALIZATION multiruns 
# ============================================================================= 
def predict_network_results_with_masks_multi(models_dict, network, use_edge_features=True, use_pnom_share=None):
    """
    Run GNN prediction for multiple models on one network.
    Returns dict: model_label -> (bus_results, line_results, prediction_masks)
    STSI260320: multi-model version for side-by-side comparison.
    STSI260407: models_dict values may be a bare model OR a dict with
    keys 'model', 'ptdf_params', 'max_buses' for learned_edge support.
    """
    all_results = {}
    for label, entry in models_dict.items():
        if isinstance(entry, dict):
            model       = entry["model"]
            ptdf_params = entry.get("ptdf_params")
            max_buses   = entry.get("max_buses", 0)
        else:
            model       = entry
            ptdf_params = None
            max_buses   = 0
        bus_res, line_res, masks = predict_network_results_with_masks(
            model, network, use_edge_features=use_edge_features,
            ptdf_params=ptdf_params, max_buses=max_buses,
            use_pnom_share=use_pnom_share,
        )
        all_results[label] = {"bus": bus_res, "line": line_res, "masks": masks}
    return all_results


def plot_comparison_results_with_masks(
    multi_results, network, true_bus_results=None, true_line_results=None,
    title_prefix="", snapshot_idx=0
):
    """
    Plot all models side-by-side against true PyPSA solution.
    multi_results: dict label -> {"bus": df, "line": df, "masks": df}
    true_bus_results: from model_results(network) — the PyPSA ground truth.
    STSI260320: overlays multiple models + truth on same axes.
    """
    buses = list(network.buses.index)
    props = [("V pu", "V"), ("Angle deg", "Angle"), ("P pu", "P"), ("Q pu", "Q")]
    n_models = len(multi_results)
    colors = plt.cm.tab10.colors

    # Use first model's masks for predicted/known coloring
    first_masks = list(multi_results.values())[0]["masks"]

    fig, axs = plt.subplots(
        len(buses), len(props),
        figsize=(4 * len(props), 2.5 * len(buses)),
        sharex=True
    )
    if len(buses) == 1:
        axs = axs.reshape(1, -1)

    snap = network.snapshots[snapshot_idx]

    for i, bus in enumerate(buses):
        bus_label = get_bus_type_label(network, bus)
        for j, (prop_name, mask_key) in enumerate(props):
            ax = axs[i, j]
            is_predicted = first_masks[mask_key][bus].any()

            # Plot true PyPSA solution
            if true_bus_results is not None and (bus, prop_name) in true_bus_results.columns:
                ax.plot(
                    true_bus_results.index,
                    true_bus_results[bus][prop_name],
                    color="black", linewidth=2.5, linestyle="-",
                    label="PyPSA (true)", zorder=10
                )

            # Plot each model
            for k, (label, res) in enumerate(multi_results.items()):
                df = res["bus"]
                if (bus, prop_name) not in df.columns:
                    continue
                style = "-" if is_predicted else "--"
                alpha = 0.85
                ax.plot(
                    df.index, df[bus][prop_name],
                    color=colors[k % len(colors)],
                    linestyle=style, linewidth=1.5, alpha=alpha,
                    label=label
                )

            pred_str = "predicted" if is_predicted else "known"
            ax.set_title(f"{bus} ({bus_label}) — {prop_name} [{pred_str}]",
                         fontsize=8, fontweight="bold" if is_predicted else "normal")
            ax.grid(True, alpha=0.3)
            if i == 0 and j == 0:
                ax.legend(fontsize=7, loc="upper right")

    fig.suptitle(f"{title_prefix} — Bus Results", fontsize=12, y=1.01)
    fig.text(0.5, 0.02, "Time Step", ha="center", fontsize=11)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show


def plot_predictions_comparison_per_node(
    multi_results, network, true_bus_results=None,
    title_prefix="", predicted_only=True
):
    """
    Scatter: predicted vs true per node, one panel per output variable, one figure per model.
    STSI260320: replaces single-model version.
    """
    buses = list(network.buses.index)
    props = [("V pu", "V", "Vmag [p.u.]"),
             ("Angle deg", "Angle", "Vang [deg]"),
             ("P pu", "P", "P [pu]"),
             ("Q pu", "Q", "Q [pu]")]

    for label, res in multi_results.items():
        masks = res["masks"]
        df_model = res["bus"]

        fig, axs = plt.subplots(1, len(props), figsize=(5 * len(props), 5))

        for j, (prop_name, mask_key, axis_label) in enumerate(props):
            ax = axs[j]
            pred_vals, true_vals = [], []
            for bus in buses:
                if predicted_only and not masks[mask_key][bus].any():
                    continue
                if true_bus_results is None or (bus, prop_name) not in true_bus_results.columns:
                    continue
                if (bus, prop_name) not in df_model.columns:
                    continue
                pred_vals.extend(df_model[bus][prop_name].values)
                true_vals.extend(true_bus_results[bus][prop_name].values)

            if pred_vals:
                ax.scatter(true_vals, pred_vals,
                           alpha=0.5, s=15)#, label=label)

            # Identity line
            if true_bus_results is not None:
                all_true = []
                for bus in buses:
                    if (bus, prop_name) in true_bus_results.columns:
                        all_true.extend(true_bus_results[bus][prop_name].values)
                if all_true:
                    mn, mx = min(all_true), max(all_true)
                    ax.plot([mn, mx], [mn, mx], "k--", linewidth=1.5, label="Perfect")

            ax.set_xlabel(f"True {axis_label}", fontsize=10)
            ax.set_ylabel(f"Predicted {axis_label}", fontsize=10)
            ax.set_title(axis_label, fontsize=11)
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)

        fig.suptitle(f"{title_prefix} — Predicted vs True (scatter) — {label}", fontsize=12)
        plt.tight_layout()
        plt.show()


def plot_errors_comparison_per_node(
    multi_results, network, true_bus_results=None,
    title_prefix="", predicted_only=True
):
    """
    Per-bus error bars: MAE per bus per model, grouped by bus.
    STSI260320: shows which buses benefit/suffer from physics loss.
    """
    buses = list(network.buses.index)
    props = [("V pu", "V", "Vmag MAE [p.u.]"),
             ("Angle deg", "Angle", "Vang MAE [deg]"),
             ("P pu", "P", "P MAE [pu]"),
             ("Q pu", "Q", "Q MAE [pu]")]
    colors = plt.cm.tab10.colors
    labels = list(multi_results.keys())
    first_masks = list(multi_results.values())[0]["masks"]

    fig, axs = plt.subplots(1, len(props), figsize=(5 * len(props), 5))
    x = np.arange(len(buses))
    width = 0.8 / len(labels)

    for j, (prop_name, mask_key, axis_label) in enumerate(props):
        ax = axs[j]
        for k, (label, res) in enumerate(multi_results.items()):
            df_model = res["bus"]
            maes = []
            for bus in buses:
                if predicted_only and not first_masks[mask_key][bus].any():
                    maes.append(0.0)
                    continue
                if true_bus_results is None or (bus, prop_name) not in true_bus_results.columns:
                    maes.append(0.0)
                    continue
                if (bus, prop_name) not in df_model.columns:
                    maes.append(0.0)
                    continue
                err = np.mean(np.abs(
                    df_model[bus][prop_name].values -
                    true_bus_results[bus][prop_name].values
                ))
                maes.append(err)
            offset = (k - len(labels) / 2 + 0.5) * width
            ax.bar(x + offset, maes, width,
                   color=colors[k % len(colors)], alpha=0.8, label=label)

        bus_labels = [f"{b}\n({get_bus_type_label(network, b)})" for b in buses]
        ax.set_xticks(x)
        ax.set_xticklabels(bus_labels, fontsize=8)
        ax.set_ylabel(axis_label, fontsize=10)
        ax.set_title(axis_label, fontsize=11)
        ax.legend(fontsize=7)
        ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle(f"{title_prefix} — Per-bus MAE by model", fontsize=12)
    plt.tight_layout()
    plt.show()

def plot_line_flow_timeseries_comparison(multi_results, network, true_line_results=None, title_prefix=""):
    """One figure per line: P0 (left) and Q0 (right), all models vs PyPSA truth."""
    lines  = list(network.lines.index)
    colors = plt.cm.tab10.colors
    for line in lines:
        from_bus = network.lines.loc[line, "bus0"]
        to_bus   = network.lines.loc[line, "bus1"]
        fig, (ax_p, ax_q) = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
        fig.suptitle(f"{title_prefix} — Line: {line}  ({from_bus} → {to_bus})", fontsize=12)
        if true_line_results is not None:
            if (line, "P0 pu") in true_line_results.columns:
                ax_p.plot(true_line_results.index, true_line_results[(line, "P0 pu")],
                          color="black", linewidth=2.5, label="PyPSA (true)", zorder=10)
            if (line, "Q0 pu") in true_line_results.columns:
                ax_q.plot(true_line_results.index, true_line_results[(line, "Q0 pu")],
                          color="black", linewidth=2.5, label="PyPSA (true)", zorder=10)
        for k, (label, res) in enumerate(multi_results.items()):
            line_df = res.get("line")
            if line_df is None:
                continue
            c = colors[k % len(colors)]
            if (line, "P0 pu") in line_df.columns:
                ax_p.plot(line_df.index, line_df[(line, "P0 pu")],
                          color=c, linewidth=1.5, alpha=0.85, label=label)
            if (line, "Q0 pu") in line_df.columns:
                ax_q.plot(line_df.index, line_df[(line, "Q0 pu")],
                          color=c, linewidth=1.5, alpha=0.85, label=label)
        ax_p.set_xlabel("Time Step"); ax_p.set_ylabel("P0 [pu]")
        ax_p.set_title("Active Power P0"); ax_p.legend(fontsize=7); ax_p.grid(True, alpha=0.3)
        ax_q.set_xlabel("Time Step"); ax_q.set_ylabel("Q0 [pu]")
        ax_q.set_title("Reactive Power Q0"); ax_q.legend(fontsize=7); ax_q.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

def plot_single_network_comparisons(
    runs,
    test_network,
    title_prefix="",
    use_pnom_share=None,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Multi-model overlay version.
    Build models dict: label -> model (or dict with ptdf info for learned_edge)
    """

    # Build true PyPSA results
    true_bus, true_line, _ = model_results(
        test_network,
        print_results=False,
        plot_results=False,
        compute_ptdf=False,
    )

    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    # Build models dict: label -> model (or dict with ptdf info for learned_edge)
    models_dict = {}
    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        ptdf_params = r.get("_ptdf_params")
        max_buses = r.get("_max_buses", 0)
        if ptdf_params is not None and max_buses > 0:
            models_dict[label] = {
                "model": r["model"],
                "ptdf_params": ptdf_params,
                "max_buses": max_buses,
            }
        else:
            models_dict[label] = r["model"]

    # Get multi predictions
    multi_results = predict_network_results_with_masks_multi(
        models_dict, test_network, use_pnom_share=use_pnom_share
    )

    # All three plots
    plot_comparison_results_with_masks(
        multi_results,
        test_network,
        true_bus_results=true_bus,
        true_line_results=true_line,
        title_prefix=title_prefix,
    )
    plot_line_flow_timeseries_comparison(
        multi_results,
        test_network,
        true_line_results=true_line,
        title_prefix=title_prefix,
    )

    plot_errors_comparison_per_node(
        multi_results,
        test_network,
        true_bus_results=true_bus,
        title_prefix=title_prefix,
    )

def evaluate_models_on_networks(models_dict, networks, use_edge_features=True, use_pnom_share=None):
    """
    Run each model on a list of networks and aggregate simple metrics.

    Returns: dict label -> dict with keys:
        'vmag_mae', 'vang_mae_deg', 'p_mae', 'q_mae'  (averaged over nets & buses)
    """
    results = {}

    for label, entry in models_dict.items():
        # Unpack model / ptdf info (entry can be bare model or dict)
        if isinstance(entry, dict):
            model       = entry["model"]
            ptdf_params = entry.get("ptdf_params")
            max_buses   = entry.get("max_buses", 0)
        else:
            model       = entry
            ptdf_params = None
            max_buses   = 0

        vmag_errs = []
        vang_errs = []
        p_errs    = []
        q_errs    = []

        for net in networks:
            true_bus, true_line, _ = model_results(
                net, print_results=False, plot_results=False, compute_ptdf=False
            )
            try:
                bus_res, _, masks = predict_network_results_with_masks(
                    model, net, use_edge_features=use_edge_features,
                    ptdf_params=ptdf_params, max_buses=max_buses,
                    use_pnom_share=use_pnom_share,
                )
            except RuntimeError as exc:
                print(f"[WARNING] {label} failed on network: {exc}")
                continue

            buses = list(net.buses.index)
            for bus in buses:
                # Vmag
                if (bus, "V pu") in true_bus.columns and (bus, "V pu") in bus_res.columns:
                    vmag_errs.extend(
                        np.abs(
                            bus_res[bus]["V pu"].values -
                            true_bus[bus]["V pu"].values
                        )
                    )
                # Vang (deg)
                if (bus, "Angle deg") in true_bus.columns and (bus, "Angle deg") in bus_res.columns:
                    vang_errs.extend(
                        np.abs(
                            bus_res[bus]["Angle deg"].values -
                            true_bus[bus]["Angle deg"].values
                        )
                    )
                # P
                if (bus, "P pu") in true_bus.columns and (bus, "P pu") in bus_res.columns:
                    p_errs.extend(
                        np.abs(
                            bus_res[bus]["P pu"].values -
                            true_bus[bus]["P pu"].values
                        )
                    )
                # Q
                if (bus, "Q pu") in true_bus.columns and (bus, "Q pu") in bus_res.columns:
                    q_errs.extend(
                        np.abs(
                            bus_res[bus]["Q pu"].values -
                            true_bus[bus]["Q pu"].values
                        )
                    )
    
        results[label] = {
            "vmag_mae":     float(np.mean(vmag_errs)) if vmag_errs else np.nan,
            "vang_mae_deg": float(np.mean(vang_errs)) if vang_errs else np.nan,
            "p_mae":        float(np.mean(p_errs))    if p_errs    else np.nan,
            "q_mae":        float(np.mean(q_errs))    if q_errs    else np.nan,
        }

    return results

def plot_model_metrics_across_networks(models_dict, networks, title_prefix="Multi-net", use_pnom_share=None):
    metrics = evaluate_models_on_networks(models_dict, networks, use_pnom_share=use_pnom_share)
    labels  = list(metrics.keys())
    keys    = ["vmag_mae", "vang_mae_deg", "p_mae", "q_mae"]
    pretty  = ["Vmag MAE [p.u.]", "Vang MAE [deg]", "P MAE [pu]", "Q MAE [pu]"]

    x = np.arange(len(labels))
    fig, axes = plt.subplots(1, len(keys), figsize=(12 + len(labels), 5))
    for ax, k, name in zip(axes, keys, pretty):
        vals = [metrics[l][k] for l in labels]
        ax.bar(x, vals, color=[f"C{i}" for i in x])
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
        ax.set_ylabel(name)
        ax.set_title(name)
        ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle(f"{title_prefix} — averaged over {len(networks)} networks", fontsize=13)
    plt.tight_layout()
    plt.show()

    

In [ ]:


def _collect_test_set_predictions(models_dict, networks, use_pnom_share=None):
    """
    Run every model on every network.
    Returns dict: label -> {'pred': [N,4], 'true': [N,4], optionally 'p0_pred', 'p0_true', 'q0_pred', 'q0_true'}
    """
    all_preds = {}
    for label, entry in models_dict.items():
        if isinstance(entry, dict):
            model       = entry["model"]
            ptdf_params = entry.get("ptdf_params")
            max_buses   = entry.get("max_buses", 0)
        else:
            model, ptdf_params, max_buses = entry, None, 0

        pred_rows, true_rows = [], []
        p0_pred_rows, p0_true_rows = [], []
        q0_pred_rows, q0_true_rows = [], []

        for net in networks:
            bus_names = list(net.buses.index)
            try:
                bus_res, pred_line, masks = predict_network_results_with_masks(
                    model, net, use_edge_features=True,
                    ptdf_params=ptdf_params, max_buses=max_buses,
                    use_pnom_share=use_pnom_share)
            except RuntimeError as exc:
                print(f"[WARNING] {label} skipped: {exc}"); continue

            true_bus, true_line, _ = model_results(net, print_results=False, plot_results=False, compute_ptdf=False)

            for t in net.snapshots:
                for bus in bus_names:
                    try:
                        pred_rows.append([bus_res.loc[t,(bus,"V pu")], bus_res.loc[t,(bus,"Angle deg")],
                                          bus_res.loc[t,(bus,"P pu")], bus_res.loc[t,(bus,"Q pu")]])
                        true_rows.append([true_bus.loc[t,(bus,"V pu")], true_bus.loc[t,(bus,"Angle deg")],
                                          true_bus.loc[t,(bus,"P pu")], true_bus.loc[t,(bus,"Q pu")]])
                    except KeyError:
                        continue
                if pred_line is not None and true_line is not None:
                    for line in net.lines.index:
                        try:
                            p0_pred_rows.append(pred_line.loc[t,(line,"P0 pu")]); p0_true_rows.append(true_line.loc[t,(line,"P0 pu")])
                            q0_pred_rows.append(pred_line.loc[t,(line,"Q0 pu")]); q0_true_rows.append(true_line.loc[t,(line,"Q0 pu")])
                        except KeyError:
                            continue

        entry_dict = {"pred": np.array(pred_rows), "true": np.array(true_rows)}
        if p0_pred_rows:
            entry_dict["p0_pred"] = np.array(p0_pred_rows); entry_dict["p0_true"] = np.array(p0_true_rows)
        if q0_pred_rows:
            entry_dict["q0_pred"] = np.array(q0_pred_rows); entry_dict["q0_true"] = np.array(q0_true_rows)
        all_preds[label] = entry_dict
    return all_preds


def plot_full_test_scatter(all_preds, title_prefix=""):
    """Scatter (predicted vs true): 4 bus panels + optional 2 line flow panels."""
    bus_col_names = ["V_mag (pu)", "V_ang (deg)", "P (pu)", "Q (pu)"]
    for label, data in all_preds.items():
        has_lf = "p0_pred" in data and len(data["p0_pred"]) > 0
        n_panels = 6 if has_lf else 4
        fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5))
        for col_idx, (ax, cname) in enumerate(zip(axes[:4], bus_col_names)):
            pred = data["pred"][:, col_idx]; true = data["true"][:, col_idx]
            lo, hi = min(true.min(), pred.min()), max(true.max(), pred.max())
            ax.scatter(true, pred, s=2, alpha=0.3, color=f"C{col_idx}")
            ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, label="ideal")
            ax.set_aspect("equal", adjustable="datalim")
            ax.set_xlabel(f"True {cname}"); ax.set_ylabel(f"Predicted {cname}")
            ax.set_title(cname); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
        if has_lf:
            for ci, (kp, kt, cname) in enumerate([("p0_pred","p0_true","P0 Line (p.u.)"),("q0_pred","q0_true","Q0 Line (p.u.)")]):
                ax = axes[4 + ci]
                ax.scatter(data[kt], data[kp], s=2, alpha=0.3, color=f"C{4+ci}")
                lo, hi = data[kt].min(), data[kt].max()
                ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, label="ideal")
                ax.set_xlabel(f"True {cname}"); ax.set_ylabel(f"Predicted {cname}")
                ax.set_title(cname); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
        fig.suptitle(f"{title_prefix} Full test-set scatter — {label}", fontsize=14)
        plt.tight_layout(); plt.show()


def plot_error_histograms(all_preds, title_prefix="", bins=80):
    """Error histograms: 4 bus panels + optional 2 line flow panels."""
    bus_col_names = ["V_mag (pu)", "V_ang (deg)", "P (pu)", "Q (pu)"]
    for label, data in all_preds.items():
        has_lf = "p0_pred" in data and len(data["p0_pred"]) > 0
        n_panels = 6 if has_lf else 4
        fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5))
        for col_idx, (ax, cname) in enumerate(zip(axes[:4], bus_col_names)):
            err = data["pred"][:, col_idx] - data["true"][:, col_idx]
            mae = np.mean(np.abs(err)); mu = np.mean(err); sigma = np.std(err)
            ci_lo, ci_hi = np.percentile(err, [2.5, 97.5])
            ax.hist(err, bins=bins, alpha=0.6, color=f"C{col_idx}",
                    label=f"MAE={mae:.4g}\nμ={mu:.4g}  σ={sigma:.4g}\n95% CI=[{ci_lo:.4g},{ci_hi:.4g}]")
            ax.axvline(mu, color="k", linestyle="-", linewidth=1.2, label="mean")
            ax.axvline(ci_lo, color="k", linestyle="--", linewidth=0.9)
            ax.axvline(ci_hi, color="k", linestyle="--", linewidth=0.9, label="95% CI")
            ax.set_xlabel(f"Error ({cname})"); ax.set_ylabel("Count")
            ax.set_title(cname); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        if has_lf:
            for ci, (kp, kt, cname) in enumerate([("p0_pred","p0_true","P0 Line (p.u.)"),("q0_pred","q0_true","Q0 Line (p.u.)")]):
                ax = axes[4 + ci]
                if kp in data and len(data[kp]) > 0:
                    err = data[kp] - data[kt]
                    mae = np.mean(np.abs(err)); mu = np.mean(err); sigma = np.std(err)
                    ci_lo, ci_hi = np.percentile(err, [2.5, 97.5])
                    ax.hist(err, bins=bins, alpha=0.6, color=f"C{4+ci}",
                            label=f"MAE={mae:.4g}\nμ={mu:.4g}  σ={sigma:.4g}\n95% CI=[{ci_lo:.4g},{ci_hi:.4g}]")
                    ax.axvline(mu, color="k", linestyle="-", linewidth=1.2, label="mean")
                    ax.axvline(ci_lo, color="k", linestyle="--", linewidth=0.9)
                    ax.axvline(ci_hi, color="k", linestyle="--", linewidth=0.9, label="95% CI")
                ax.set_xlabel(f"Error ({cname})"); ax.set_ylabel("Count")
                ax.set_title(cname); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        fig.suptitle(f"{title_prefix} {label}", fontsize=14)
        plt.tight_layout(); plt.show()

In [ ]:
def plot_errors_vs_operating_range(
    all_preds,
    n_bins: int = 20,
    title_prefix: str = "",
    smoothing_mode: str = None,   # None | "gaussian" | "rolling_avg"
    smoothing_sigma: float = 1.0, # used for gaussian
    smoothing_window: int = 5,    # used for rolling_avg
):
    """
    For each of the 4 bus quantities: one figure, one subplot per model (stacked).
    x-axis = operating range (bin centres of true values)
    y-axis = signed prediction error with filled IQR, std bands, and smoothed median.
    All subplots share unified y-axis range for comparability.

    smoothing_mode:
        None        — raw median line (no smoothing)
        "gaussian"  — scipy gaussian_filter1d (sigma=smoothing_sigma)
        "rolling_avg" — uniform rolling average (window=smoothing_window)
    """
    from scipy import ndimage

    def _smooth(arr):
        if smoothing_mode == "gaussian":
            return ndimage.gaussian_filter1d(arr, sigma=smoothing_sigma)
        elif smoothing_mode == "rolling_avg":
            kernel = np.ones(smoothing_window) / smoothing_window
            return np.convolve(arr, kernel, mode="same")
        return arr  # None — no smoothing

    col_specs = [
        (0, "V_mag (p.u.)"),
        (1, "V_ang (deg)"),
        (2, "P (p.u.)"),
        (3, "Q (p.u.)"),
    ]
    colors   = plt.cm.tab10.colors
    labels   = list(all_preds.keys())
    n_models = len(labels)

    for col_idx, col_name in col_specs:
        # ── Pre-pass: bin all models, collect bounds for unified y-range ────────
        all_bounds = []
        all_data_per_model = {}

        for lbl, data in all_preds.items():
            t_col = data["true"][:, col_idx]
            p_col = data["pred"][:, col_idx]
            err   = p_col - t_col
            vmin, vmax = t_col.min(), t_col.max()
            if vmin == vmax:
                continue
            bin_edges   = np.linspace(vmin, vmax, n_bins + 1)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            bin_idx     = np.clip(np.digitize(t_col, bin_edges[:-1]) - 1, 0, n_bins - 1)

            q05, q25, q50, q75, q95, std_vals, valid_centers = [], [], [], [], [], [], []
            for b in range(n_bins):
                mask = bin_idx == b
                if mask.sum() >= 5:
                    vals = err[mask]
                    q05.append(np.percentile(vals, 5))
                    q25.append(np.percentile(vals, 25))
                    q50.append(np.percentile(vals, 50))
                    q75.append(np.percentile(vals, 75))
                    q95.append(np.percentile(vals, 95))
                    std_vals.append(np.std(vals))
                    valid_centers.append(bin_centers[b])

            if valid_centers:
                all_bounds.extend(q05 + q95 + std_vals)
                all_data_per_model[lbl] = {
                    "xc": np.array(valid_centers),
                    "q05": np.array(q05), "q25": np.array(q25),
                    "q50": np.array(q50), "q75": np.array(q75),
                    "q95": np.array(q95), "std": np.array(std_vals),
                }

        # ── Unified y-range ───────────────────────────────────────────────────
        if all_bounds:
            y_min = np.percentile(all_bounds, 2.5)
            y_max = np.percentile(all_bounds, 97.5)
            margin = (y_max - y_min) * 0.1
            y_min -= margin; y_max += margin
        else:
            y_min, y_max = -1, 1

        # ── Main plot loop ────────────────────────────────────────────────────
        fig, axes = plt.subplots(
            n_models, 1,
            figsize=(10, max(3.5, n_models * 3.5)),
            squeeze=False,
        )
        axes = axes.flatten()
        fig.suptitle(
            f"{title_prefix} -- {col_name}: error spread vs operating range",
            fontsize=13,
        )

        for row, (lbl, data) in enumerate(all_preds.items()):
            ax = axes[row]
            if lbl not in all_data_per_model:
                ax.set_title(lbl); continue

            pd_ = all_data_per_model[lbl]
            xc  = pd_["xc"]
            q05, q25, q50 = pd_["q05"], pd_["q25"], pd_["q50"]
            q75, q95      = pd_["q75"], pd_["q95"]
            std_vals      = pd_["std"]
            color         = colors[row % 10]

            q50_smooth   = _smooth(q50)
            std_smooth   = _smooth(std_vals)

            ax.fill_between(xc, q05, q95,   alpha=0.15, color=color, label="5-95 percentile")
            ax.fill_between(xc, q25, q75,   alpha=0.40, color=color, label="IQR (25-75%)")
            ax.fill_between(xc, q50 - std_smooth, q50 + std_smooth, alpha=0.25, color=color, label="Mean ± 1σ")
            ax.plot(xc, q50_smooth, color=color, linewidth=2.0, label="Median" + ("" if smoothing_mode is None else f" ({smoothing_mode})"))
            ax.axhline(0, color="red", linewidth=0.9, linestyle="--", alpha=0.6, label="zero error")
            ax.set_xlabel(f"True {col_name}"); ax.set_ylabel("Prediction error")
            ax.set_title(lbl); ax.set_ylim(y_min, y_max)
            ax.legend(fontsize=7, loc="upper right"); ax.grid(True, alpha=0.3)

        plt.tight_layout(); plt.show()


def plot_line_flow_errors(
    runs,
    title_prefix="Line Flow Errors",
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Bar chart comparing line_flow_mae across runs.
    Reads test_metrics['line_flow_mae'] from each run dict.
    Runs that pre-date line-flow tracking will show NaN bars.
    """
    x = list(range(len(runs)))
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    styles = _get_run_styles(runs)
    colors = [styles[id(r)][0] for r in runs]
    tick_labels = [entry["plot_label"] for entry in label_entries]
    tick_rotation = 0 if active_label_mode == "index" else 30
    tick_ha = "center" if active_label_mode == "index" else "right"
    tick_fontsize = 9 if active_label_mode == "index" else 8

    means = []
    for r in runs:
        tm = r.get("test_metrics", {})
        val = tm.get("line_flow_mae", float("nan"))
        means.append(float(val) if val is not None else float("nan"))

    fig, ax = plt.subplots(figsize=(max(8, len(runs) * 1.5), 6))
    fig.suptitle(f"{title_prefix} — Line Flow MAE [p.u.]", fontsize=13)

    bars = ax.bar(x, means, alpha=0.8, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=tick_rotation, ha=tick_ha, fontsize=tick_fontsize)
    ax.set_ylabel("Line Flow MAE [p.u.]")
    ax.set_title("Line Flow MAE per run")
    ax.grid(True, axis="y", alpha=0.3)

    valid = [m for m in means if m == m]
    top = max(valid) if valid else 1.0
    for bar, mean in zip(bars, means):
        if mean == mean:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + top * 0.02,
                f"{mean:.4f}",
                ha="center", va="bottom", fontsize=7,
            )

    if active_label_mode == "index":
        legend_lines = [
            plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                       label=label_entries[i]["mapping_label"])
            for i in range(len(runs))
        ]
        fig.legend(
            handles=legend_lines,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=max(1, min(3, len(runs))),
            fontsize=8,
            frameon=True,
        )
        plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    else:
        plt.tight_layout()
    plt.show()



In [ ]:

def plot_timing_vs_accuracy(
    runs,
    x_metric="training_time",
    y_metric="vmag_mae",
    title_prefix="Timing vs Accuracy",
    ref_baselines: dict = None,
    x_axis_log: bool = False,
    # [STSI260502]: allow overriding the timing key for GNN runs
    timing_key: str = "both",
    label_mode: str = "auto",
    compact_threshold: int = RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold: int = RUN_LABEL_INDEX_THRESHOLD,
    annotation_mode: str = "auto",
):
    """
    Scatter plot of a timing metric vs an accuracy metric across runs.

    ref_baselines : dict {label: metrics_dict}, optional
        Reference points from evaluate_dc_baseline (or any solver).
        Plotted as black stars with bold labels.

    x_metric options:
        "training_time"         — seconds per full training run (run['training_time'])
        "inference_time_s"      — seconds for entire test-set inference (test_metrics key)
        "infer_ms_per_snap"     — ms per snapshot (test_metrics key)

    y_metric options:
        "vmag_mae", "vang_rmse", "vang_mae", "p_mae", "q_mae", "line_flow_mae"
        (all read from run['test_metrics'])
    """
    x_metric_labels = {
        "training_time":     "Training time [s]",
        "inference_time_s":  "Inference time [s] (full test set)",
        "infer_ms_per_snap": "Inference time [ms / snapshot]",
    }
    y_metric_labels = {
        "vmag_mae":      "Vmag MAE [p.u.]",
        "vang_mae":      "Vang MAE [rad]",
        "vang_rmse":     "Vang RMSE [rad]",
        "p_mae":         "P MAE [p.u.]",
        "q_mae":         "Q MAE [p.u.]",
        "line_flow_mae": "Line Flow MAE [p.u.]",
    }

    # ── resolve timing keys ───────────────────────────────────────────────
    _alias = {"infer_ms_per_snap": "inference_time_per_snapshot_ms"}
    _timing_modes = {
        "single": [("total_time_ms", "o", "per-snap")],
        "batch":  [("batch_solve_time_ms", "D", "batched")],
        "both":   [("total_time_ms", "o", "per-snap"),
                    ("batch_solve_time_ms", "D", "batched")],
    }
    modes = _timing_modes.get(timing_key, [("total_time_ms", "o", "")])

    styles = _get_run_styles(runs)
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    annotation_mode = _resolve_annotation_mode(annotation_mode, active_label_mode)
    _leg_handles, _leg_labels_seen = [], []
    annotated_runs = set()
    fig, ax = plt.subplots(figsize=(10, 7))
    fig.suptitle(
        f"{title_prefix} — {x_metric_labels.get(x_metric, x_metric)}"
        f" vs {y_metric_labels.get(y_metric, y_metric)}",
        fontsize=12,
    )

    for tkey, default_mk, mode_suffix in modes:
        _alias_runs = {"infer_ms_per_snap": tkey}
        for i, r in enumerate(runs):
            tm = r.get("test_metrics", {})
            if x_metric == "training_time":
                xval = r.get("training_time", float("nan"))
            else:
                xval = tm.get(_alias_runs.get(x_metric, x_metric), float("nan"))
            yval = tm.get(y_metric, float("nan"))
            xval = float(xval) if xval is not None else float("nan")
            yval = float(yval) if yval is not None else float("nan")
            if xval != xval or yval != yval:
                continue
            color, _, mk = styles[id(r)]
            mk_use = default_mk if len(modes) > 1 else (mk or "o")
            suffix = f" ({mode_suffix})" if mode_suffix else ""
            plot_label = label_entries[i]["plot_label"]
            legend_label = label_entries[i]["legend_label"]
            sc = ax.scatter(xval, yval, color=color, marker=mk_use, s=80, zorder=3)
            if legend_label not in _leg_labels_seen:
                _leg_handles.append(sc)
                _leg_labels_seen.append(legend_label)
            should_annotate = (
                annotation_mode == "all"
                or (annotation_mode == "first-per-run" and i not in annotated_runs)
            )
            if should_annotate:
                ax.annotate(
                    plot_label + suffix,
                    xy=(xval, yval),
                    xytext=(5, 4),
                    textcoords="offset points",
                    fontsize=7,
                    color=color,
                )
                if annotation_mode == "first-per-run":
                    annotated_runs.add(i)
    # ── DC / reference baselines ─────────────────────────────────────────────
    if ref_baselines:
        for dc_label, dc_metrics in ref_baselines.items():
            if x_metric == "training_time":
                continue  # DC has no training time
            xval_dc = float(dc_metrics.get(_alias.get(x_metric, x_metric), float("nan")))
            yval_dc = float(dc_metrics.get(y_metric, float("nan")))
            if xval_dc != xval_dc or yval_dc != yval_dc:
                continue
            sc = ax.scatter(xval_dc, yval_dc, color="black", marker="*", s=280, zorder=5)
            if dc_label not in _leg_labels_seen:
                _leg_handles.append(sc)
                _leg_labels_seen.append(dc_label)
            ax.annotate(
                dc_label,
                xy=(xval_dc, yval_dc),
                xytext=(6, 4),
                textcoords="offset points",
                fontsize=8,
                color="black",
                fontweight="bold",
            )

    ax.set_xlabel(x_metric_labels.get(x_metric, x_metric))
    ax.set_ylabel(y_metric_labels.get(y_metric, y_metric))
    if x_axis_log:
        ax.set_xscale("log")
    ax.grid(True, alpha=0.3)
    if len(modes) > 1:
        from matplotlib.lines import Line2D
        for _mk, _ml in [("o", "per-snap"), ("D", "batched")]:
            _leg_handles.append(Line2D([0], [0], marker=_mk, color="gray", linestyle="None", markersize=7))
            _leg_labels_seen.append(_ml)
    if _leg_handles:
        ax.legend(handles=_leg_handles, labels=_leg_labels_seen, fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()

def plot_timing_vs_accuracy_multi(
    runs,
    y_metrics: list,
    x_metric: str = "infer_ms_per_snap",
    title_prefix: str = "Timing vs Accuracy",
    ref_baselines: dict = None,
    label_mode: str = "auto",
    compact_threshold: int = RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold: int = RUN_LABEL_INDEX_THRESHOLD,
    annotation_mode: str = "auto",
):
    """
    Scatter plot of inference time vs multiple accuracy metrics on one axes.

    y_metrics : list of (metric_key, short_label, marker_char) tuples
                e.g. [("p_mae", "P MAE", "o"), ("q_mae", "Q MAE", "s")]

    ref_baselines : dict {label: metrics_dict}, optional
        Each entry gets one star per y_metric that is present in the dict.
    """
    _x_labels = {
        "infer_ms_per_snap": "Inference time [ms / snapshot]",
        "inference_time_s":  "Inference time [s]",
        "training_time":     "Training time [s]",
    }
    _alias = {"infer_ms_per_snap": "inference_time_per_snapshot_ms"}

    styles = _get_run_styles(runs)
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    annotation_mode = _resolve_annotation_mode(annotation_mode, active_label_mode)

    fig, ax = plt.subplots(figsize=(11, 7))
    fig.suptitle(
        f"{title_prefix} — {_x_labels.get(x_metric, x_metric)}"
        f" vs {', '.join(lbl for _, lbl, _ in y_metrics)}",
        fontsize=12,
    )

    legend_handles = []

    for run_i, r in enumerate(runs):
        tm = r.get("test_metrics", {})
        if x_metric == "training_time":
            xval = r.get("training_time", float("nan"))
        else:
            xval = tm.get(_alias.get(x_metric, x_metric), float("nan"))
        if xval is None or xval != xval:
            continue
        xval = float(xval)

        color, _, _ = styles[id(r)]
        first_metric = True
        for metric_key, metric_lbl, mk in y_metrics:
            yval = tm.get(metric_key, float("nan"))
            if yval is None or yval != yval:
                continue
            yval = float(yval)
            ax.scatter(
                xval, yval, color=color, marker=mk, s=90, zorder=3,
                linewidths=0.8, edgecolors="white",
            )
            should_annotate = (
                annotation_mode == "all"
                or (annotation_mode == "first-per-run" and first_metric)
            )
            if should_annotate:
                ax.annotate(
                    label_entries[run_i]["plot_label"],
                    xy=(xval, yval),
                    xytext=(5, 4),
                    textcoords="offset points",
                    fontsize=6.5,
                    color=color,
                )
            first_metric = False

    # ── Metric-shape legend entries ───────────────────────────────────────────
    for metric_key, metric_lbl, mk in y_metrics:
        legend_handles.append(
            plt.scatter([], [], marker=mk, color="gray", s=70, label=metric_lbl)
        )

    # ── Run-color legend entries ──────────────────────────────────────────────
    for run_i, r in enumerate(runs):
        color, _, _ = styles[id(r)]
        legend_handles.append(
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color,
                       markersize=8, label=label_entries[run_i]["legend_label"])
        )

    # ── DC / reference baselines ──────────────────────────────────────────────
    if ref_baselines:
        for dc_label, dc_metrics in ref_baselines.items():
            if x_metric == "training_time":
                continue
            xval_dc = float(dc_metrics.get(_alias.get(x_metric, x_metric), float("nan")))
            if xval_dc != xval_dc:
                continue
            for metric_key, metric_lbl, mk in y_metrics:
                yval_dc = float(dc_metrics.get(metric_key, float("nan")))
                if yval_dc != yval_dc:
                    continue
                ax.scatter(xval_dc, yval_dc, color="black", marker="*", s=300, zorder=5)
                ax.annotate(
                    f"{dc_label} ({metric_lbl})",
                    xy=(xval_dc, yval_dc),
                    xytext=(6, 4),
                    textcoords="offset points",
                    fontsize=8,
                    color="black",
                    fontweight="bold",
                )

    ax.set_xlabel(_x_labels.get(x_metric, x_metric))
    ax.set_ylabel("MAE [p.u.]")
    ax.grid(True, alpha=0.3)
    ax.legend(handles=legend_handles, fontsize=7,
              bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
    plt.tight_layout()
    plt.show()

def plot_inference_time_bar(
    runs,
    title_prefix="Timing",
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Bar chart of inference time per snapshot (ms) across runs.
    Reads test_metrics['inference_time_per_snapshot_ms'].
    """
    x = list(range(len(runs)))
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    styles = _get_run_styles(runs)
    colors = [styles[id(r)][0] for r in runs]
    tick_labels = [entry["plot_label"] for entry in label_entries]
    tick_rotation = 0 if active_label_mode == "index" else 30
    tick_ha = "center" if active_label_mode == "index" else "right"
    tick_fontsize = 9 if active_label_mode == "index" else 8

    ms_vals = []
    for r in runs:
        tm = r.get("test_metrics", {})
        val = tm.get("inference_time_per_snapshot_ms", float("nan"))
        ms_vals.append(float(val) if val is not None else float("nan"))

    fig, ax = plt.subplots(figsize=(max(8, len(runs) * 1.5), 6))
    fig.suptitle(f"{title_prefix} — Inference time per snapshot [ms]", fontsize=13)

    bars = ax.bar(x, ms_vals, alpha=0.8, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=tick_rotation, ha=tick_ha, fontsize=tick_fontsize)
    ax.set_ylabel("Inference time [ms / snapshot]")
    ax.set_title("Inference time per snapshot")
    ax.grid(True, axis="y", alpha=0.3)

    valid = [v for v in ms_vals if v == v]
    top = max(valid) if valid else 1.0
    for bar, val in zip(bars, ms_vals):
        if val == val:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + top * 0.02,
                f"{val:.2f}",
                ha="center", va="bottom", fontsize=7,
            )

    if active_label_mode == "index":
        legend_lines = [
            plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                       label=label_entries[i]["mapping_label"])
            for i in range(len(runs))
        ]
        fig.legend(
            handles=legend_lines,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=max(1, min(3, len(runs))),
            fontsize=8,
            frameon=True,
        )
        plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    else:
        plt.tight_layout()
    plt.show()



In [ ]:
# =============================================================================
# SECTION 12d: CROSS-SYSTEM GENERALISATION EVALUATION
# =============================================================================

def cross_system_eval(
    runs_by_system: dict,
    networks_by_system: dict,
    metric: str = "vmag_mae",
    seed: int = 42,
):
    """
    Evaluate every run in each train-system against every test-system's networks.

    Args:
        runs_by_system:    {"ieee9": runs_list, "cigre14": runs_list, "ieee30": runs_list, ...}
        networks_by_system: {"ieee9": full_networks_list, ...}
            Uses the stored test split if runs[0]["test_networks"] exists,
            otherwise reconstructs from the full list + seed.
        metric:  which test_metrics key to compare (default "vmag_mae")
        seed:    seed for reconstructing test split when not stored

    Returns:
        results: dict[train_system][run_label][test_system] = float (metric value)
        test_nets: dict[test_system] = list of test networks used
    """
    from collections import defaultdict

    # ── Build test network sets per system ────────────────────────────────────
    test_nets = {}
    for sys_name, nets in networks_by_system.items():
        runs = runs_by_system.get(sys_name, [])
        stored = runs[0].get("test_networks") if runs else None
        if stored is not None:
            test_nets[sys_name] = stored
            print(f"  {sys_name}: using stored test split ({len(stored)} networks)")
        else:
            _, _, split = reconstruct_test_split(nets, seed=seed)
            test_nets[sys_name] = split
            print(f"  {sys_name}: reconstructed test split ({len(split)} networks)")

    # ── Evaluate every run on every test system ───────────────────────────────
    results = defaultdict(lambda: defaultdict(dict))

    for train_sys, runs in runs_by_system.items():
        _, label_entries = resolve_run_label_policy(runs)
        label_entry_by_run = _index_label_entries(label_entries)
        for r in runs:
            lbl = label_entry_by_run[id(r)]["legend_label"]

            # Skip learned_edge PTDF runs — fixed dimension, can't cross-test
            if r.get("_ptdf_params") is not None and r.get("_max_buses", 0) > 0:
                print(f"  SKIP (learned_edge PTDF): {train_sys} / {lbl}")
                continue

            model = r["model"]

            for test_sys, eval_networks in test_nets.items():
                vmag_errs, vang_errs, p_errs, q_errs = [], [], [], []
                for net in eval_networks:
                    try:
                        true_bus, _, _ = model_results(
                            net, print_results=False, plot_results=False, compute_ptdf=False
                        )
                        bus_res, _, _ = predict_network_results_with_masks(
                            model, net, use_edge_features=True,
                            ptdf_params=None, max_buses=0,
                        )
                    except RuntimeError:
                        continue

                    for bus in net.buses.index:
                        if (bus, "V pu") in true_bus.columns and (bus, "V pu") in bus_res.columns:
                            vmag_errs.extend(np.abs(bus_res[bus]["V pu"].values - true_bus[bus]["V pu"].values))
                        if (bus, "Angle deg") in true_bus.columns and (bus, "Angle deg") in bus_res.columns:
                            vang_errs.extend(np.abs(bus_res[bus]["Angle deg"].values - true_bus[bus]["Angle deg"].values))
                        if (bus, "P pu") in true_bus.columns and (bus, "P pu") in bus_res.columns:
                            p_errs.extend(np.abs(bus_res[bus]["P pu"].values - true_bus[bus]["P pu"].values))
                        if (bus, "Q pu") in true_bus.columns and (bus, "Q pu") in bus_res.columns:
                            q_errs.extend(np.abs(bus_res[bus]["Q pu"].values - true_bus[bus]["Q pu"].values))

                metric_val = {
                    "vmag_mae": float(np.mean(vmag_errs)) if vmag_errs else float("nan"),
                    "vang_mae": float(np.mean(vang_errs)) if vang_errs else float("nan"),
                    "p_mae": float(np.mean(p_errs)) if p_errs else float("nan"),
                    "q_mae": float(np.mean(q_errs)) if q_errs else float("nan"),
                }.get(metric, float("nan"))
                results[train_sys][lbl][test_sys] = metric_val

    return results, test_nets



## Load Networks

In [ ]:

# (# ← result load"ieee9_large_single_slack_ptdf_sweep_060426.json"))
#networks_ieee9_500_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_500_wide.json"))
#networks_cigre14_large_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "cigre14_large.json"))
#runs_cigre14_500_phys_vs_ptdf=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"cigre14_large_single_slack_ptdf_sweep_060426.json"))

#networks_cigre14_xlarge_single_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "cigre14_xlarge.json"))
#networks_ieee9_mega_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_mega.json"))
#networks_cigre14_mega_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "cigre14_mega.json"))
networks_ieee9_500_wide=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_500_wide.json"))
networks_cigre14_500_wide=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "cigre14_500_wide.json"))
networks_ieee30_500_wide=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee30_500_wide.json"))

### Load Sweep Results

In [ ]:
# reload runs and networks if they are not in the list already (check if exist and only load if not already loaded to save time):

def load_runs_if_needed(run_names_and_files):
    loaded_runs = {}
    for var_name, file_name in run_names_and_files:
        if var_name:# not in globals():
            loaded_runs[var_name] = load_sweep_results(os.path.join(TRAINING_RESULTS_DIR, file_name))
            print(f"Loaded {var_name} from {file_name}")
        else:
            print(f"{var_name} already in memory, skipping load")
    return loaded_runs
runs_to_load = [
    ("runs_ieee9_500_wide_head_compare", "ieee9_500_single_slack_head_mode_060526.json"),
    ("runs_cigre14_500_wide_head_compare", "cigre14_500_wide_head_mode_060526.json"),
    #("runs_cigre14_750_phys_vs_ptdf", "cigre14_xlarge_single_slack_ptdf_sweep_110426.json"),
    #("runs_ieee30_1000_phys_vs_ptdf", "ieee30_xlarge_single_slack_ptdf_sweep_210426.json"),
    ("runs_ieee30_500_wide_head_compare", "cigre14_500_wide_head_mode_060526.json"),
    #("runs_ieee9_1500_phys_vs_ptdf", "ieee9_mega_single_slack_ptdf_sweep_0204526.json"),
    #("runs_cigre14_1500_phys_vs_ptdf", "cigre14_mega_single_slack_ptdf_sweep_030526.json"),
    #("runs_mixed_1500_phys_vs_ptdf", "mixed_1500_phys_vs_ptdf_sweep.json"),
]
loaded_runs = load_runs_if_needed(runs_to_load)


### Load Network lists

In [ ]:
def load_networks_if_needed(network_names_and_files):
    loaded_networks = {}
    for var_name, file_name in network_names_and_files:
        if var_name not in globals():
            loaded_networks[var_name] = load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, file_name))
            print(f"Loaded {var_name} from {file_name}")
        else:
            print(f"{var_name} already in memory, skipping load")
    return loaded_networks
networks_to_load = [
    ("networks_ieee9_large_singel_slack", "ieee9_large_singel_slack.json"),
    ("networks_cigre14_large_singel_slack", "cigre14_large.json"),
    ("networks_cigre14_xlarge_singel_slack", "cigre14_xlarge.json"),
    ("networks_ieee30_xlarge_singel_slack", "ieee30_xlarge.json"),
    ("networks_ieee30_large_singel_slack", "ieee30_large2.json"),
    ("networks_ieee30_xlarge_medium_timesteps", "ieee30_xlarge_medium_timesteps.json"),
    ("networks_ieee9_mega_singel_slack", "ieee9_mega.json"),
    ("networks_cigre14_mega_singel_slack", "cigre14_mega.json"),
]
loaded_networks = load_networks_if_needed(networks_to_load)

In [ ]:

runs_ieee9_1500_phys_vs_ptdf=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"ieee9_mega_single_slack_ptdf_sweep_0204526.json"))

In [ ]:
runs_cigre14_1500_phys_vs_ptdf=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"cigre14_mega_single_slack_ptdf_sweep_030526.json"))

### Baseline Solver Timing

In [ ]:
ref_singel_ieee9_500_wide,ref_batch_ieee9_500_wide=build_ref_baselines(networks_ieee9_500_wide, n_sample=50)

In [ ]:
ref_singel_cigre14_500_wide,ref_batch_cigre14_500_wide=build_ref_baselines(networks_cigre14_500_wide, n_sample=50)

In [ ]:
ref_singel_ieee30_500_wide,ref_batch_ieee30_500_wide=build_ref_baselines(networks_ieee30_500_wide, n_sample=50)

In [ ]:


# =============================================================================
# SECTION 14: Test
# =============================================================================
if run_regression_test:
    reg_results = run_full_regression_test(
        run_unit_tests=True,
        num_scenarios=500,
        steps_per_scenario=24,
        seed=42,
        num_epochs=50,
        batch_size=34,
        lr=1e-3,
        conv_type="gatv2",
        hidden_dim=64,
        num_layers=3,
        y_matrix_source="manual",
        warmup_epochs=10,

        base_system="ieee9",
        distribute_slack_flag=False,
        distributed_slack_mode="proportional",
        enable_extended_topology=True,
        extended_bus_prob=0.2,
        line_modification_prob=0.4,
        line_param_variation=0.25,
        verbose=False,

        use_edge_features=True,
        use_pnet_balance=True,
        return_model=True,
    )


## Old plots and old model

### Refresh Test Metrics

In [ ]:
# one time refresh to add std to results for all runs
refresh_test_metrics_with_std(
    runs_by_system={
        "cigre14": runs_cigre14_750_phys_vs_ptdf,
        "ieee30":  runs_ieee30_1000_phys_vs_ptdf,
    },
    training_networks_by_system={
        "cigre14": networks_cigre14_xlarge_singel_slack,
        "ieee30":  networks_ieee30_xlarge_medium_timesteps
    },
)

### Cross-System Factorial Analysis

### Backfill to prepare old runs

In [ ]:
# Backfill for the old runs that don't have bus count info in the test_metrics — add min, max, range based on the networks used for testing.
for sys_name, runs, nets in [
    ("ieee9",   runs_ieee9_500_phys_vs_ptdf,   networks_ieee9_large_singel_slack
     ),
    ("cigre14", runs_cigre14_500_phys_vs_ptdf, loaded_networks.get("networks_cigre14_large_singel_slack", [])
     ),
    ("cigre14", runs_cigre14_750_phys_vs_ptdf, loaded_networks.get("networks_cigre14_xlarge_singel_slack", [])
     ),
    ("ieee30",  runs_ieee30_500_phys_vs_ptdf,  loaded_networks.get("networks_ieee30_large_singel_slack", [])
     ),
    ("ieee30",  runs_ieee30_1000_phys_vs_ptdf, loaded_networks.get("networks_ieee30_xlarge_singel_slack", [])
     ),
    ("ieee9",   runs_ieee9_1500_phys_vs_ptdf,  loaded_networks.get("networks_ieee9_mega_singel_slack", [])
     ),
    ("cigre14", runs_cigre14_1500_phys_vs_ptdf, loaded_networks.get("networks_cigre14_mega_singel_slack", [])
     ),
]:
    bcmin = min(len(n.buses) for n in nets)
    bcmax = max(len(n.buses) for n in nets)
    for r in runs:
        r.setdefault("bus_count_min",   bcmin)
        r.setdefault("bus_count_max",   bcmax)
        r.setdefault("bus_count_range", bcmax - bcmin)
        r.setdefault("base_system", sys_name)
        r.setdefault("num_scenarios", len(nets))

In [ ]:
# Backfill standard deviation for the old runs that don't have it, using the test set results across the networks.
refresh_test_metrics_with_std(
    runs_by_system={
        "ieee9":        runs_ieee9_500_phys_vs_ptdf,
        "ieee9_1500":   runs_ieee9_1500_phys_vs_ptdf,
        "cigre14_500":  runs_cigre14_500_phys_vs_ptdf,
        "cigre14_750":  runs_cigre14_750_phys_vs_ptdf,
        "cigre14_1500": runs_cigre14_1500_phys_vs_ptdf,
    },
    training_networks_by_system={
        "ieee9":        networks_ieee9_large_singel_slack,
        "ieee9_1500":   loaded_networks["networks_ieee9_mega_singel_slack"],
        "cigre14_500":  loaded_networks["networks_cigre14_large_singel_slack"],
        "cigre14_750":  loaded_networks["networks_cigre14_xlarge_singel_slack"],
        "cigre14_1500": loaded_networks["networks_cigre14_mega_singel_slack"],
    },
)

In [ ]:
refresh_test_metrics_with_std(
    runs_by_system={
        "ieee30":  runs_ieee30_500_phys_vs_ptdf,
        "ieee30b": runs_ieee30_1000_phys_vs_ptdf,
        "cigre14_500": runs_cigre14_500_phys_vs_ptdf,   # 1 missing
    },
    training_networks_by_system={
        "ieee30":  loaded_networks["networks_ieee30_large_singel_slack"],
        "ieee30b": loaded_networks["networks_ieee30_xlarge_singel_slack"],
        "cigre14_500": loaded_networks["networks_cigre14_large_singel_slack"],
    },
)

In [ ]:
networks_cigre14_xlarge_singel_slack=loaded_networks.get("networks_cigre14_xlarge_singel_slack", [])

In [ ]:
networks_ieee30_mega_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee30_mega.json"))

##¤ Cross-System Generalisation

In [ ]:
# ── Cross-system generalisation experiment ────────────────────────────────────
runs_by_system = {
    "ieee9":    runs_ieee9_500_phys_vs_ptdf,
    "cigre14":  runs_cigre14_500_phys_vs_ptdf,
    "ieee30":   runs_ieee30_500_phys_vs_ptdf,
}
networks_by_system = {
    "ieee9":    networks_ieee9_large_singel_slack,
    "cigre14":  networks_cigre14_large_singel_slack,
    "ieee30":   networks_ieee30_xlarge_medium_timesteps
}

cross_results, _ = cross_system_eval(
    runs_by_system,
    networks_by_system,
    metric="vmag_mae",
)

# Matrix heatmap — all train×test combinations
plot_cross_system_heatmap(cross_results, metric_label="Vmag MAE [p.u.]")

# Per-test-system grouped bars — one figure per test system
for test_sys in runs_by_system:
    plot_cross_system_by_physics_mode(
        cross_results,
        test_system=test_sys,
        metric_label="Vmag MAE [p.u.]",
    )

##¤ check_hparam_results — all systems

In [ ]:
check_hparam_results(
    runs_cigre14_500_phys_vs_ptdf + runs_cigre14_750_phys_vs_ptdf,
    title_prefix="cigre14 all",
    training_networks=networks_cigre14_large_singel_slack,  # used only for fallback eval
    run_factorial_analysis=False
)

In [ ]:
check_hparam_results(runs_ieee9_500_phys_vs_ptdf,run_factorial_analysis=False,test_network=networks_ieee9_large_singel_slack[36],training_networks=networks_ieee9_large_singel_slack)

In [ ]:
check_hparam_results(runs_cigre14_500_phys_vs_ptdf,run_factorial_analysis=False,test_network=networks_cigre14_large_singel_slack[36],training_networks=networks_cigre14_large_singel_slack)

In [ ]:
check_hparam_results(runs_cigre14_750_phys_vs_ptdf,run_factorial_analysis=False,test_network=networks_cigre14_xlarge_single_slack[36],training_networks=networks_cigre14_xlarge_single_slack)

## Plots with new figures (old model)

#### IEEE9 - 500

In [ ]:
if 'runs_ieee9_500_phys_vs_ptdf' not in locals():
    runs_ieee9_500_phys_vs_ptdf=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"ieee9_large_single_slack_ptdf_sweep_060426.json")) 
if 'networks_ieee9_large_singel_slack' not in locals():
    networks_ieee9_large_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_large_singel_slack.json"))


In [ ]:
if 'runs_ieee9_500_wide' not in locals():
    runs_ieee9_test_phys_vs_ptdf=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"ieee9_500_single_slack_head_mode_060526.json"))

In [ ]:
ref_single_ieee9_500,ref_batch_ieee9_500=build_ref_baselines(networks_ieee9_500_singel_slack, n_sample=50)#,ref_single=ref_single_ieee9_500,ref_batch=ref_batch_ieee9_500)

In [ ]:
# Timing scatter: single-snapshot vs batch-amortised
# Populate ref_single and ref_batch first (see baseline cell above).
# ref_single = {'PyPower': pp_base, 'PyPSA-single': pypsa_single}
# ref_batch  = {'PyPSA-batch': pypsa_batch, 'DC-lpf': dc_base}

check_hparam_results(
    runs_ieee9_500_phys_vs_ptdf,
    run_factorial_analysis=True,
    test_network=networks_ieee9_large_singel_slack[36],
    training_networks=networks_ieee9_large_singel_slack,
    ref_baselines=ref_single_ieee9_500,
    ref_baselines_batch=ref_batch_ieee9_500,
    physics_comparison_results=False
)

#### IEEE9 - 1500

In [ ]:
check_hparam_results(
    runs_ieee9_1500_phys_vs_ptdf,
    training_networks=networks_ieee9_mega_singel_slack,
    title_prefix="IEEE9 mega (1500 scenarios) — PTDF sweep",
    run_factorial_analysis=True,
    test_network=networks_ieee9_mega_singel_slack[36],
    ref_baselines=ref_single,
    ref_baselines_batch=ref_batch,
)

#### CIGRE14 1500

In [ ]:
check_hparam_results(
    runs_cigre14_1500_phys_vs_ptdf,
    training_networks=networks_cigre14_mega_singel_slack,
    title_prefix="CIGRE14 mega (1500 scenarios) — PTDF sweep",
    run_factorial_analysis=True,
    test_network=networks_cigre14_mega_singel_slack[36],
    ref_baselines=ref_single_cigre14,
    ref_baselines_batch=ref_batch_cigre14,
)

#### IEEE30

In [ ]:
runs_ieee30_1500_phys_vs_ptdf=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"ieee30_mega_single_slack_ptdf_sweep_030526.json"))

In [ ]:
check_hparam_results(
    runs_ieee30_1500_phys_vs_ptdf,
    training_networks=networks_ieee30_mega_singel_slack,
    title_prefix="IEEE30 mega (1500 scenarios) — PTDF sweep",
    #run_factorial_analysis=True,
    test_network=networks_ieee30_mega_singel_slack[36],
    #ref_baselines=ref_single,
    #ref_baselines_batch=ref_batch,
)

### Mixed training

In [ ]:
runs_mixed_1500_phys_vs_ptdf= load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"mixed_1500_phys_vs_ptdf_sweep.json"))
networks_mixed_1500= load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR,"mixed_1500.json"))

In [ ]:
# make the ref baselines for the mixed dataset by averaging the ieee9 and cigre14 baselines (since the mixed dataset contains both types of networks)
# Baseline solver timing — Mixed 1500
# Each baseline is checked individually and only run if not already in memory.
if 'ref_single_mixed_1500' not in locals(): ref_single_mixed_1500 = {}
if 'ref_batch_mixed_1500'  not in locals(): ref_batch_mixed_1500  = {}

if '_sample' not in locals():
    _, _, _test_nets_mixed_1500 = reconstruct_test_split(networks_mixed_1500)
    _sample = _test_nets_mixed_1500[:20]

if "PyPower AC"   not in ref_single_mixed_1500:
    print("Running PyPower AC..."); ref_single_mixed_1500["PyPower AC"]   = evaluate_pypower_baseline(_sample)
if "PyPSA single" not in ref_single_mixed_1500:
    print("Running PyPSA single..."); ref_single_mixed_1500["PyPSA single"] = evaluate_pypsa_timing_single(_sample)
if "FDLF"         not in ref_single_mixed_1500:
    print("Running FDLF..."); ref_single_mixed_1500["FDLF"]         = evaluate_fdlf_baseline(_sample)
if "pandapower"   not in ref_single_mixed_1500:
    print("Running pandapower..."); ref_single_mixed_1500["pandapower"]   = evaluate_pandapower_baseline(_sample)

if "PyPSA batch"  not in ref_batch_mixed_1500:
    print("Running PyPSA batch..."); ref_batch_mixed_1500["PyPSA batch"]  = evaluate_pypsa_timing(_sample)
if "DC lpf"       not in ref_batch_mixed_1500:
    print("Running DC lpf..."); ref_batch_mixed_1500["DC lpf"]       = evaluate_dc_baseline(_sample)

print("\nref_single timings (ms/snap):")
for k, v in ref_single_mixed_1500.items():
    print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")
print("ref_batch timings (ms/snap):")
for k, v in ref_batch_mixed_1500.items():
    print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")


In [ ]:
check_hparam_results(
    runs_mixed_1500_phys_vs_ptdf,
    title_prefix="Mixed 1500",
    training_networks=networks_mixed_1500,  # used only for fallback eval
    test_network=networks_mixed_1500[36],
    factors=["weight_ptdf ,weight_phys","batch_size"],
    run_factorial_analysis=True,
    physics_comparison_results=True,
    ref_baselines=ref_single_mixed_1500,
    ref_baselines_batch=ref_batch_mixed_1500,
)

### Multi system comparison

In [ ]:
# print all the names currently in loaded_runs
for r in loaded_runs:
    print(r)

In [ ]:
# extract runs from the loaded runs dict

runs_ieee9_500_phys_vs_ptdf = loaded_runs.get("runs_ieee9_500_phys_vs_ptdf", [])
runs_cigre14_500_phys_vs_ptdf = loaded_runs.get("runs_cigre14_500_phys_vs_ptdf", [])
runs_cigre14_750_phys_vs_ptdf = loaded_runs.get("runs_cigre14_750_phys_vs_ptdf", [])
runs_ieee30_500_phys_vs_ptdf = loaded_runs.get("runs_ieee30_500_phys_vs_ptdf", [])
runs_ieee30_1000_phys_vs_ptdf = loaded_runs.get("runs_ieee30_1000_phys_vs_ptdf", [])
runs_ieee9_1500_phys_vs_ptdf = loaded_runs.get("runs_ieee9_1500_phys_vs_ptdf", [])
runs_cigre14_1500_phys_vs_ptdf = loaded_runs.get("runs_cigre14_1500_phys_vs_ptdf", [])
runs_mixed_1500_phys_vs_ptdf = loaded_runs.get("runs_mixed_1500_phys_vs_ptdf", [])




multi_system_factorial_analysis(
    runs_by_system={
        "ieee9":         runs_ieee9_500_phys_vs_ptdf,
        "cigre14_500": [r for r in runs_cigre14_500_phys_vs_ptdf if _has_std(r)],
        "cigre14_750":   runs_cigre14_750_phys_vs_ptdf,
        #"ieee30":        runs_ieee30_500_phys_vs_ptdf,
        #"ieee30b":       runs_ieee30_1000_phys_vs_ptdf,
        "ieee9_1500":    runs_ieee9_1500_phys_vs_ptdf,
        "cigre14_1500":  runs_cigre14_1500_phys_vs_ptdf,
        #"mixed_1500":   runs_mixed_1500_phys_vs_ptdf,
    },
    response_metrics=["vmag_mae_norm", "vang_mae_norm", "p_mae_norm", "q_mae_norm"],
    factors=["weight_physics", "weight_ptdf", "num_layers", "n_buses", "dataset_size"],
    interactions=[
        ("n_buses",      "weight_physics"),
        ("n_buses",      "weight_ptdf"),
        ("dataset_size", "weight_physics"),
    ],
    normalization="zscore",
)

## Plots with new results (New datasets and new model head)

#### IEEE9

##### first

In [ ]:
check_hparam_results(
    runs_ieee9_500_head_mode,
    training_networks=networks_ieee9_500_wide,
    title_prefix="IEEE9 (500 scenarios) — Head Mode sweep",
    run_factorial_analysis=True,
    factors=["weight_physics", "weight_ptdf", "head_mode"],
    test_network=networks_ieee9_500_wide[36],
    ref_baselines=ref_singel_ieee9_500_wide,
    ref_baselines_batch=ref_batch_ieee9_500_wide,
)

##### second

In [ ]:
runs_ieee9_150_test=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"ieee9_150_single_slack_new_physics_090526.json"))

In [ ]:
networks_ieee9_150=networks_ieee9_500_wide[0:150]

In [ ]:
check_hparam_results(
    runs_ieee9_150_test,
    training_networks=networks_ieee9_150,
    title_prefix="IEEE9 (150 scenarios) — New physics sweep",
    run_factorial_analysis=True,
    factors=["weight_physics", "weight_ptdf","head_mode","conv_mode"],
    interactions=[("weight_physics", "conv_mode"),("weight_ptdf", "conv_mode")],
    physics_comparison_results=False
)

#### CIGRE14

In [ ]:
print(loaded_runs.keys())

#### First run 070826

In [ ]:
check_hparam_results(
    loaded_runs["runs_cigre14_500_wide_head_compare"],
    training_networks=networks_cigre14_500_wide,
    title_prefix="CIGRE14 (500 scenarios) — Head Mode sweep",
    run_factorial_analysis=True,
    factors=["weight_physics", "weight_ptdf" ,"head_mode"],
    test_network=networks_cigre14_500_wide[36],
    ref_baselines=ref_singel_cigre14_500_wide,
    ref_baselines_batch=ref_batch_cigre14_500_wide,
)

#### second run 080526 Cigre14 - 500

In [ ]:
networks_cigre14_500_wide

In [ ]:
runs_cigre14_500_head=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"cigre14_500_wide_head_mode_070526.json"))

#### second run 080526 Cigre14 - 750

In [ ]:
networks_cigre14_750_wide=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "cigre14_750_wide.json"))

In [ ]:
runs_cigre14_750_head = load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"cigre14_750_wide_head_mode_070526.json"))

In [ ]:
ref_cigre14_750_wide_single,ref_cigre14_750_wide_batch=build_ref_baselines(networks_cigre14_750_wide, n_sample=50)

In [ ]:
check_hparam_results(
    runs_cigre14_750_head,
    training_networks=networks_cigre14_750_wide,
    title_prefix="CIGRE14 (750 scenarios) — Head Mode sweep",
    run_factorial_analysis=True,
    factors=["weight_physics", "weight_ptdf" ,"head_mode"],
    test_network=networks_cigre14_750_wide[36],
    ref_baselines=ref_cigre14_750_wide_single,
    ref_baselines_batch=ref_cigre14_750_wide_batch,
    use_pnom_share=True
)

### IEEE30

#### IEEE30 second run 080526

### Mixed Training

#### First attempt 080526

In [ ]:
networks_mixed_500=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "mixed_500_wide.json"))

In [ ]:
runs_mixed_500_head_mode=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"mixed_500_head_mode_070526.json"))

In [ ]:
ref_single_mixed_500,ref_batch_mixed_500=build_ref_baselines(networks_mixed_500, n_sample=50)

In [ ]:
check_hparam_results(
    runs_mixed_500_head_mode,
    training_networks=networks_mixed_500,
    title_prefix="Mixed (500 scenarios) — Head Mode sweep",
    run_factorial_analysis=True,
    factors=["weight_physics", "weight_ptdf","head_mode"],
    test_network=networks_mixed_500[36],
    ref_baselines=ref_single_mixed_500,
    ref_baselines_batch=ref_batch_mixed_500,
    use_pnom_share=True
)

#### Second run

In [ ]:
networks_mixed_1500_wide=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "mixed_1500_wide.json"))

In [ ]:
runs_mixed_1500=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"mixed_1500_wide_100526.json"))

In [ ]:
## keep only the runs with  conv_mode "old"
runs_mixed_1500_no_dropout=[r for r in runs_mixed_1500 if r.get("conv_mode") != "res_norm_drop"]

In [ ]:
check_hparam_results(
    runs_mixed_1500_no_dropout,
    training_networks=networks_mixed_1500_wide,
    title_prefix="Mixed (1500 scenarios) — Convolution Mode sweep",
    run_factorial_analysis=True,
    factors=["weight_physics", "weight_ptdf" ,"conv_mode","warmup_epochs"],
    interactions=[("weight_physics", "conv_mode"),("weight_ptdf", "conv_mode"),("warmup_epochs", "conv_mode")],
    test_network=networks_mixed_1500_wide[35],
    ref_baselines=ref_singel_mixed_1500_wide,
    ref_baselines_batch=ref_batch_mixed_1500_wide,
    use_pnom_share=False,
    physics_comparison_results=True,

)

## Misc testing

In [ ]:
print(list(runs_ieee9_500_phys_vs_ptdf[0].keys()))

In [ ]:
all_runs_by_sys = {
    "ieee9":         runs_ieee9_500_phys_vs_ptdf,
    "cigre14_500":   runs_cigre14_500_phys_vs_ptdf,
    "cigre14_750":   runs_cigre14_750_phys_vs_ptdf,
    "ieee30":        runs_ieee30_500_phys_vs_ptdf,
    "ieee30b":       runs_ieee30_1000_phys_vs_ptdf,
    "ieee9_1500":    runs_ieee9_1500_phys_vs_ptdf,
    "cigre14_1500":  runs_cigre14_1500_phys_vs_ptdf,
}
for sys, runs in all_runs_by_sys.items():
    n_total = len(runs)
    n_missing = sum(1 for r in runs if np.isnan(r.get("test_metrics",{}).get("vmag_true_std", float("nan"))))
    std_vals = [r.get("test_metrics",{}).get("vmag_true_std", float("nan")) for r in runs]
    print(f"{sys:15s}: {n_total} runs, {n_missing} missing std  | sample stds: {[round(v,4) for v in std_vals[:3]]}")

In [ ]:
import pickle, os

_runs_to_save = [
    (runs_ieee9_500_phys_vs_ptdf,   'ieee9_large_single_slack_ptdf_sweep_060426.json'),
    (runs_cigre14_500_phys_vs_ptdf, 'cigre14_large_single_slack_ptdf_sweep_060426.json'),
    (runs_cigre14_750_phys_vs_ptdf, 'cigre14_xlarge_single_slack_ptdf_sweep_110426.json'),
    (runs_ieee30_500_phys_vs_ptdf,  'ieee30_large_single_slack_ptdf_sweep_140426.json'),
    (runs_ieee30_1000_phys_vs_ptdf, 'ieee30_xlarge_single_slack_ptdf_sweep_210426.json'),
    (runs_ieee9_1500_phys_vs_ptdf,  'ieee9_mega_single_slack_ptdf_sweep_0204526.json'),
    (runs_cigre14_1500_phys_vs_ptdf,'cigre14_mega_single_slack_ptdf_sweep_030526.json'),
]

for runs_list, fname in _runs_to_save:
    path = os.path.join(TRAINING_RESULTS_DIR, fname)
    with open(path, "wb") as f:
        pickle.dump(runs_list, f)
    print(f"Saved {len(runs_list)} runs → {fname}")

In [ ]:
def assess_dangling_buses(networks, name=""):
    total_buses, total_dangling, total_snapshots = 0, 0, 0
    dangling_snapshots = 0
    for net in networks:
        buses = net.buses.index.tolist()
        load_buses  = set(net.loads.bus)
        gen_buses   = set(net.generators.bus)
        # degree: count line endpoints
        from collections import Counter
        deg = Counter()
        for _, row in net.lines.iterrows():
            deg[row.bus0] += 1
            deg[row.bus1] += 1
        for _, row in net.transformers.iterrows():
            deg[row.bus0] += 1
            deg[row.bus1] += 1
        dangling = [b for b in buses
                    if b not in load_buses
                    and b not in gen_buses
                    and deg.get(b, 0) == 1]
        total_buses    += len(buses)
        total_dangling += len(dangling)
        if dangling:
            dangling_snapshots += len(net.snapshots)
        total_snapshots += len(net.snapshots)

    pct = 100 * total_dangling / total_buses if total_buses else 0
    snap_pct = 100 * dangling_snapshots / total_snapshots if total_snapshots else 0
    print(f"{name:30s}: {len(networks):4d} nets | "
          f"{total_dangling:4d}/{total_buses:5d} dangling buses ({pct:.1f}%) | "
          f"snapshots with ≥1 dangling: {dangling_snapshots}/{total_snapshots} ({snap_pct:.0f}%)")

for name, nets in [
    ("ieee9_500",   networks_ieee9_large_singel_slack),
    ("cigre14_1500", loaded_networks.get("networks_cigre14_mega_singel_slack", [])),
    ("ieee30_1500", loaded_networks.get("networks_ieee30_xlarge_singel_slack", [])),
]:
    assess_dangling_buses(nets, name)

#### test the dangeling bus case

In [ ]:
print(f"{'Run':>3}  {'vmag':>12} {'→':>3} {'vmag':>12}  {'vang':>12} {'→':>3} {'vang':>12}  {'p':>12} {'→':>3} {'p':>12}  {'q':>12} {'→':>3} {'q':>12}")
print(f"{'':>3}  {'before':>12} {'':>3} {'after':>12}  {'before':>12} {'':>3} {'after':>12}  {'before':>12} {'':>3} {'after':>12}  {'before':>12} {'':>3} {'after':>12}")
print("-" * 110)
for i, r in enumerate(runs_ieee9_500_phys_vs_ptdf):
    if r.get("model") is None: continue
    _, _, test_nets = reconstruct_test_split(networks_ieee9_large_singel_slack, seed=r.get("seed", 42))
    pt = r.get("_ptdf_params"); max_b = int(r.get("_max_buses", 0))

    m0 = evaluate_gnn_on_test_set(r["model"], test_nets, ptdf_params=pt, max_buses=max_b, postprocess_dangling=False)
    m1 = evaluate_gnn_on_test_set(r["model"], test_nets, ptdf_params=pt, max_buses=max_b, postprocess_dangling=True)

    print(f"{i:>3}  "
          f"{m0['vmag_mae']:>12.5f} → {m1['vmag_mae']:>12.5f}  "
          f"{m0['vang_mae']:>12.5f} → {m1['vang_mae']:>12.5f}  "
          f"{m0['p_mae']:>12.5f} → {m1['p_mae']:>12.5f}  "
          f"{m0['q_mae']:>12.5f} → {m1['q_mae']:>12.5f}")

#### test the ieee30 crazy losses

In [ ]:
print(list(loaded_networks.keys()))

In [ ]:
import torch
nets = networks_ieee30_mega_singel_slack
nets3 = networks_ieee9_large_singel_slack
nets2= loaded_networks.get("networks_cigre14_mega_singel_slack", [])[0]
print(f"Total networks: {len(nets)}")
p_maxes, q_maxes, v_maxes, v_mins = [], [], [], []
for net in nets:
    for snap in net.snapshots:
        p = net.buses_t.p.loc[snap].abs()
        q = net.buses_t.q.loc[snap].abs()
        v = net.buses_t.v_mag_pu.loc[snap]
        p_maxes.append(float(p.max())); q_maxes.append(float(q.max()))
        v_maxes.append(float(v.max())); v_mins.append(float(v.min()))

import numpy as np
for name, vals in [("P max [pu]", p_maxes), ("Q max [pu]", q_maxes),
                   ("Vmag max", v_maxes), ("Vmag min", v_mins)]:
    arr = np.array(vals)
    print(f"{name:15s}  mean={arr.mean():.3f}  std={arr.std():.3f}  "
          f"p95={np.percentile(arr,95):.3f}  max={arr.max():.3f}  "
          f"n_outlier={int((arr > arr.mean()+5*arr.std()).sum())}")

In [ ]:
# Find networks with extreme P or Q values
outlier_nets = []
for i, net in enumerate(nets):
    for snap in net.snapshots:
        p_max = float(net.buses_t.p.loc[snap].abs().max())
        q_max = float(net.buses_t.q.loc[snap].abs().max())
        if p_max > 50 or q_max > 50 or np.isnan(p_max):
            outlier_nets.append((i, snap, p_max, q_max))
            break  # one per network is enough

print(f"Networks with extreme P/Q (>50 pu or NaN): {len(outlier_nets)}")
for i, snap, p, q in outlier_nets[:10]:
    print(f"  net[{i:4d}]  snap={snap}  P_max={p:.1f}  Q_max={q:.1f}")

In [ ]:
# Reconstruct the same split used in training (seed from a run)
r = runs_ieee30_1000_phys_vs_ptdf[0]  # use first run's seed
nets_full = networks_ieee30_mega_singel_slack
train_nets, val_nets, test_nets = reconstruct_test_split(nets_full, seed=r.get("seed", 42))

def mean_p_max(nets_list):
    vals = []
    for net in nets_list:
        for snap in net.snapshots:
            vals.append(float(net.buses_t.p.loc[snap].abs().max()))
    return np.mean(vals), np.std(vals), np.max(vals)

tm, ts, tmx = mean_p_max(train_nets)
vm, vs, vmx = mean_p_max(val_nets)
print(f"Train  P_max: mean={tm:.3f} std={ts:.3f} max={tmx:.3f}  ({len(train_nets)} nets)")
print(f"Val    P_max: mean={vm:.3f} std={vs:.3f} max={vmx:.3f}  ({len(val_nets)} nets)")

In [ ]:
net0 = nets[0]
print("Generator columns:", list(net0.generators.columns))
print(net0.generators[["bus", "p_nom", "control"]].to_string())

print(f"\nLoads in net[0]:")
print(net0.loads[["bus", "p_set", "q_set"]].to_string())

# Key check: what is the Q/P ratio across all loads?
all_p, all_q = [], []
for net in nets[:50]:  # sample 50 networks
    all_p.extend(net.loads["p_set"].values.tolist())
    all_q.extend(net.loads["q_set"].values.tolist())

import numpy as np
all_p, all_q = np.array(all_p), np.array(all_q)
nonzero = all_p > 0.01
print(f"\nLoad Q/P ratio (power factor proxy):")
print(f"  mean Q/P = {(all_q[nonzero]/all_p[nonzero]).mean():.3f}")
print(f"  p50  Q/P = {np.median(all_q[nonzero]/all_p[nonzero]):.3f}")
print(f"  p95  Q/P = {np.percentile(all_q[nonzero]/all_p[nonzero], 95):.3f}")
print(f"\n  P range: [{all_p.min():.3f}, {all_p.max():.3f}]  (pu on 100 MVA base, should be ~0.1–5.0)")
print(f"  Q range: [{all_q.min():.3f}, {all_q.max():.3f}]  (pu on 100 MVA base, should be ~0.05–2.0)")

# Also check the raw CSV to compare
import pandas as pd
csv_loads = pd.read_csv("grid_model_files/ieee30_loads.csv")
print(f"\nRaw CSV ieee30_loads sample:")
print(csv_loads.head(10).to_string())

In [ ]:
import numpy as np

def summarize_network_set(nets, name):
    all_p, all_q, all_vmag = [], [], []
    for net in nets[:100]:  # sample 100 for speed
        for snap in net.snapshots:
            p = net.buses_t.p.loc[snap].values
            q = net.buses_t.q.loc[snap].values
            v = net.buses_t.v_mag_pu.loc[snap].values
            all_p.extend(p.tolist())
            all_q.extend(q.tolist())
            all_vmag.extend(v.tolist())

    all_p, all_q, all_vmag = np.array(all_p), np.array(all_q), np.array(all_vmag)
    nonzero = np.abs(all_p) > 0.01
    qp_ratio = np.abs(all_q[nonzero]) / np.abs(all_p[nonzero])

    # Also grab raw load p_set/q_set from static network
    load_p = np.concatenate([net.loads["p_set"].values for net in nets[:20]])
    load_q = np.concatenate([net.loads["q_set"].values for net in nets[:20]])

    print(f"\n{'='*60}")
    print(f"  {name}  ({len(nets)} networks)")
    print(f"{'='*60}")
    print(f"  Bus injections (buses_t, all snapshots):")
    print(f"    P range:      [{all_p.min():.3f}, {all_p.max():.3f}]  mean={all_p.mean():.3f}")
    print(f"    Q range:      [{all_q.min():.3f}, {all_q.max():.3f}]  mean={all_q.mean():.3f}")
    print(f"    Vmag range:   [{all_vmag.min():.3f}, {all_vmag.max():.3f}]  mean={all_vmag.mean():.3f}")
    print(f"    Q/P ratio:    mean={qp_ratio.mean():.3f}  p95={np.percentile(qp_ratio,95):.3f}  max={qp_ratio.max():.3f}")
    print(f"  Load static (p_set/q_set):")
    print(f"    p_set range:  [{load_p.min():.3f}, {load_p.max():.3f}]  mean={load_p.mean():.3f}")
    print(f"    q_set range:  [{load_q.min():.3f}, {load_q.max():.3f}]  mean={load_q.mean():.3f}")
    if (load_p > 0.01).any():
        lqp = load_q[load_p > 0.01] / load_p[load_p > 0.01]
        print(f"    q/p ratio:    mean={lqp.mean():.3f}  max={lqp.max():.3f}")

nets_ieee30  = networks_ieee30_mega_singel_slack  # user's variable name
nets_ieee9   = loaded_networks.get("networks_ieee9_mega_singel_slack", [])
nets_cigre14 = loaded_networks.get("networks_cigre14_mega_singel_slack", [])

for nets, name in [
    (nets_ieee9,   "ieee9  large"),
    (nets_cigre14, "cigre14 mega"),
    (nets_ieee30,  "ieee30  mega"),
]:
    if len(nets) == 0:
        print(f"\n{name}: NOT LOADED")
        continue
    summarize_network_set(nets, name)

In [ ]:
thresholds = [0.80, 0.82, 0.85, 0.90,0.92, 0.95, 0.98, 0.99]
for v_thresh in thresholds:
    n_rejected = sum(1 for net in nets_cigre14
                     if net.buses_t.v_mag_pu.min().min() < v_thresh)
    print(f"  Vmag < {v_thresh}: {n_rejected}/{len(nets_cigre14)} "
          f"rejected ({100*n_rejected/len(nets_cigre14):.1f}%)")

In [ ]:
v_mins_c14 = np.array([net.buses_t.v_mag_pu.min().min() 
                        for net in loaded_networks["networks_cigre14_mega_singel_slack"]])
print(f"cigre14 current Vmag_min: mean={v_mins_c14.mean():.3f}  min={v_mins_c14.min():.3f}")

# Estimate what volatility is needed to get 10-20% of samples below 0.95 pu
current_vrange = 0.5  # assumed
for v_range in [0.3, 0.5, 0.75, 1.0, 1.5,1.6, 1.75,2.0]:
    scale = v_range / current_vrange
    v_mins_scaled = 1.0 - (1.0 - v_mins_c14) * scale
    pct_below_95 = 100 * (v_mins_scaled < 0.95).mean()
    pct_below_82 = 100 * (v_mins_scaled < 0.82).mean()
    print(f"  volatility_range={v_range:.2f} → <0.95: {pct_below_95:.1f}%  <0.82: {pct_below_82:.1f}%")

#### Test the p_weights

In [ ]:
runs_test_pnom=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR,"ieee9_test_slack_weights_no_p_bal.json"))
networks_ieee9_test_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_test.json"))

In [ ]:
# --- Smoke test: pnom_weighted feature in Analysis ---


ds_test = PowerFlowDataset(networks_ieee9_test_singel_slack, use_pnom_share=True)
d = ds_test[0]
model=runs_test_pnom[0].get("model")
# 1. Feature shape check
assert d.x.shape[1] == 8, f"Expected 8 features, got {d.x.shape[1]}"
print("Feature shape OK:", d.x.shape)

# 2. Evaluate
metrics = evaluate_gnn_on_test_set(model, networks_ieee9_test_singel_slack, use_pnom_share=True)
print("vmag_mae:", metrics["vmag_mae"])
print("in_features:", model.node_embedding.in_features)  # expect 8

In [ ]:
ref_singe_test_pnom, ref_batch_test_pnom = build_ref_baselines(networks_ieee9_test_singel_slack, n_sample=20)

#### Checking the use_pnom_shared treading

In [ ]:
check_hparam_results(
    runs_test_pnom,
    training_networks=networks_ieee9_test_singel_slack,
    title_prefix="IEEE9 test pnom_weighted",
    run_factorial_analysis=True,
    factors=["use_pnom_share"],
    ref_baselines=ref_singe_test_pnom,
    ref_baselines_batch=ref_batch_test_pnom,
    test_network=networks_ieee9_test_singel_slack[6],
)